In [1]:
import os
import sys
import json
import logging
import hashlib
from datetime import datetime, timezone
import pandas as pd
import requests

# 1. Configure Enterprise Production Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] [%(name)s] %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger("EnterpriseDataPipeline")

# 2. Define Unified Data Lake Absolute Architecture (Google Colab / Local Execution)
BASE_DIR = "/content" if os.path.exists("/content") else os.getcwd()

DATA_DIRS = {
    # Bronze Layer Sources (Raw Data Lake)
    "bronze_fda": os.path.join(BASE_DIR, "data", "bronze", "openfda"),
    "bronze_rxnav": os.path.join(BASE_DIR, "data", "bronze", "rxnav"),
    "bronze_who": os.path.join(BASE_DIR, "data", "bronze", "who"),
    "bronze_kaggle": os.path.join(BASE_DIR, "data", "bronze", "kaggle"),
    "bronze_eda": os.path.join(BASE_DIR, "data", "bronze", "eda"),
    "bronze_eda_pdfs": os.path.join(BASE_DIR, "data", "bronze", "eda", "pdfs"),
    "bronze_rag_corpus": os.path.join(BASE_DIR, "data", "bronze", "rag_corpus"),

    # Silver Layer (Cleansed, Standardized, Integrated)
    "silver": os.path.join(BASE_DIR, "data", "silver"),

    # Gold / Audit & Reports Layer
    "reports": os.path.join(BASE_DIR, "reports")
}

# Create complete directory tree
for name, dir_path in DATA_DIRS.items():
    os.makedirs(dir_path, exist_ok=True)
    logger.info(f"Directory Anchor Verified [{name}]: {dir_path}")

logger.info(f"Pipeline Root Path Locked to: {BASE_DIR}")

# 3. Core Cryptographic & Persistence Utilities
def generate_record_hash(record_dict: dict) -> str:
    """Generates a deterministic SHA256 cryptographic hash for data lineage tracking."""
    serialized = json.dumps(record_dict, sort_keys=True, default=str)
    return hashlib.sha256(serialized.encode('utf-8')).hexdigest()

def save_raw_json(data: dict, filepath: str) -> None:
    """Persists structured raw payloads with encoding safeguards."""
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    logger.info(f"Persisted payload artifact: {filepath} ({os.path.getsize(filepath)} bytes)")

print("\n" + "="*80)
print("✅ CELL 1 EXECUTED SUCCESSFULLY: ENTERPRISE DATA LAKE ANCHORS READY")
print("="*80)


✅ CELL 1 EXECUTED SUCCESSFULLY: ENTERPRISE DATA LAKE ANCHORS READY


In [2]:
import os
import requests
import zipfile
import io
from datetime import datetime, timezone

def fetch_openfda_paginated_strict(endpoint: str, search_query: str = None, target_total: int = 500, page_size: int = 100) -> dict:
    """
    Paginated openFDA REST Extractor with explicit execution state auditing.
    """
    base_url = f"https://api.fda.gov/drug/{endpoint}.json"
    headers = {"User-Agent": "EnterpriseDataPipeline/6.0 (StrictProduction)"}

    all_results = []
    skip = 0
    status_code = None
    execution_status = "SUCCESS"
    error_message = None

    logger.info(f"Initiating openFDA Extraction [{endpoint}] | Target: {target_total} records...")

    while len(all_results) < target_total:
        current_limit = min(page_size, target_total - len(all_results))
        params = {"limit": current_limit, "skip": skip}
        if search_query:
            params["search"] = search_query

        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=20)
            status_code = response.status_code

            if response.status_code == 404:
                execution_status = "COMPLETED_END_OF_API_DATA"
                logger.info(f"openFDA [{endpoint}]: API returned 404 (No more data available at skip={skip}).")
                break

            response.raise_for_status()
            payload = response.json()
            batch_results = payload.get("results", [])

            if not batch_results:
                execution_status = "COMPLETED_EMPTY_BATCH"
                break

            all_results.extend(batch_results)
            skip += len(batch_results)

            if len(batch_results) < current_limit:
                execution_status = "COMPLETED_FULL_AVAILABLE"
                break

        except requests.exceptions.RequestException as e:
            logger.error(f"HTTP Error on openFDA [{endpoint}] at skip={skip}: {str(e)}")
            execution_status = "FAILED_NETWORK_ERROR"
            error_message = str(e)
            break

    return {
        "metadata": {
            "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "source_endpoint": base_url,
            "query_filter": search_query or "NONE",
            "requested_target": target_total,
            "retrieved_count": len(all_results),
            "http_last_status": status_code,
            "execution_status": execution_status,
            "error_details": error_message
        },
        "results": all_results
    }

def download_and_extract_openfda_bulk_ndc(target_dir: str) -> dict:
    """
    Downloads and extracts the official openFDA Bulk NDC directory ZIP file directly.
    """
    bulk_url = "https://download.open.fda.gov/drug/ndc/drug-ndc-0001-of-0001.json.zip"
    logger.info(f"Attempting live openFDA Bulk ZIP download from: {bulk_url}")

    execution_status = "SUCCESS"
    extracted_file_path = None
    extracted_records_count = 0
    error_message = None

    try:
        response = requests.get(bulk_url, stream=True, timeout=30)
        if response.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                z.extractall(target_dir)
                extracted_files = [os.path.join(target_dir, f) for f in z.namelist()]
                if extracted_files:
                    extracted_file_path = extracted_files[0]
                    # Quick size validation
                    file_size_mb = os.path.getsize(extracted_file_path) / (1024 * 1024)
                    logger.info(f"Bulk openFDA ZIP extracted successfully: {extracted_file_path} ({file_size_mb:.2f} MB)")
                    execution_status = "EXTRACTED_SUCCESSFULLY"
        else:
            execution_status = f"FAILED_HTTP_{response.status_code}"
    except Exception as e:
        logger.warning(f"Bulk openFDA download skipped or failed: {str(e)}")
        execution_status = "DOWNLOAD_FAILED"
        error_message = str(e)

    return {
        "metadata": {
            "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "source_url": bulk_url,
            "execution_status": execution_status,
            "extracted_file_path": extracted_file_path,
            "error_details": error_message
        }
    }

# 1. Source 1: openFDA NDC Directory Ingestion
ndc_payload = fetch_openfda_paginated_strict(endpoint="ndc", search_query='finished:true', target_total=300, page_size=100)
save_raw_json(ndc_payload, os.path.join(DATA_DIRS["bronze_fda"], "fda_ndc_raw.json"))

# 2. Source 2: openFDA Enforcement Recalls Ingestion
enforcement_payload = fetch_openfda_paginated_strict(endpoint="enforcement", search_query='status:"Ongoing"', target_total=150, page_size=100)
save_raw_json(enforcement_payload, os.path.join(DATA_DIRS["bronze_fda"], "fda_enforcement_raw.json"))

# 3. Source 3: openFDA Drug Shortages Ingestion
shortages_payload = fetch_openfda_paginated_strict(endpoint="drugshortages", target_total=150, page_size=100)
save_raw_json(shortages_payload, os.path.join(DATA_DIRS["bronze_fda"], "fda_shortages_raw.json"))

# 4. Source 4: openFDA Bulk ZIP Ingestion
bulk_payload = download_and_extract_openfda_bulk_ndc(DATA_DIRS["bronze_fda"])
save_raw_json(bulk_payload, os.path.join(DATA_DIRS["bronze_fda"], "fda_bulk_download_raw.json"))

print("\n" + "="*80)
print(f"✅ CELL 2 EXECUTED (STRICT PRODUCTION INGESTION):")
print(f"   - FDA NDC Records: {len(ndc_payload['results'])} (Status: {ndc_payload['metadata']['execution_status']})")
print(f"   - FDA Enforcement Records: {len(enforcement_payload['results'])} (Status: {enforcement_payload['metadata']['execution_status']})")
print(f"   - FDA Shortages Records: {len(shortages_payload['results'])} (Status: {shortages_payload['metadata']['execution_status']})")
print(f"   - FDA Bulk ZIP Bulk Download Status: {bulk_payload['metadata']['execution_status']}")
print("="*80)


✅ CELL 2 EXECUTED (STRICT PRODUCTION INGESTION):
   - FDA NDC Records: 300 (Status: SUCCESS)
   - FDA Enforcement Records: 150 (Status: SUCCESS)
   - FDA Shortages Records: 0 (Status: COMPLETED_END_OF_API_DATA)
   - FDA Bulk ZIP Bulk Download Status: EXTRACTED_SUCCESSFULLY


In [3]:
import os
import glob
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timezone

# 1. EDA Safety Alerts and PDF Corpus Extraction Engine
def process_eda_alerts_strict() -> dict:
    target_url = "https://www.edaegypt.gov.eg/en/media-center/safety-alerts/"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    alerts = []

    # Live Scrape
    try:
        response = requests.get(target_url, headers=headers, timeout=12)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            cards = soup.find_all(['div', 'article', 'tr'], class_=['alert', 'post', 'card', 'notice'])
            for idx, card in enumerate(cards):
                title = card.get_text(strip=True)
                link_tag = card.find('a', href=True)
                pdf_url = link_tag['href'] if link_tag else None
                if title:
                    alerts.append({
                        "alert_id": f"EDA-SCRAPED-{idx+1:03d}",
                        "alert_title": title,
                        "pdf_url": pdf_url,
                        "source_type": "LIVE_EDA_SCRAPE",
                        "ingestion_date_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d")
                    })
    except Exception as e:
        logger.warning(f"EDA Live Scrape attempted: {str(e)}")

    # Local Raw PDF Ingestion (Excluding output silver/report directories)
    local_pdfs = glob.glob(os.path.join(DATA_DIRS["bronze_eda_pdfs"], "*.pdf"))
    for pdf_p in set(glob.glob(os.path.join(BASE_DIR, "**", "*.pdf"), recursive=True)):
        if "data/bronze" in pdf_p or "eda" in pdf_p.lower():
            if not any(x in pdf_p for x in ["silver", "reports", "checkpoint"]):
                local_pdfs.append(pdf_p)

    valid_pdfs = list(set(local_pdfs))

    for idx, pdf_path in enumerate(valid_pdfs):
        extracted_text, status = extract_pdf_text_robust(pdf_path) if 'extract_pdf_text_robust' in globals() else ("", "PENDING")
        alerts.append({
            "alert_id": f"EDA-PDF-{idx+1:03d}",
            "alert_title": os.path.basename(pdf_path),
            "pdf_path": pdf_path,
            "extracted_text_sample": extracted_text[:300] if extracted_text else "",
            "extraction_status": status,
            "source_type": "LOCAL_PDF_RAW",
            "file_modified_date_utc": datetime.fromtimestamp(os.path.getmtime(pdf_path), tz=timezone.utc).strftime("%Y-%m-%d")
        })

    for alert in alerts:
        alert["lineage_hash"] = generate_record_hash(alert)

    return {
        "metadata": {
            "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "source": "Egyptian Drug Authority (EDA)",
            "total_records": len(alerts),
            "local_pdfs_processed": len(valid_pdfs),
            "has_synthetic_data": False,
            "execution_status": "SUCCESS" if alerts else "NO_LOCAL_DATA_FOUND"
        },
        "results": alerts
    }

# 2. Source 10: FDA Safety Corpus (Derived Analysis Corpus)
def build_fda_safety_corpus_strict() -> dict:
    enforcement_path = os.path.join(DATA_DIRS["bronze_fda"], "fda_enforcement_raw.json")
    corpus_records = []

    if os.path.exists(enforcement_path):
        with open(enforcement_path, 'r', encoding='utf-8') as f:
            enforcement_data = json.load(f)
            for idx, item in enumerate(enforcement_data.get("results", [])):
                corpus_records.append({
                    "corpus_id": f"SAFETY-CORPUS-{idx+1:04d}",
                    "source_recall_number": item.get("recall_number", "UNKNOWN"),
                    "safety_text": f"Reason: {item.get('reason_for_recall', '')} | Product: {item.get('product_description', '')}",
                    "classification": item.get("classification", "UNKNOWN"),
                    "lineage_hash": generate_record_hash(item)
                })

    return {
        "metadata": {
            "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "source": "FDA Safety Corpus (Derived from Enforcement Raw)",
            "is_independent_primary_source": False,
            "total_records": len(corpus_records)
        },
        "results": corpus_records
    }

# 3. Source 11: Drugs.com Price Directory (NO MOCK DATA POLICY)
def process_drugs_com_pricing_strict() -> dict:
    """
    Strict audit function. Absolutely NO hardcoded/fake prices allowed.
    If no official raw CSV/API artifact is placed in bronze, it registers as UNAVAILABLE.
    """
    raw_pricing_file = os.path.join(DATA_DIRS["bronze_rag_corpus"], "drugs_com_pricing.csv")
    records = []
    execution_status = "SOURCE_UNAVAILABLE_NO_MOCK_DATA"

    if os.path.exists(raw_pricing_file):
        try:
            df = pd.read_csv(raw_pricing_file)
            records = df.to_dict(orient='records')
            execution_status = "SUCCESS"
        except Exception as e:
            execution_status = f"FAILED_READING_FILE: {str(e)}"

    return {
        "metadata": {
            "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "source": "Drugs.com Pricing Reference",
            "execution_status": execution_status,
            "total_records": len(records),
            "has_synthetic_data": False
        },
        "results": records
    }

eda_payload = process_eda_alerts_strict()
save_raw_json(eda_payload, os.path.join(DATA_DIRS["bronze_eda"], "eda_alerts_raw.json"))

safety_corpus_payload = build_fda_safety_corpus_strict()
save_raw_json(safety_corpus_payload, os.path.join(DATA_DIRS["bronze_rag_corpus"], "fda_safety_corpus.json"))

drugs_com_payload = process_drugs_com_pricing_strict()
save_raw_json(drugs_com_payload, os.path.join(DATA_DIRS["bronze_rag_corpus"], "drugs_com_pricing_raw.json"))

print("\n" + "="*80)
print(f"✅ CELL 3 EXECUTED (ZERO-MOCK DATA GUARANTEE):")
print(f"   - EDA Safety Notices: {eda_payload['metadata']['total_records']} Records (Status: {eda_payload['metadata']['execution_status']})")
print(f"   - FDA Safety Corpus Built: {safety_corpus_payload['metadata']['total_records']} Records")
print(f"   - Drugs.com Price Source Status: {drugs_com_payload['metadata']['execution_status']} (Records: {drugs_com_payload['metadata']['total_records']})")
print("="*80)


✅ CELL 3 EXECUTED (ZERO-MOCK DATA GUARANTEE):
   - EDA Safety Notices: 0 Records (Status: NO_LOCAL_DATA_FOUND)
   - FDA Safety Corpus Built: 150 Records
   - Drugs.com Price Source Status: SOURCE_UNAVAILABLE_NO_MOCK_DATA (Records: 0)


In [4]:
import os
import json
import time
import requests
from datetime import datetime, timezone

def extract_target_ingredients_from_fda(fda_ndc_path: str, max_drugs: int = 30) -> list[str]:
    """
    Dynamically parses active ingredients and generic names from openFDA raw NDC JSON artifact.
    """
    if not os.path.exists(fda_ndc_path):
        logger.warning(f"FDA NDC file missing at {fda_ndc_path}. Returning fallback dynamic list.")
        return ["amoxicillin", "ibuprofen", "metformin"]

    with open(fda_ndc_path, 'r', encoding='utf-8') as f:
        fda_data = json.load(f)

    extracted_names = set()
    for record in fda_data.get("results", []):
        ingredients = record.get("active_ingredients", [])
        for ing in ingredients:
            name = ing.get("name", "").strip().lower()
            if name and len(name) > 3:
                extracted_names.add(name)

        generic = record.get("generic_name", "").strip().lower()
        if generic and len(generic) > 3:
            extracted_names.add(generic.split()[0])

        if len(extracted_names) >= max_drugs:
            break

    return list(extracted_names)[:max_drugs]


def fetch_rxnav_rxcui_enterprise(drug_name: str) -> dict:
    """
    Queries NIH RxNav REST API for RxCUI concepts.
    Handles rate-limiting safely to respect NIH endpoints.
    """
    base_url = "https://rxnav.nlm.nih.gov/REST/rxcui.json"
    params = {"name": drug_name}
    headers = {"Accept": "application/json"}

    try:
        response = requests.get(base_url, params=params, headers=headers, timeout=10)
        time.sleep(0.1)  # Respect NIH Rate Limits

        if response.status_code == 200:
            payload = response.json()
            rxnorm_ids = payload.get("idGroup", {}).get("rxnormId", [])

            return {
                "query_drug_name": drug_name,
                "rxcui": rxnorm_ids[0] if rxnorm_ids else None,
                "all_rxcuis": rxnorm_ids,
                "status": "MATCHED" if rxnorm_ids else "NO_RXCUI_MATCH"
            }
        else:
            return {
                "query_drug_name": drug_name,
                "rxcui": None,
                "all_rxcuis": [],
                "status": f"HTTP_{response.status_code}"
            }

    except Exception as e:
        logger.error(f"RxNav Query Exception for '{drug_name}': {str(e)}")
        return {
            "query_drug_name": drug_name,
            "rxcui": None,
            "all_rxcuis": [],
            "status": "ERROR",
            "error_details": str(e)
        }

# 1. Extract dynamic targets directly from real FDA Bronze artifact
fda_ndc_file = os.path.join(DATA_DIRS["bronze_fda"], "fda_ndc_raw.json")
dynamic_targets = extract_target_ingredients_from_fda(fda_ndc_file, max_drugs=30)

logger.info(f"Extracted {len(dynamic_targets)} dynamic target drugs for RxNav mapping.")

# 2. Execute Dynamic Ingestion
rxnav_results = []
for drug in dynamic_targets:
    res = fetch_rxnav_rxcui_enterprise(drug)
    res["lineage_hash"] = generate_record_hash(res)
    rxnav_results.append(res)

rxnav_payload = {
    "metadata": {
        "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "source": "NIH RxNav API",
        "total_queries_executed": len(rxnav_results),
        "successful_matches": sum(1 for r in rxnav_results if r["rxcui"] is not None)
    },
    "results": rxnav_results
}

rxnav_output_path = os.path.join(DATA_DIRS["bronze_rxnav"], "rxnav_concepts_raw.json")
save_raw_json(rxnav_payload, rxnav_output_path)

print("\n" + "="*80)
print(f"✅ CELL 4 EXECUTED SUCCESSFULLY:")
print(f"   - Dynamic Drug Targets Extracted: {len(dynamic_targets)}")
print(f"   - RxNav Successful RxCUI Matches: {rxnav_payload['metadata']['successful_matches']} / {len(rxnav_results)}")
print("="*80)


✅ CELL 4 EXECUTED SUCCESSFULLY:
   - Dynamic Drug Targets Extracted: 30
   - RxNav Successful RxCUI Matches: 26 / 30


In [5]:
import os
import glob
import json
import pandas as pd
from datetime import datetime, timezone

def ingest_who_eml_strict_no_fallback() -> dict:
    """
    Ingests official WHO EML datasets ONLY from legitimate raw files.
    Strictly forbids generating fake WHO records from FDA ingredients.
    Explicitly excludes previous pipeline output files.
    """
    logger.info("Scanning for official raw WHO EML files...")

    # 1. Search for potential raw WHO files
    all_found = (
        glob.glob(os.path.join(DATA_DIRS["bronze_who"], "*.*")) +
        glob.glob(os.path.join(BASE_DIR, "**", "*who*.*"), recursive=True) +
        glob.glob(os.path.join(BASE_DIR, "**", "*eml*.*"), recursive=True)
    )

    # Strictly exclude pipeline outputs to prevent self-reading
    valid_raw_files = []
    for f in set(all_found):
        f_lower = f.lower()
        if any(bad_dir in f_lower for bad_dir in ["silver", "reports", "checkpoint", "output_bundle"]):
            continue
        if f_lower.endswith(('.csv', '.json', '.xlsx')) and not f_lower.endswith('.py'):
            if os.path.getsize(f) > 200:  # Must contain actual content
                valid_raw_files.append(f)

    records = []
    source_file = "NONE"
    execution_status = "NO_WHO_DATA_FOUND"

    if valid_raw_files:
        chosen_file = valid_raw_files[0]
        source_file = chosen_file
        try:
            if chosen_file.endswith('.csv'):
                df = pd.read_csv(chosen_file)
                records = df.to_dict(orient='records')
            elif chosen_file.endswith('.xlsx'):
                df = pd.read_excel(chosen_file)
                records = df.to_dict(orient='records')
            elif chosen_file.endswith('.json'):
                with open(chosen_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    records = data.get("results", data) if isinstance(data, dict) else data

            if records:
                execution_status = "SUCCESS_PARSED_RAW_FILE"
        except Exception as e:
            logger.error(f"Error parsing WHO file {chosen_file}: {str(e)}")
            execution_status = f"PARSE_ERROR: {str(e)}"

    # ZERO Fallback/Synthetic Generation: If no raw file, records remain EMPTY
    processed_records = []
    for r in records:
        if isinstance(r, dict):
            r["lineage_hash"] = generate_record_hash(r)
            processed_records.append(r)

    return {
        "metadata": {
            "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "source": "WHO Essential Medicines List (EML)",
            "source_file": source_file,
            "execution_status": execution_status,
            "total_records": len(processed_records),
            "has_synthetic_data": False
        },
        "results": processed_records
    }


def ingest_kaggle_datasets_full() -> dict:
    """
    Full Ingestion Engine for Kaggle 1mg and 11,000+ Medicine Details.
    Loads entire datasets (CSV, JSON, Parquet) without arbitrary truncations.
    """
    logger.info("Scanning for Kaggle raw datasets (1mg & 11,000+ details)...")

    kaggle_files = (
        glob.glob(os.path.join(DATA_DIRS["bronze_kaggle"], "*.*")) +
        glob.glob(os.path.join(BASE_DIR, "**", "*1mg*.*"), recursive=True) +
        glob.glob(os.path.join(BASE_DIR, "**", "*11000*.*"), recursive=True)
    )

    valid_files = [
        f for f in set(kaggle_files)
        if f.endswith(('.csv', '.json', '.parquet'))
        and not any(x in f for x in ["silver", "reports", ".py"])
    ]

    kaggle_records = []
    for file_path in valid_files:
        try:
            df_k = pd.DataFrame()
            if file_path.endswith('.csv'):
                df_k = pd.read_csv(file_path)
            elif file_path.endswith('.parquet'):
                df_k = pd.read_parquet(file_path)
            elif file_path.endswith('.json'):
                df_k = pd.read_json(file_path)

            if not df_k.empty:
                # Add lineage tracking
                records = df_k.to_dict(orient='records')
                for r in records:
                    r["kaggle_source_file"] = os.path.basename(file_path)
                    r["lineage_hash"] = generate_record_hash(r)
                kaggle_records.extend(records)
                logger.info(f"Loaded {len(records)} full records from Kaggle file: {file_path}")

        except Exception as e:
            logger.error(f"Error reading Kaggle dataset {file_path}: {str(e)}")

    return {
        "metadata": {
            "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "sources": ["Kaggle 1mg Medicine Data", "Kaggle 11,000+ Medicine Details"],
            "discovered_files_count": len(valid_files),
            "total_records_ingested": len(kaggle_records),
            "execution_status": "SUCCESS" if kaggle_records else "NO_KAGGLE_FILES_FOUND"
        },
        "results": kaggle_records
    }

who_payload = ingest_who_eml_strict_no_fallback()
save_raw_json(who_payload, os.path.join(DATA_DIRS["bronze_who"], "who_eml_raw.json"))

kaggle_payload = ingest_kaggle_datasets_full()
save_raw_json(kaggle_payload, os.path.join(DATA_DIRS["bronze_kaggle"], "kaggle_medicines_raw.json"))

print("\n" + "="*80)
print(f"✅ CELL 5 EXECUTED (STRICT RAW DATA INGESTION):")
print(f"   - WHO EML Raw Records: {who_payload['metadata']['total_records']} (Status: {who_payload['metadata']['execution_status']} | Has Synthetic: False)")
print(f"   - Kaggle Full Records: {kaggle_payload['metadata']['total_records_ingested']} (Discovered Files: {kaggle_payload['metadata']['discovered_files_count']})")
print("="*80)


✅ CELL 5 EXECUTED (STRICT RAW DATA INGESTION):
   - WHO EML Raw Records: 0 (Status: NO_WHO_DATA_FOUND | Has Synthetic: False)
   - Kaggle Full Records: 0 (Discovered Files: 0)


In [6]:
import os
import json
import zipfile
import pandas as pd
from datetime import datetime, timezone

logger.info("Executing Honest Enterprise Silver Transformation & Multi-Source Quality Audit Engine...")

# 1. Load Raw Payloads
with open(os.path.join(DATA_DIRS["bronze_fda"], "fda_ndc_raw.json"), 'r', encoding='utf-8') as f:
    fda_ndc_raw = json.load(f)

with open(os.path.join(DATA_DIRS["bronze_fda"], "fda_enforcement_raw.json"), 'r', encoding='utf-8') as f:
    fda_enforcement_raw = json.load(f)

with open(os.path.join(DATA_DIRS["bronze_fda"], "fda_shortages_raw.json"), 'r', encoding='utf-8') as f:
    fda_shortages_raw = json.load(f)

with open(os.path.join(DATA_DIRS["bronze_eda"], "eda_alerts_raw.json"), 'r', encoding='utf-8') as f:
    eda_raw = json.load(f)

with open(os.path.join(DATA_DIRS["bronze_rxnav"], "rxnav_concepts_raw.json"), 'r', encoding='utf-8') as f:
    rxnav_raw = json.load(f)

with open(os.path.join(DATA_DIRS["bronze_who"], "who_eml_raw.json"), 'r', encoding='utf-8') as f:
    who_raw = json.load(f)

with open(os.path.join(DATA_DIRS["bronze_kaggle"], "kaggle_medicines_raw.json"), 'r', encoding='utf-8') as f:
    kaggle_raw = json.load(f)

with open(os.path.join(DATA_DIRS["bronze_rag_corpus"], "drugs_com_pricing_raw.json"), 'r', encoding='utf-8') as f:
    drugs_com_raw = json.load(f)

# 2. Silver Transformations
# FDA NDC
ndc_records_raw = []
for item in fda_ndc_raw.get("results", []):
    active_ings = item.get("active_ingredients", [])
    ing_names = [ing.get("name", "").strip().lower() for ing in active_ings if ing.get("name")]
    full_active_ing_str = " + ".join(ing_names) if ing_names else "UNSPECIFIED"

    ndc_records_raw.append({
        "product_id": item.get("product_id"),
        "product_ndc": item.get("product_ndc"),
        "brand_name": str(item.get("brand_name", "")).strip().upper(),
        "generic_name": str(item.get("generic_name", "")).strip().lower(),
        "active_ingredients_all": full_active_ing_str,
        "dosage_form": item.get("dosage_form", "UNKNOWN"),
        "labeler_name": item.get("labeler_name", "UNKNOWN"),
        "lineage_hash": generate_record_hash(item)
    })

df_fda_raw_ndc = pd.DataFrame(ndc_records_raw)
ndc_pre_duplicates = int(df_fda_raw_ndc.duplicated(subset=["product_id"]).sum()) if not df_fda_raw_ndc.empty else 0
df_silver_fda_ndc = df_fda_raw_ndc.drop_duplicates(subset=["product_id"]) if not df_fda_raw_ndc.empty else pd.DataFrame()

# FDA Drug Shortages
shortages_list = fda_shortages_raw.get("results", [])
df_silver_shortages = pd.DataFrame(shortages_list) if shortages_list else pd.DataFrame(columns=["ingredient", "status"])

# Kaggle Silver Table
kaggle_list = kaggle_raw.get("results", [])
df_silver_kaggle = pd.DataFrame(kaggle_list) if kaggle_list else pd.DataFrame(columns=["kaggle_source_file", "drug_name"])

# EDA & WHO Silver Tables
df_silver_eda = pd.DataFrame(eda_raw.get("results", [])) if eda_raw.get("results") else pd.DataFrame(columns=["alert_id", "alert_title"])
df_silver_who = pd.DataFrame(who_raw.get("results", [])) if who_raw.get("results") else pd.DataFrame(columns=["eml_id", "medicine_name"])

# 3. Cross-Source Integrated Catalog Construction
rxnav_lookup = {r["query_drug_name"].lower(): r.get("rxcui") for r in rxnav_raw.get("results", [])}
pricing_lookup = {r["drug_name"].lower(): r for r in drugs_com_raw.get("results", [])} if drugs_com_raw.get("results") else {}

integrated_records = []
if not df_silver_who.empty:
    for idx, who_item in df_silver_who.iterrows():
        med_name = str(who_item.get("medicine_name", "")).lower()
        rxcui_matched = rxnav_lookup.get(med_name, None)
        pricing_data = pricing_lookup.get(med_name, None)

        fda_matches = df_silver_fda_ndc[
            df_silver_fda_ndc["active_ingredients_all"].str.contains(med_name, na=False, case=False, regex=False)
        ] if not df_silver_fda_ndc.empty else pd.DataFrame()

        integrated_records.append({
            "eml_id": who_item.get("eml_id", f"WHO-LOCAL-{idx+1}"),
            "medicine_name": who_item.get("medicine_name"),
            "rxcui": rxcui_matched,
            "fda_matched_products_count": len(fda_matches),
            "pricing_usd": pricing_data.get("avg_retail_price_usd") if pricing_data else "PRICING_UNAVAILABLE",
            "availability_status": pricing_data.get("availability_status", "UNKNOWN") if pricing_data else "UNKNOWN",
            "last_processed_utc": datetime.now(timezone.utc).isoformat()
        })

df_silver_integrated_catalog = pd.DataFrame(integrated_records)

# 4. Multi-Source Quality Metrics Calculation
def calculate_dataset_quality(df: pd.DataFrame, primary_key: str = None) -> dict:
    if df.empty:
        return {"record_count": 0, "completeness_score_pct": 0.0, "null_values_count": 0, "primary_key_unique": False}
    total_cells = df.size
    null_cells = int(df.isnull().sum().sum())
    completeness = round((1.0 - (null_cells / total_cells)) * 100, 2)
    pk_unique = bool(df[primary_key].is_unique) if primary_key and primary_key in df.columns else None
    return {
        "record_count": len(df),
        "completeness_score_pct": completeness,
        "null_values_count": null_cells,
        "primary_key_unique": pk_unique
    }

# Calculate Quality Scores
quality_report = {
    "execution_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "pre_cleaning_metrics": {
        "fda_ndc_raw_duplicates_purged": ndc_pre_duplicates
    },
    "post_cleaning_metrics": {
        "openfda_ndc": calculate_dataset_quality(df_silver_fda_ndc, "product_id"),
        "openfda_shortages": calculate_dataset_quality(df_silver_shortages),
        "who_eml": calculate_dataset_quality(df_silver_who, "eml_id"),
        "eda_alerts": calculate_dataset_quality(df_silver_eda, "alert_id"),
        "kaggle_medicines": calculate_dataset_quality(df_silver_kaggle),
        "integrated_catalog": calculate_dataset_quality(df_silver_integrated_catalog, "eml_id")
    }
}

# Honest Pipeline Verdict System
active_sources_count = sum([
    not df_silver_fda_ndc.empty,
    not df_silver_shortages.empty,
    not df_silver_eda.empty,
    not df_silver_who.empty,
    not df_silver_kaggle.empty,
    not df_silver_integrated_catalog.empty
])

if active_sources_count >= 5:
    pipeline_verdict = "PASSED_HIGH_COVERAGE"
elif active_sources_count >= 3:
    pipeline_verdict = "PASSED_PARTIAL_COVERAGE"
else:
    pipeline_verdict = "FAILED_INSUFFICIENT_SOURCES"

quality_report["pipeline_verdict"] = pipeline_verdict

# 5. Export Silver Data Lake Artifacts
df_silver_fda_ndc.to_csv(os.path.join(DATA_DIRS["silver"], "silver_fda_ndc.csv"), index=False)
df_silver_shortages.to_csv(os.path.join(DATA_DIRS["silver"], "silver_fda_shortages.csv"), index=False)
df_silver_eda.to_csv(os.path.join(DATA_DIRS["silver"], "silver_eda_alerts.csv"), index=False)
df_silver_who.to_csv(os.path.join(DATA_DIRS["silver"], "silver_who_catalog.csv"), index=False)
df_silver_kaggle.to_csv(os.path.join(DATA_DIRS["silver"], "silver_kaggle_medicines.csv"), index=False)
df_silver_integrated_catalog.to_csv(os.path.join(DATA_DIRS["silver"], "silver_integrated_catalog.csv"), index=False)

with open(os.path.join(DATA_DIRS["reports"], "data_quality_report.json"), 'w', encoding='utf-8') as f:
    json.dump(quality_report, f, ensure_ascii=False, indent=2)

# Save Zip Archive
zip_path = os.path.join(BASE_DIR, "data_lake_silver_bundle.zip")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zip_file:
    for root, _, files in os.walk(DATA_DIRS["silver"]):
        for file in files:
            full_p = os.path.join(root, file)
            zip_file.write(full_p, arcname=os.path.relpath(full_p, BASE_DIR))

print("\n" + "="*80)
print(f"✅ CELL 6 EXECUTED (HONEST PRODUCTION DATA LAKE):")
print(f"   - Pipeline Verdict: {pipeline_verdict} ({active_sources_count} Active Silver Sources Processed)")
print(f"   - FDA NDC Records: {len(df_silver_fda_ndc)}")
print(f"   - FDA Shortages Records: {len(df_silver_shortages)}")
print(f"   - WHO EML Raw Records: {len(df_silver_who)} (Synthetic Data: FALSE)")
print(f"   - Kaggle Records: {len(df_silver_kaggle)}")
print(f"   - Integrated Catalog Records Built: {len(df_silver_integrated_catalog)}")
print(f"   - Output Zip Archive: data_lake_silver_bundle.zip")
print("="*80)


✅ CELL 6 EXECUTED (HONEST PRODUCTION DATA LAKE):
   - Pipeline Verdict: FAILED_INSUFFICIENT_SOURCES (1 Active Silver Sources Processed)
   - FDA NDC Records: 300
   - FDA Shortages Records: 0
   - WHO EML Raw Records: 0 (Synthetic Data: FALSE)
   - Kaggle Records: 0
   - Integrated Catalog Records Built: 0
   - Output Zip Archive: data_lake_silver_bundle.zip


In [7]:

import os
import json
import pandas as pd
from datetime import datetime, timezone

logger.info("Executing Cell 7: Gold Analytics, RAG Vector Corpus Preparation, and Master Execution Audit...")

# Safe CSV Reader Utility preventing EmptyDataError on zero-byte files
def safe_read_csv(filepath: str) -> pd.DataFrame:
    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        try:
            return pd.read_csv(filepath)
        except Exception as e:
            logger.warning(f"Failed to parse CSV {filepath}: {str(e)}")
            return pd.DataFrame()
    return pd.DataFrame()

# 1. Load Cleaned Silver Artifacts Safely
silver_dir = DATA_DIRS["silver"]
reports_dir = DATA_DIRS["reports"]

df_silver_ndc = safe_read_csv(os.path.join(silver_dir, "silver_fda_ndc.csv"))
df_silver_shortages = safe_read_csv(os.path.join(silver_dir, "silver_fda_shortages.csv"))
df_silver_eda = safe_read_csv(os.path.join(silver_dir, "silver_eda_alerts.csv"))
df_silver_who = safe_read_csv(os.path.join(silver_dir, "silver_who_catalog.csv"))
df_silver_kaggle = safe_read_csv(os.path.join(silver_dir, "silver_kaggle_medicines.csv"))
df_integrated_catalog = safe_read_csv(os.path.join(silver_dir, "silver_integrated_catalog.csv"))

# Load Safety Corpus & Price Directory safely from Bronze/RAG Corpus
fda_safety_corpus = []
safety_corpus_path = os.path.join(DATA_DIRS["bronze_rag_corpus"], "fda_safety_corpus.json")
if os.path.exists(safety_corpus_path) and os.path.getsize(safety_corpus_path) > 0:
    try:
        with open(safety_corpus_path, 'r', encoding='utf-8') as f:
            fda_safety_corpus = json.load(f).get("results", [])
    except Exception as e:
        logger.warning(f"Could not load safety corpus: {str(e)}")

# 2. Build Gold Table 1: Master Pharmaceutical Analytics Table
gold_analytics_records = []
if not df_silver_ndc.empty:
    grouped = df_silver_ndc.groupby("active_ingredients_all")
    for ing_name, group in grouped:
        has_who_match = False
        if not df_integrated_catalog.empty and "medicine_name" in df_integrated_catalog.columns:
            has_who_match = bool(any(df_integrated_catalog["medicine_name"].astype(str).str.contains(str(ing_name), case=False, na=False, regex=False)))

        gold_analytics_records.append({
            "active_ingredient": ing_name,
            "total_commercial_products": len(group),
            "labelers_count": group["labeler_name"].nunique() if "labeler_name" in group.columns else 0,
            "dosage_forms": ", ".join(group["dosage_form"].dropna().unique()[:3]) if "dosage_form" in group.columns else "UNKNOWN",
            "has_who_essential_match": has_who_match,
            "last_updated_utc": datetime.now(timezone.utc).isoformat()
        })

df_gold_analytics = pd.DataFrame(gold_analytics_records)

# 3. Build Gold Table 2: Unified RAG Knowledge Corpus
rag_documents = []

if not df_silver_eda.empty:
    for _, row in df_silver_eda.iterrows():
        rag_documents.append({
            "doc_id": f"RAG-EDA-{row.get('alert_id', '000')}",
            "source_type": "EDA_SAFETY_ALERT",
            "content": f"Egyptian Drug Authority Safety Notice: {row.get('alert_title', '')}. Details: {row.get('extracted_text_sample', 'N/A')}",
            "metadata": {"source": "EDA", "file_path": row.get("pdf_path", "N/A")}
        })

for item in fda_safety_corpus:
    rag_documents.append({
        "doc_id": f"RAG-FDA-{item.get('corpus_id', '000')}",
        "source_type": "FDA_SAFETY_RECALL",
        "content": item.get("safety_text", ""),
        "metadata": {"source": "FDA_ENFORCEMENT", "classification": item.get("classification", "UNKNOWN")}
    })

df_gold_rag_corpus = pd.DataFrame(rag_documents)

# 4. Save Gold Layer Artifacts
gold_dir = os.path.join(BASE_DIR, "data", "gold")
os.makedirs(gold_dir, exist_ok=True)

df_gold_analytics.to_csv(os.path.join(gold_dir, "gold_pharma_analytics.csv"), index=False)
df_gold_rag_corpus.to_csv(os.path.join(gold_dir, "gold_rag_knowledge_corpus.csv"), index=False)

# 5. Master 11-Source Execution Audit Summary Report
quality_summary = {}
report_json_path = os.path.join(reports_dir, "data_quality_report.json")
if os.path.exists(report_json_path) and os.path.getsize(report_json_path) > 0:
    try:
        with open(report_json_path, 'r', encoding='utf-8') as f:
            quality_summary = json.load(f)
    except Exception:
        pass

master_execution_audit = {
    "pipeline_title": "Enterprise 11-Source Pharmaceutical Data Pipeline",
    "execution_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "overall_pipeline_verdict": quality_summary.get("pipeline_verdict", "PASSED_PARTIAL_COVERAGE"),
    "sources_ingestion_summary": {
        "1_openfda_ndc": {"status": "SUCCESS", "records": len(df_silver_ndc)},
        "2_openfda_enforcement": {"status": "SUCCESS", "records": quality_summary.get("post_cleaning_metrics", {}).get("openfda_ndc", {}).get("record_count", 0)},
        "3_openfda_shortages": {"status": "SUCCESS", "records": len(df_silver_shortages)},
        "4_openfda_bulk_ndc": {"status": "EXTRACTED_AND_LOGGED"},
        "5_nih_rxnav": {"status": "SUCCESS", "matched_concepts": len(df_integrated_catalog)},
        "6_who_eml": {"status": "SUCCESS_NO_MOCK", "records": len(df_silver_who)},
        "7_eda_safety_notices": {"status": "PROCESSED", "records": len(df_silver_eda)},
        "8_kaggle_1mg": {"status": "INSPECTED", "records": len(df_silver_kaggle)},
        "9_kaggle_11000_details": {"status": "INSPECTED", "records": len(df_silver_kaggle)},
        "10_fda_safety_corpus": {"status": "BUILT_FOR_RAG", "records": len(fda_safety_corpus)},
        "11_drugs_com_pricing": {"status": "AUDITED_NO_MOCK"}
    },
    "gold_layer_outputs": {
        "gold_pharma_analytics_rows": len(df_gold_analytics),
        "gold_rag_corpus_documents": len(df_gold_rag_corpus)
    }
}

with open(os.path.join(reports_dir, "master_pipeline_audit_report.json"), 'w', encoding='utf-8') as f:
    json.dump(master_execution_audit, f, ensure_ascii=False, indent=2)

print("\n" + "="*80)
print("✅ CELL 7 EXECUTED SUCCESSFULLY (PROTECTED AGAINST EMPTY FILES):")
print(f"   - Pipeline Verdict: {master_execution_audit['overall_pipeline_verdict']}")
print(f"   - Gold Pharma Analytics Records: {len(df_gold_analytics)}")
print(f"   - Gold RAG Knowledge Corpus Documents: {len(df_gold_rag_corpus)}")
print(f"   - Master Audit Report Generated: reports/master_pipeline_audit_report.json")
print("="*80)


✅ CELL 7 EXECUTED SUCCESSFULLY (PROTECTED AGAINST EMPTY FILES):
   - Pipeline Verdict: FAILED_INSUFFICIENT_SOURCES
   - Gold Pharma Analytics Records: 227
   - Gold RAG Knowledge Corpus Documents: 150
   - Master Audit Report Generated: reports/master_pipeline_audit_report.json


In [8]:
import os
import json
import pandas as pd
from datetime import datetime, timezone

who_raw_path = os.path.join(DATA_DIRS["bronze_who"], "who_eml_raw.json")
who_csv_path = os.path.join(DATA_DIRS["bronze_who"], "who_eml_official_raw.csv")

synthetic_names = {
    "Amoxicillin", "Ibuprofen", "Metformin", "Paracetamol",
    "Amlodipine", "Omeprazole", "Azithromycin", "Atorvastatin"
}

if os.path.exists(who_csv_path):
    try:
        existing_df = pd.read_csv(who_csv_path)

        is_synthetic = (
            len(existing_df) == 8
            and "medicine_name" in existing_df.columns
            and set(existing_df["medicine_name"].astype(str)) == synthetic_names
            and "eml_id" in existing_df.columns
            and existing_df["eml_id"].astype(str).str.startswith("WHO-EML-").all()
        )

        if is_synthetic:
            os.remove(who_csv_path)
            logger.warning(
                "Previously generated unverified WHO sample data rejected."
            )

    except Exception as e:
        logger.error(f"WHO source validation failed: {e}")

who_payload = {
    "metadata": {
        "source": "WHO Essential Medicines List",
        "verification_status": "NOT_VERIFIED",
        "source_file": None,
        "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "total_records": 0,
        "has_synthetic_data": False,
        "execution_status": "WAITING_FOR_VERIFIED_WHO_SOURCE"
    },
    "results": []
}

if os.path.exists(who_csv_path) and os.path.getsize(who_csv_path) > 0:
    try:
        df = pd.read_csv(who_csv_path)

        required_columns = {"eml_id", "medicine_name"}

        if required_columns.issubset(df.columns) and not df.empty:
            who_payload["results"] = df.to_dict(orient="records")

            who_payload["metadata"].update({
                "verification_status":
                    "STRUCTURE_VALIDATED_SOURCE_NOT_INDEPENDENTLY_VERIFIED",
                "source_file": who_csv_path,
                "total_records": len(df),
                "execution_status": "SOURCE_STRUCTURE_VALIDATED"
            })

    except Exception as e:
        logger.error(f"WHO CSV loading failed: {e}")

save_raw_json(who_payload, who_raw_path)

print("\n" + "=" * 75)
print("WHO SOURCE VALIDATION")
print("=" * 75)
print(f"Records: {who_payload['metadata']['total_records']}")
print(f"Status: {who_payload['metadata']['execution_status']}")
print(f"Source verification: {who_payload['metadata']['verification_status']}")
print("=" * 75)

print("\n✅ CELL 8.1 EXECUTED SUCCESSFULLY (WHO SOURCE VALIDATION COMPLETED)")


WHO SOURCE VALIDATION
Records: 0
Status: WAITING_FOR_VERIFIED_WHO_SOURCE
Source verification: NOT_VERIFIED

✅ CELL 8.1 EXECUTED SUCCESSFULLY (WHO SOURCE VALIDATION COMPLETED)


In [9]:
import os
import json
import pandas as pd
from datetime import datetime, timezone

logger.info("Executing CELL 8.2: WHO-Aware Silver Integration & Gold Rebuild")

# ============================================================
# 1. LOAD WHO VALIDATION OUTPUT FROM CELL 8.1
# ============================================================

who_raw_path = os.path.join(
    DATA_DIRS["bronze_who"],
    "who_eml_raw.json"
)

who_records = []
who_metadata = {}

if os.path.exists(who_raw_path):
    with open(who_raw_path, "r", encoding="utf-8") as f:
        who_payload = json.load(f)

    who_metadata = who_payload.get("metadata", {})
    who_results = who_payload.get("results", [])

    verification_status = who_metadata.get(
        "verification_status",
        "NOT_VERIFIED"
    )

    if (
        verification_status == "STRUCTURE_VALIDATED_SOURCE_NOT_INDEPENDENTLY_VERIFIED"
        and isinstance(who_results, list)
    ):
        required_columns = {"eml_id", "medicine_name"}

        df_who_candidate = pd.DataFrame(who_results)

        if (
            not df_who_candidate.empty
            and required_columns.issubset(df_who_candidate.columns)
        ):
            df_who_candidate["medicine_name"] = (
                df_who_candidate["medicine_name"]
                .astype(str)
                .str.strip()
            )

            df_who_candidate = df_who_candidate[
                df_who_candidate["medicine_name"].ne("")
            ]

            who_records = df_who_candidate.to_dict(orient="records")

else:
    logger.warning("WHO validation output not found. WHO integration skipped.")

df_silver_who = pd.DataFrame(who_records)

# ============================================================
# 2. LOAD FDA AND RXNAV BRONZE DATA
# ============================================================

fda_path = os.path.join(
    DATA_DIRS["bronze_fda"],
    "fda_ndc_raw.json"
)

rxnav_path = os.path.join(
    DATA_DIRS["bronze_rxnav"],
    "rxnav_concepts_raw.json"
)

with open(fda_path, "r", encoding="utf-8") as f:
    fda_ndc_raw = json.load(f)

with open(rxnav_path, "r", encoding="utf-8") as f:
    rxnav_raw = json.load(f)

# ============================================================
# 3. BUILD FDA SILVER DATASET
# ============================================================

ndc_records_raw = []

for item in fda_ndc_raw.get("results", []):

    active_ings = item.get("active_ingredients", [])

    ing_names = [
        str(ing.get("name", "")).strip().lower()
        for ing in active_ings
        if ing.get("name")
    ]

    ndc_records_raw.append({
        "product_id": item.get("product_id"),
        "product_ndc": item.get("product_ndc"),
        "brand_name": str(
            item.get("brand_name", "")
        ).strip().upper(),
        "generic_name": str(
            item.get("generic_name", "")
        ).strip().lower(),
        "active_ingredients_all": (
            " + ".join(ing_names) if ing_names else "UNSPECIFIED"
        ),
        "dosage_form": item.get("dosage_form", "UNKNOWN"),
        "labeler_name": item.get("labeler_name", "UNKNOWN"),
        "lineage_hash": generate_record_hash(item)
    })

df_silver_fda_ndc = (
    pd.DataFrame(ndc_records_raw)
    if ndc_records_raw
    else pd.DataFrame(columns=[
        "product_id",
        "product_ndc",
        "brand_name",
        "generic_name",
        "active_ingredients_all",
        "dosage_form",
        "labeler_name",
        "lineage_hash"
    ])
)

if not df_silver_fda_ndc.empty:
    df_silver_fda_ndc = df_silver_fda_ndc.drop_duplicates(
        subset=["product_id"]
    )

# ============================================================
# 4. BUILD RXNAV LOOKUP
# ============================================================

rxnav_lookup = {}

for record in rxnav_raw.get("results", []):

    drug_name = str(
        record.get("query_drug_name", "")
    ).strip().lower()

    if drug_name:
        rxnav_lookup[drug_name] = record.get("rxcui")

# ============================================================
# 5. BUILD INTEGRATED CATALOG
# ============================================================

integrated_records = []

if not df_silver_who.empty:

    for idx, who_item in df_silver_who.iterrows():

        med_name = str(
            who_item.get("medicine_name", "")
        ).strip().lower()

        rxcui_matched = rxnav_lookup.get(med_name)

        if not df_silver_fda_ndc.empty:

            fda_matches = df_silver_fda_ndc[
                df_silver_fda_ndc[
                    "active_ingredients_all"
                ].str.contains(
                    med_name,
                    na=False,
                    case=False,
                    regex=False
                )
            ]

        else:
            fda_matches = pd.DataFrame()

        integrated_records.append({
            "eml_id": who_item.get("eml_id"),
            "medicine_name": who_item.get("medicine_name"),
            "category": who_item.get("category", "UNKNOWN"),
            "rxcui": rxcui_matched,
            "fda_matched_products_count": len(fda_matches),
            "fda_matched_product_ids": ", ".join(
                fda_matches["product_id"]
                .dropna()
                .astype(str)
                .tolist()[:5]
            ) if not fda_matches.empty else "NONE",
            "who_source_verification": who_metadata.get(
                "verification_status",
                "NOT_VERIFIED"
            ),
            "last_processed_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

df_silver_integrated_catalog = pd.DataFrame(
    integrated_records
)

# ============================================================
# 6. SAVE SILVER DATASETS
# ============================================================

silver_who_path = os.path.join(
    DATA_DIRS["silver"],
    "silver_who_catalog.csv"
)

integrated_path = os.path.join(
    DATA_DIRS["silver"],
    "silver_integrated_catalog.csv"
)

fda_silver_path = os.path.join(
    DATA_DIRS["silver"],
    "silver_fda_ndc.csv"
)

df_silver_who.to_csv(
    silver_who_path,
    index=False
)

df_silver_fda_ndc.to_csv(
    fda_silver_path,
    index=False
)

df_silver_integrated_catalog.to_csv(
    integrated_path,
    index=False
)

# ============================================================
# 7. DATA QUALITY VERDICT
# ============================================================

who_source_verified = (
    who_metadata.get("verification_status") == "VERIFIED"
)

who_structure_validated = (
    who_metadata.get("verification_status")
    == "STRUCTURE_VALIDATED_SOURCE_NOT_INDEPENDENTLY_VERIFIED"
)

if (
    not df_silver_fda_ndc.empty
    and not df_silver_who.empty
    and not df_silver_integrated_catalog.empty
    and who_source_verified
):
    pipeline_verdict = "PASSED_VERIFIED_WHO_INTEGRATION"

elif not df_silver_fda_ndc.empty and df_silver_who.empty:
    pipeline_verdict = "PARTIAL_WHO_SOURCE_UNAVAILABLE"

elif not df_silver_fda_ndc.empty and who_structure_validated:
    pipeline_verdict = "PARTIAL_WHO_SOURCE_NOT_INDEPENDENTLY_VERIFIED"

else:
    pipeline_verdict = "FAILED_DATA_REQUIREMENTS_NOT_MET"

quality_report = {
    "execution_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "pipeline_verdict": pipeline_verdict,
    "who_source_verification": who_metadata.get(
        "verification_status",
        "NOT_VERIFIED"
    ),
    "post_cleaning_metrics": {
        "openfda_ndc": {
            "record_count": len(df_silver_fda_ndc)
        },
        "who_eml": {
            "record_count": len(df_silver_who)
        },
        "integrated_catalog": {
            "record_count": len(df_silver_integrated_catalog)
        }
    }
}

quality_report_path = os.path.join(
    DATA_DIRS["reports"],
    "data_quality_report.json"
)

with open(
    quality_report_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        quality_report,
        f,
        ensure_ascii=False,
        indent=2
    )

# ============================================================
# 8. REBUILD GOLD ANALYTICS
# ============================================================

gold_analytics_records = []

if not df_silver_fda_ndc.empty:

    for ing_name, group in df_silver_fda_ndc.groupby(
        "active_ingredients_all"
    ):

        gold_analytics_records.append({
            "active_ingredient": ing_name,
            "total_commercial_products": len(group),
            "labelers_count": group["labeler_name"].nunique(),
            "has_who_essential_match": False,
            "last_updated_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

df_gold_analytics = pd.DataFrame(
    gold_analytics_records
)

gold_dir = os.path.join(
    BASE_DIR,
    "data",
    "gold"
)

os.makedirs(gold_dir, exist_ok=True)

gold_path = os.path.join(
    gold_dir,
    "gold_pharma_analytics.csv"
)

df_gold_analytics.to_csv(
    gold_path,
    index=False
)

# ============================================================
# 9. MASTER EXECUTION AUDIT
# ============================================================

master_execution_audit = {
    "pipeline_title": "Pharmaceutical Data Pipeline",
    "execution_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "overall_pipeline_verdict": pipeline_verdict,
    "who_source_verification": who_metadata.get(
        "verification_status",
        "NOT_VERIFIED"
    ),
    "sources_summary": {
        "openfda_ndc_records": len(df_silver_fda_ndc),
        "who_eml_records": len(df_silver_who),
        "integrated_catalog_records": len(
            df_silver_integrated_catalog
        )
    },
    "gold_layer_outputs": {
        "gold_pharma_analytics_rows": len(
            df_gold_analytics
        )
    }
}

audit_path = os.path.join(
    DATA_DIRS["reports"],
    "master_pipeline_audit_report.json"
)

with open(
    audit_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        master_execution_audit,
        f,
        ensure_ascii=False,
        indent=2
    )

# ============================================================
# 10. EXECUTION SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("CELL 8.2 EXECUTION SUMMARY")
print("=" * 80)

print(f"Pipeline Verdict: {pipeline_verdict}")

print(
    f"WHO Verification: "
    f"{who_metadata.get('verification_status', 'NOT_VERIFIED')}"
)

print(f"WHO Records: {len(df_silver_who)}")
print(f"FDA Silver Records: {len(df_silver_fda_ndc)}")
print(
    f"Integrated Catalog Records: "
    f"{len(df_silver_integrated_catalog)}"
)
print(f"Gold Analytics Records: {len(df_gold_analytics)}")

print("=" * 80)

print(
    "\n✅ CELL 8.2 EXECUTED SUCCESSFULLY "
    "(WHO-AWARE SILVER INTEGRATION & GOLD REBUILD)"
)


CELL 8.2 EXECUTION SUMMARY
Pipeline Verdict: PARTIAL_WHO_SOURCE_UNAVAILABLE
WHO Verification: NOT_VERIFIED
WHO Records: 0
FDA Silver Records: 300
Integrated Catalog Records: 0
Gold Analytics Records: 227

✅ CELL 8.2 EXECUTED SUCCESSFULLY (WHO-AWARE SILVER INTEGRATION & GOLD REBUILD)


In [10]:
import os
import json
import html
import pandas as pd
from datetime import datetime, timezone

logger.info(
    "Executing CELL 8.3: Data Quality Validation, Parquet Serialization & Audit Dashboard"
)

# ============================================================
# 1. SAFE DATASET READER
# ============================================================

def read_dataset_safely(file_path):
    if not os.path.exists(file_path):
        return pd.DataFrame()

    if os.path.getsize(file_path) == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(file_path)
    except pd.errors.EmptyDataError:
        logger.warning(f"Empty dataset detected: {file_path}")
        return pd.DataFrame()
    except Exception as e:
        logger.error(f"Dataset loading failed: {file_path}: {e}")
        return pd.DataFrame()


# ============================================================
# 2. DATA QUALITY VALIDATION ENGINE
# ============================================================

def validate_gold_data_integrity(df_analytics, df_rag):

    validation_results = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "passed_all_rules": False,
        "overall_status": "FAILED",
        "rules_checked": []
    }

    # Rule 1: Gold Analytics Non-Empty
    rule_1 = not df_analytics.empty

    validation_results["rules_checked"].append({
        "rule_name": "Gold_Analytics_Non_Empty",
        "status": "PASSED" if rule_1 else "FAILED",
        "details": f"Gold Analytics records: {len(df_analytics)}"
    })

    # Rule 2: Primary Key Uniqueness
    rule_2 = False

    if (
        not df_analytics.empty
        and "active_ingredient" in df_analytics.columns
    ):
        key_series = df_analytics["active_ingredient"]

        rule_2 = bool(
            key_series.notna().all()
            and key_series.astype(str).str.strip().ne("").all()
            and key_series.is_unique
        )

    validation_results["rules_checked"].append({
        "rule_name": "Active_Ingredient_Primary_Key_Uniqueness",
        "status": "PASSED" if rule_2 else "FAILED",
        "details": (
            "Primary key is present, non-null and unique."
            if rule_2
            else "Primary key is missing, null, empty or duplicated."
        )
    })

    # Rule 3: RAG Corpus Volume
    rag_count = len(df_rag)
    rule_3 = rag_count >= 50

    validation_results["rules_checked"].append({
        "rule_name": "RAG_Corpus_Volume_Threshold",
        "status": "PASSED" if rule_3 else "WARNING",
        "details": f"RAG records: {rag_count}; target: >= 50."
    })

    # Rule 4: Actual Gold Data Freshness
    rule_4 = False

    timestamp_column = "last_updated_utc"

    if (
        not df_analytics.empty
        and timestamp_column in df_analytics.columns
    ):
        timestamps = pd.to_datetime(
            df_analytics[timestamp_column],
            errors="coerce",
            utc=True
        )

        now_utc = pd.Timestamp.now(tz="UTC")
        age_hours = (
            now_utc - timestamps
        ).dt.total_seconds() / 3600

        rule_4 = bool(
            timestamps.notna().all()
            and (age_hours >= 0).all()
            and (age_hours <= 24).all()
        )

        freshness_details = (
            f"Maximum data age: {age_hours.max():.2f} hours."
            if timestamps.notna().all()
            else "Invalid or missing timestamps detected."
        )

    else:
        freshness_details = (
            "Required last_updated_utc column is missing."
        )

    validation_results["rules_checked"].append({
        "rule_name": "SLA_Data_Freshness_24_Hours",
        "status": "PASSED" if rule_4 else "FAILED",
        "details": freshness_details
    })

    # Required Rules: 1, 2 and 4
    validation_results["passed_all_rules"] = bool(
        rule_1 and rule_2 and rule_4
    )

    validation_results["overall_status"] = (
        "PASSED_WITH_WARNINGS"
        if validation_results["passed_all_rules"] and not rule_3
        else "PASSED"
        if validation_results["passed_all_rules"]
        else "FAILED"
    )

    return validation_results


# ============================================================
# 3. PARQUET SERIALIZATION
# ============================================================

def serialize_data_lake_to_parquet():

    converted_files = []
    skipped_files = []
    failed_files = []

    for layer in ["silver", "gold"]:

        source_dir = os.path.join(
            BASE_DIR, "data", layer
        )

        parquet_dir = os.path.join(
            BASE_DIR, "data", f"{layer}_parquet"
        )

        os.makedirs(parquet_dir, exist_ok=True)

        if not os.path.exists(source_dir):
            continue

        for file_name in sorted(os.listdir(source_dir)):

            if not file_name.endswith(".csv"):
                continue

            csv_path = os.path.join(source_dir, file_name)

            parquet_path = os.path.join(
                parquet_dir,
                file_name.replace(".csv", ".parquet")
            )

            if os.path.getsize(csv_path) == 0:
                skipped_files.append({
                    "file": file_name,
                    "reason": "EMPTY_FILE"
                })
                continue

            try:
                df = pd.read_csv(csv_path)

                if df.empty or len(df.columns) == 0:
                    skipped_files.append({
                        "file": file_name,
                        "reason": "NO_DATA_OR_COLUMNS"
                    })
                    continue

                df.to_parquet(
                    parquet_path,
                    index=False
                )

                converted_files.append({
                    "file": file_name,
                    "parquet_path": parquet_path,
                    "records": len(df)
                })

            except pd.errors.EmptyDataError:
                skipped_files.append({
                    "file": file_name,
                    "reason": "EMPTY_CSV"
                })

            except Exception as e:
                failed_files.append({
                    "file": file_name,
                    "error": str(e)
                })

    status = (
        "FAILED" if failed_files
        else "PARTIAL" if skipped_files
        else "SUCCESS"
    )

    return {
        "status": status,
        "converted_count": len(converted_files),
        "skipped_count": len(skipped_files),
        "failed_count": len(failed_files),
        "converted_files": converted_files,
        "skipped_files": skipped_files,
        "failed_files": failed_files
    }


# ============================================================
# 4. EXECUTE VALIDATION
# ============================================================

gold_analytics_path = os.path.join(
    BASE_DIR,
    "data",
    "gold",
    "gold_pharma_analytics.csv"
)

gold_rag_path = os.path.join(
    BASE_DIR,
    "data",
    "gold",
    "gold_rag_knowledge_corpus.csv"
)

gold_analytics_df = read_dataset_safely(
    gold_analytics_path
)

gold_rag_df = read_dataset_safely(
    gold_rag_path
)

validation_summary = validate_gold_data_integrity(
    gold_analytics_df,
    gold_rag_df
)


# ============================================================
# 5. EXECUTE PARQUET CONVERSION
# ============================================================

parquet_summary = serialize_data_lake_to_parquet()


# ============================================================
# 6. LOAD MASTER AUDIT
# ============================================================

master_audit_file = os.path.join(
    DATA_DIRS["reports"],
    "master_pipeline_audit_report.json"
)

master_audit_data = {}

if os.path.exists(master_audit_file):
    with open(
        master_audit_file,
        "r",
        encoding="utf-8"
    ) as f:
        master_audit_data = json.load(f)


# ============================================================
# 7. GENERATE HTML AUDIT DASHBOARD
# ============================================================

def generate_interactive_html_report(
    master_audit,
    validation_res,
    parquet_res,
    output_path
):

    verdict = html.escape(
        str(master_audit.get(
            "overall_pipeline_verdict", "UNKNOWN"
        ))
    )

    validation_status = html.escape(
        str(validation_res.get("overall_status", "UNKNOWN"))
    )

    status_class = (
        "passed"
        if validation_status.startswith("PASSED")
        else "failed"
    )

    html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Pharmaceutical Data Pipeline Audit</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                background: #f4f6f9;
                color: #333;
                margin: 20px;
            }}
            .container {{
                max-width: 1100px;
                margin: auto;
                background: white;
                padding: 25px;
                border-radius: 10px;
            }}
            h1 {{ color: #1a365d; }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 15px;
            }}
            th, td {{
                padding: 12px;
                border: 1px solid #ddd;
                text-align: left;
            }}
            th {{
                background: #2b6cb0;
                color: white;
            }}
            .passed {{ color: #16803c; font-weight: bold; }}
            .failed {{ color: #c53030; font-weight: bold; }}
            .warning {{ color: #b7791f; font-weight: bold; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Pharmaceutical Data Pipeline Audit Dashboard</h1>

            <p><strong>Pipeline Verdict:</strong> {verdict}</p>
            <p><strong>Validation Status:</strong>
                <span class="{status_class}">{validation_status}</span>
            </p>

            <h2>Data Quality Rules</h2>
            <table>
                <tr>
                    <th>Rule</th>
                    <th>Status</th>
                    <th>Details</th>
                </tr>
    """

    for rule in validation_res.get("rules_checked", []):

        rule_status = str(rule.get("status", "UNKNOWN"))

        css_class = (
            "passed" if rule_status == "PASSED"
            else "warning" if rule_status == "WARNING"
            else "failed"
        )

        html_content += f"""
                <tr>
                    <td>{html.escape(str(rule.get('rule_name', '')))}</td>
                    <td class="{css_class}">
                        {html.escape(rule_status)}
                    </td>
                    <td>{html.escape(str(rule.get('details', '')))}</td>
                </tr>
        """

    html_content += f"""
            </table>

            <h2>Parquet Serialization</h2>
            <p>Status: {html.escape(str(parquet_res['status']))}</p>
            <p>Converted: {parquet_res['converted_count']}</p>
            <p>Skipped: {parquet_res['skipped_count']}</p>
            <p>Failed: {parquet_res['failed_count']}</p>

            <h2>Execution Metadata</h2>
            <p>Generated UTC:
                {datetime.now(timezone.utc).isoformat()}
            </p>
        </div>
    </body>
    </html>
    """

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(html_content)

    logger.info(
        f"Audit Dashboard generated: {output_path}"
    )


html_report_path = os.path.join(
    DATA_DIRS["reports"],
    "enterprise_audit_dashboard.html"
)

generate_interactive_html_report(
    master_audit_data,
    validation_summary,
    parquet_summary,
    html_report_path
)


# ============================================================
# 8. SAVE VALIDATION REPORT
# ============================================================

validation_report_path = os.path.join(
    DATA_DIRS["reports"],
    "cell_8_3_validation_report.json"
)

with open(
    validation_report_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "validation": validation_summary,
            "parquet": parquet_summary,
            "execution_timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat()
        },
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 9. EXECUTION SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("CELL 8.3 EXECUTION SUMMARY")
print("=" * 80)

print(
    f"Validation Status: "
    f"{validation_summary['overall_status']}"
)

print(
    f"Required Rules Passed: "
    f"{validation_summary['passed_all_rules']}"
)

for rule in validation_summary["rules_checked"]:
    print(
        f"- {rule['rule_name']}: {rule['status']}"
    )

print(
    f"Parquet Status: {parquet_summary['status']}"
)

print(
    f"Parquet Converted: {parquet_summary['converted_count']}"
)

print(
    f"Parquet Skipped: {parquet_summary['skipped_count']}"
)

print(
    f"Parquet Failed: {parquet_summary['failed_count']}"
)

print(
    f"Dashboard: {html_report_path}"
)

print("=" * 80)

print(
    "\n✅ CELL 8.3 EXECUTED SUCCESSFULLY "
    "(HONEST DATA QUALITY VALIDATION & PARQUET AUDIT)"
)


CELL 8.3 EXECUTION SUMMARY
Validation Status: PASSED
Required Rules Passed: True
- Gold_Analytics_Non_Empty: PASSED
- Active_Ingredient_Primary_Key_Uniqueness: PASSED
- RAG_Corpus_Volume_Threshold: PASSED
- SLA_Data_Freshness_24_Hours: PASSED
Parquet Status: PARTIAL
Parquet Converted: 3
Parquet Skipped: 5
Parquet Failed: 0
Dashboard: /content/reports/enterprise_audit_dashboard.html

✅ CELL 8.3 EXECUTED SUCCESSFULLY (HONEST DATA QUALITY VALIDATION & PARQUET AUDIT)


In [11]:
print("=" * 70)
print("CELL 8.3 - PARQUET SKIPPED FILES AUDIT")
print("=" * 70)

for item in parquet_summary["skipped_files"]:
    print(f"File: {item['file']}")
    print(f"Reason: {item['reason']}")
    print("-" * 50)

print(f"Total Skipped: {parquet_summary['skipped_count']}")
print("=" * 70)

CELL 8.3 - PARQUET SKIPPED FILES AUDIT
File: silver_eda_alerts.csv
Reason: NO_DATA_OR_COLUMNS
--------------------------------------------------
File: silver_fda_shortages.csv
Reason: NO_DATA_OR_COLUMNS
--------------------------------------------------
File: silver_integrated_catalog.csv
Reason: EMPTY_CSV
--------------------------------------------------
File: silver_kaggle_medicines.csv
Reason: NO_DATA_OR_COLUMNS
--------------------------------------------------
File: silver_who_catalog.csv
Reason: EMPTY_CSV
--------------------------------------------------
Total Skipped: 5


In [12]:
import os
import json
import zipfile
import hashlib
import html
import pandas as pd

from datetime import datetime, timezone

logger.info(
    "Executing CELL 9: Dynamic Data Catalog, Verified Lineage & Master Bundle..."
)

# ============================================================
# 1. PATHS & CONFIGURATION
# ============================================================

silver_dir = DATA_DIRS["silver"]
gold_dir = DATA_DIRS.get(
    "gold",
    os.path.join(BASE_DIR, "data", "gold")
)
reports_dir = DATA_DIRS["reports"]

os.makedirs(reports_dir, exist_ok=True)
os.makedirs(gold_dir, exist_ok=True)

catalog_file = os.path.join(
    reports_dir,
    "DATA_CATALOG_DICTIONARY.md"
)

lineage_file = os.path.join(
    reports_dir,
    "data_lineage_dag.html"
)

manifest_file = os.path.join(
    reports_dir,
    "MASTER_BUNDLE_MANIFEST.json"
)

master_zip_file = os.path.join(
    BASE_DIR,
    "enterprise_master_production_bundle.zip"
)

generation_time = datetime.now(timezone.utc)


# ============================================================
# 2. SAFE DATASET INSPECTION
# ============================================================

def inspect_dataset(file_path):

    result = {
        "file": os.path.basename(file_path),
        "path": file_path,
        "status": "UNKNOWN",
        "records": 0,
        "columns": 0,
        "column_details": [],
        "size_bytes": 0,
        "error": None
    }

    try:

        result["size_bytes"] = os.path.getsize(file_path)

        if result["size_bytes"] == 0:
            result["status"] = "EMPTY_FILE"
            return result

        df = pd.read_csv(file_path)

        result["records"] = len(df)
        result["columns"] = len(df.columns)

        if len(df.columns) == 0:
            result["status"] = "NO_COLUMNS"
            return result

        result["column_details"] = [
            {
                "name": str(col),
                "dtype": str(df[col].dtype),
                "non_null": int(df[col].count())
            }
            for col in df.columns
        ]

        result["status"] = (
            "AVAILABLE" if len(df) > 0 else "EMPTY_DATASET"
        )

    except pd.errors.EmptyDataError:
        result["status"] = "EMPTY_DATASET"

    except Exception as exc:
        result["status"] = "READ_ERROR"
        result["error"] = str(exc)

    return result


# ============================================================
# 3. AUTOMATED DATA CATALOG
# ============================================================

def generate_markdown_data_catalog(
    silver_path,
    gold_path,
    output_path
):

    lines = [
        "# Pharmaceutical Data Lake - Data Catalog",
        "",
        f"Generated: {generation_time.strftime('%Y-%m-%d %H:%M:%S UTC')}",
        "",
        "Architecture: Bronze -> Silver -> Gold",
        "",
        "This catalog describes the CSV datasets physically available "
        "at generation time.",
        "",
        "---",
        ""
    ]

    catalog_summary = {
        "generated_at_utc": generation_time.isoformat(),
        "datasets": [],
        "total_datasets": 0,
        "available_datasets": 0,
        "empty_datasets": 0,
        "error_datasets": 0
    }

    for layer_name, layer_path in [
        ("Silver Layer", silver_path),
        ("Gold Layer", gold_path)
    ]:

        lines.extend([
            f"## {layer_name}",
            ""
        ])

        if not os.path.isdir(layer_path):
            lines.extend([
                "_Directory not found._",
                ""
            ])
            continue

        csv_files = sorted([
            name for name in os.listdir(layer_path)
            if name.lower().endswith(".csv")
            and os.path.isfile(os.path.join(layer_path, name))
        ])

        if not csv_files:
            lines.extend([
                "_No CSV datasets found._",
                ""
            ])
            continue

        for file_name in csv_files:

            file_path = os.path.join(layer_path, file_name)
            info = inspect_dataset(file_path)

            info["layer"] = layer_name
            catalog_summary["datasets"].append(info)

            lines.extend([
                f"### Dataset: `{file_name}`",
                "",
                f"- Status: **{info['status']}**",
                f"- Records: {info['records']:,}",
                f"- Columns: {info['columns']:,}",
                f"- File Size: {info['size_bytes'] / 1024:.2f} KB",
                ""
            ])

            if info["column_details"]:

                lines.extend([
                    "| Column | Data Type | Non-Null Count |",
                    "|---|---|---:|"
                ])

                for column in info["column_details"]:

                    col_name = (
                        column["name"]
                        .replace("|", "\\|")
                        .replace("`", "\\`")
                    )

                    lines.append(
                        f"| `{col_name}` "
                        f"| `{column['dtype']}` "
                        f"| {column['non_null']:,} |"
                    )

            else:

                lines.append("_No column metadata available._")

            lines.extend(["", "---", ""])

    catalog_summary["total_datasets"] = len(
        catalog_summary["datasets"]
    )

    catalog_summary["available_datasets"] = sum(
        d["status"] == "AVAILABLE"
        for d in catalog_summary["datasets"]
    )

    catalog_summary["empty_datasets"] = sum(
        d["status"] in ["EMPTY_FILE", "EMPTY_DATASET", "NO_COLUMNS"]
        for d in catalog_summary["datasets"]
    )

    catalog_summary["error_datasets"] = sum(
        d["status"] == "READ_ERROR"
        for d in catalog_summary["datasets"]
    )

    lines.extend([
        "## Catalog Summary",
        "",
        f"- Total datasets: {catalog_summary['total_datasets']}",
        f"- Available datasets: {catalog_summary['available_datasets']}",
        f"- Empty datasets: {catalog_summary['empty_datasets']}",
        f"- Read errors: {catalog_summary['error_datasets']}",
        "",
        "> Dataset availability does not independently verify source "
        "authenticity or data correctness.",
        ""
    ])

    with open(output_path, "w", encoding="utf-8") as file:
        file.write("\n".join(lines))

    logger.info("Data catalog generated: %s", output_path)

    return catalog_summary


# ============================================================
# 4. DYNAMIC DATA LINEAGE INVENTORY
# ============================================================

def discover_csv_datasets(folder_path):

    datasets = []

    if not os.path.isdir(folder_path):
        return datasets

    for root, _, files in os.walk(folder_path):

        for file_name in sorted(files):

            if not file_name.lower().endswith(".csv"):
                continue

            file_path = os.path.join(root, file_name)

            datasets.append(
                inspect_dataset(file_path)
            )

    return datasets


def generate_interactive_lineage_graph(
    output_path,
    bronze_datasets,
    silver_datasets,
    gold_datasets
):

    layers = [
        {
            "name": "Bronze",
            "description": "Raw source artifacts",
            "datasets": bronze_datasets
        },
        {
            "name": "Silver",
            "description": "Cleansed and transformed datasets",
            "datasets": silver_datasets
        },
        {
            "name": "Gold",
            "description": "Analytics and serving datasets",
            "datasets": gold_datasets
        }
    ]

    def render_layer(layer):

        cards = ""

        if not layer["datasets"]:
            cards = '<div class="empty">No datasets discovered</div>'

        for dataset in layer["datasets"]:

            status = html.escape(dataset["status"])
            name = html.escape(dataset["file"])

            cards += f"""
            <div class="dataset">
                <strong>{name}</strong>
                <span class="status">{status}</span>
                <small>
                    Records: {dataset['records']:,}
                    | Columns: {dataset['columns']:,}
                </small>
            </div>
            """

        return f"""
        <section class="layer">
            <h2>{html.escape(layer['name'])}</h2>
            <p>{html.escape(layer['description'])}</p>
            {cards}
        </section>
        """

    layer_html = "".join(
        render_layer(layer)
        for layer in layers
    )

    total_datasets = sum(
        len(layer["datasets"])
        for layer in layers
    )

    available_datasets = sum(
        dataset["status"] == "AVAILABLE"
        for layer in layers
        for dataset in layer["datasets"]
    )

    page = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Pharmaceutical Data Lineage</title>

        <style>
            body {{
                margin: 0;
                padding: 30px;
                background: #0f172a;
                color: #f8fafc;
                font-family: Arial, sans-serif;
            }}

            h1 {{
                text-align: center;
                color: #60a5fa;
            }}

            .summary {{
                text-align: center;
                margin: 25px 0;
                color: #cbd5e1;
            }}

            .container {{
                display: grid;
                grid-template-columns: repeat(3, minmax(0, 1fr));
                gap: 20px;
                align-items: start;
            }}

            .layer {{
                background: #1e293b;
                border: 1px solid #475569;
                border-radius: 12px;
                padding: 18px;
                min-width: 0;
            }}

            .layer h2 {{
                color: #60a5fa;
                border-bottom: 1px solid #475569;
                padding-bottom: 12px;
            }}

            .dataset {{
                background: #334155;
                border-radius: 8px;
                padding: 12px;
                margin: 10px 0;
                overflow-wrap: anywhere;
            }}

            .status {{
                display: block;
                margin-top: 6px;
                color: #86efac;
                font-size: 12px;
            }}

            .dataset small {{
                display: block;
                margin-top: 7px;
                color: #cbd5e1;
            }}

            .empty {{
                color: #fbbf24;
                padding: 15px;
            }}

            .note {{
                margin-top: 25px;
                color: #cbd5e1;
                font-size: 13px;
                line-height: 1.6;
            }}

            @media(max-width: 900px) {{
                .container {{
                    grid-template-columns: 1fr;
                }}
            }}
        </style>
    </head>

    <body>

        <h1>Pharmaceutical Data Lake Lineage</h1>

        <div class="summary">
            Generated: {generation_time.strftime('%Y-%m-%d %H:%M:%S UTC')}
            <br>
            Discovered datasets: {total_datasets}
            | Available datasets: {available_datasets}
        </div>

        <div class="container">
            {layer_html}
        </div>

        <div class="note">
            <strong>Lineage limitation:</strong>
            This visualization inventories datasets found in each layer.
            It does not claim verified dataset-to-dataset dependencies.
            Exact transformation lineage requires recorded pipeline
            execution metadata.
        </div>

    </body>
    </html>
    """

    with open(output_path, "w", encoding="utf-8") as file:
        file.write(page)

    logger.info("Dynamic lineage inventory generated: %s", output_path)


# ============================================================
# 5. MASTER BUNDLE MANIFEST
# ============================================================

def build_bundle_manifest():

    folders_to_include = []

    # Include Bronze source folders dynamically.
    for key, path in DATA_DIRS.items():

        if key.lower().startswith("bronze") and os.path.isdir(path):
            folders_to_include.append(path)

    # Include standard data layers.
    standard_folders = [
        silver_dir,
        gold_dir,
        os.path.join(BASE_DIR, "data", "silver_parquet"),
        os.path.join(BASE_DIR, "data", "gold_parquet"),
        reports_dir
    ]

    folders_to_include.extend(standard_folders)

    # Remove duplicate paths.
    folders_to_include = list(dict.fromkeys(
        os.path.abspath(path)
        for path in folders_to_include
        if path and os.path.isdir(path)
    ))

    manifest = {
        "bundle_name": "Pharmaceutical Data Engineering Pipeline",
        "generated_at_utc": generation_time.isoformat(),
        "base_directory": os.path.abspath(BASE_DIR),
        "included_directories": [],
        "included_files": [],
        "total_files": 0,
        "total_size_bytes": 0
    }

    for folder in folders_to_include:

        manifest["included_directories"].append(
            os.path.relpath(folder, BASE_DIR)
        )

        for root, _, files in os.walk(folder):

            for file_name in sorted(files):

                full_path = os.path.join(root, file_name)

                if not os.path.isfile(full_path):
                    continue

                if os.path.abspath(full_path) == os.path.abspath(
                    master_zip_file
                ):
                    continue

                relative_path = os.path.relpath(
                    full_path,
                    BASE_DIR
                )

                file_size = os.path.getsize(full_path)

                manifest["included_files"].append({
                    "path": relative_path,
                    "size_bytes": file_size
                })

                manifest["total_size_bytes"] += file_size

    manifest["total_files"] = len(
        manifest["included_files"]
    )

    with open(manifest_file, "w", encoding="utf-8") as file:
        json.dump(
            manifest,
            file,
            indent=4,
            ensure_ascii=False
        )

    return manifest


# ============================================================
# 6. MASTER ZIP EXPORTER
# ============================================================

def build_master_production_deliverable(
    zip_output_path,
    manifest
):

    added_files = []
    missing_files = []

    with zipfile.ZipFile(
        zip_output_path,
        mode="w",
        compression=zipfile.ZIP_DEFLATED
    ) as zip_file:

        for item in manifest["included_files"]:

            full_path = os.path.join(
                BASE_DIR,
                item["path"]
            )

            if not os.path.isfile(full_path):
                missing_files.append(item["path"])
                continue

            zip_file.write(
                full_path,
                arcname=item["path"]
            )

            added_files.append(item["path"])

    zip_size_mb = (
        os.path.getsize(zip_output_path) / (1024 * 1024)
    )

    return {
        "zip_path": zip_output_path,
        "zip_size_mb": zip_size_mb,
        "added_files": len(added_files),
        "missing_files": missing_files,
        "status": (
            "SUCCESS" if not missing_files else "PARTIAL"
        )
    }


# ============================================================
# 7. EXECUTE CELL 9
# ============================================================

bronze_dirs = [
    path
    for key, path in DATA_DIRS.items()
    if key.lower().startswith("bronze")
    and os.path.isdir(path)
]

bronze_datasets = []

for folder in bronze_dirs:
    bronze_datasets.extend(
        discover_csv_datasets(folder)
    )

silver_datasets = discover_csv_datasets(silver_dir)
gold_datasets = discover_csv_datasets(gold_dir)

catalog_summary = generate_markdown_data_catalog(
    silver_dir,
    gold_dir,
    catalog_file
)

generate_interactive_lineage_graph(
    lineage_file,
    bronze_datasets,
    silver_datasets,
    gold_datasets
)

bundle_manifest = build_bundle_manifest()

bundle_result = build_master_production_deliverable(
    master_zip_file,
    bundle_manifest
)


# ============================================================
# 8. EXECUTION SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("CELL 9 EXECUTION SUMMARY")
print("=" * 75)

print(f"Bronze CSV Datasets: {len(bronze_datasets)}")
print(f"Silver CSV Datasets: {len(silver_datasets)}")
print(f"Gold CSV Datasets: {len(gold_datasets)}")

print("-" * 75)

print(
    f"Catalog Available Datasets: "
    f"{catalog_summary['available_datasets']}"
)

print(
    f"Catalog Empty Datasets: "
    f"{catalog_summary['empty_datasets']}"
)

print(
    f"Catalog Read Errors: "
    f"{catalog_summary['error_datasets']}"
)

print("-" * 75)

print(f"Bundle Files: {bundle_result['added_files']}")
print(f"Bundle Size: {bundle_result['zip_size_mb']:.2f} MB")
print(f"Bundle Status: {bundle_result['status']}")

if bundle_result["missing_files"]:
    print("Missing Files:")
    for missing in bundle_result["missing_files"]:
        print(f" - {missing}")

print("-" * 75)
print(f"Data Catalog: {catalog_file}")
print(f"Lineage HTML: {lineage_file}")
print(f"Bundle Manifest: {manifest_file}")
print(f"Master ZIP: {master_zip_file}")

print("=" * 75)

print(
    "\n✅ CELL 9 EXECUTED SUCCESSFULLY "
    "(DYNAMIC DATA CATALOG, LINEAGE INVENTORY & MASTER BUNDLE)"
)


CELL 9 EXECUTION SUMMARY
Bronze CSV Datasets: 0
Silver CSV Datasets: 6
Gold CSV Datasets: 2
---------------------------------------------------------------------------
Catalog Available Datasets: 3
Catalog Empty Datasets: 5
Catalog Read Errors: 0
---------------------------------------------------------------------------
Bundle Files: 28
Bundle Size: 26.77 MB
Bundle Status: SUCCESS
---------------------------------------------------------------------------
Data Catalog: /content/reports/DATA_CATALOG_DICTIONARY.md
Lineage HTML: /content/reports/data_lineage_dag.html
Bundle Manifest: /content/reports/MASTER_BUNDLE_MANIFEST.json
Master ZIP: /content/enterprise_master_production_bundle.zip

✅ CELL 9 EXECUTED SUCCESSFULLY (DYNAMIC DATA CATALOG, LINEAGE INVENTORY & MASTER BUNDLE)


In [13]:
import os
import zipfile
from collections import Counter

print("=" * 70)
print("FINAL PIPELINE AUDIT")
print("=" * 70)

# 1. Bronze file inventory
print("\n[1] BRONZE FILE INVENTORY")

for key, path in DATA_DIRS.items():
    if key.lower().startswith("bronze") and os.path.isdir(path):
        files = [
            f for f in os.listdir(path)
            if os.path.isfile(os.path.join(path, f))
        ]

        extensions = Counter(
            os.path.splitext(f)[1].lower() or "[NO EXTENSION]"
            for f in files
        )

        print(f"\n{key}: {len(files)} files")
        print(f"Extensions: {dict(extensions)}")

# 2. ZIP integrity
print("\n[2] MASTER ZIP INTEGRITY")

with zipfile.ZipFile(master_zip_file, "r") as z:
    bad_file = z.testzip()
    zip_files = z.namelist()

    print(f"ZIP Files: {len(zip_files)}")
    print(f"ZIP Integrity: {'PASSED' if bad_file is None else 'FAILED'}")

    if bad_file:
        print(f"Corrupted File: {bad_file}")

# 3. Synthetic data indicators
print("\n[3] SYNTHETIC DATA INDICATORS")

for root, _, files in os.walk(BASE_DIR):
    if "bronze" not in root.lower() and "silver" not in root.lower():
        continue

    for file in files:
        if file.endswith(".csv") or file.endswith(".json"):
            path = os.path.join(root, file)
            print(f"Artifact: {os.path.relpath(path, BASE_DIR)}")

print("\n" + "=" * 70)
print("FINAL PIPELINE AUDIT COMPLETED")
print("=" * 70)

FINAL PIPELINE AUDIT

[1] BRONZE FILE INVENTORY

bronze_fda: 5 files
Extensions: {'.json': 5}

bronze_rxnav: 1 files
Extensions: {'.json': 1}

bronze_who: 1 files
Extensions: {'.json': 1}

bronze_kaggle: 1 files
Extensions: {'.json': 1}

bronze_eda: 1 files
Extensions: {'.json': 1}

bronze_eda_pdfs: 0 files
Extensions: {}

bronze_rag_corpus: 2 files
Extensions: {'.json': 2}

[2] MASTER ZIP INTEGRITY
ZIP Files: 28
ZIP Integrity: PASSED

[3] SYNTHETIC DATA INDICATORS
Artifact: data/silver/silver_who_catalog.csv
Artifact: data/silver/silver_fda_ndc.csv
Artifact: data/silver/silver_eda_alerts.csv
Artifact: data/silver/silver_kaggle_medicines.csv
Artifact: data/silver/silver_integrated_catalog.csv
Artifact: data/silver/silver_fda_shortages.csv
Artifact: data/bronze/rxnav/rxnav_concepts_raw.json
Artifact: data/bronze/who/who_eml_raw.json
Artifact: data/bronze/rag_corpus/drugs_com_pricing_raw.json
Artifact: data/bronze/rag_corpus/fda_safety_corpus.json
Artifact: data/bronze/eda/eda_alerts_raw

In [14]:
import os
import json

print("=" * 70)
print("BRONZE JSON CONTENT AUDIT")
print("=" * 70)

for key, folder in DATA_DIRS.items():

    if not key.lower().startswith("bronze"):
        continue

    if not os.path.isdir(folder):
        continue

    for file_name in sorted(os.listdir(folder)):

        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(folder, file_name)

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            if isinstance(data, list):
                record_count = len(data)
                data_type = "LIST"

            elif isinstance(data, dict):
                results = data.get("results")

                if isinstance(results, list):
                    record_count = len(results)
                else:
                    record_count = "NOT_AVAILABLE"

                data_type = "DICT"

            else:
                record_count = "NOT_AVAILABLE"
                data_type = type(data).__name__

            print(f"\nSource: {key}")
            print(f"File: {file_name}")
            print(f"Type: {data_type}")
            print(f"Records: {record_count}")
            print(f"Size: {os.path.getsize(file_path):,} bytes")

            if isinstance(data, dict):
                print(f"Top-level Keys: {list(data.keys())[:15]}")

        except Exception as e:
            print(f"\nSource: {key}")
            print(f"File: {file_name}")
            print(f"ERROR: {e}")

print("\n" + "=" * 70)
print("✅ BRONZE JSON CONTENT AUDIT COMPLETED")
print("=" * 70)

BRONZE JSON CONTENT AUDIT

Source: bronze_fda
File: drug-ndc-0001-of-0001.json
Type: DICT
Records: 138173
Size: 243,337,316 bytes
Top-level Keys: ['meta', 'results']

Source: bronze_fda
File: fda_bulk_download_raw.json
Type: DICT
Records: NOT_AVAILABLE
Size: 343 bytes
Top-level Keys: ['metadata']

Source: bronze_fda
File: fda_enforcement_raw.json
Type: DICT
Records: 150
Size: 249,752 bytes
Top-level Keys: ['metadata', 'results']

Source: bronze_fda
File: fda_ndc_raw.json
Type: DICT
Records: 300
Size: 559,670 bytes
Top-level Keys: ['metadata', 'results']

Source: bronze_fda
File: fda_shortages_raw.json
Type: DICT
Records: 0
Size: 368 bytes
Top-level Keys: ['metadata', 'results']

Source: bronze_rxnav
File: rxnav_concepts_raw.json
Type: DICT
Records: 30
Size: 7,423 bytes
Top-level Keys: ['metadata', 'results']

Source: bronze_who
File: who_eml_raw.json
Type: DICT
Records: 0
Size: 336 bytes
Top-level Keys: ['metadata', 'results']

Source: bronze_kaggle
File: kaggle_medicines_raw.json
Type

In [15]:
import os
import json
import hashlib
import pandas as pd

print("=" * 75)
print("CELL 10 - SOURCE INTEGRITY & PROVENANCE AUDIT")
print("=" * 75)

# --------------------------------------------------
# 1. Load JSON safely
# --------------------------------------------------

def load_json_file(path):

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"READ ERROR: {path} | {e}")
        return None


# --------------------------------------------------
# 2. Inspect actual record structures
# --------------------------------------------------

audit_files = {
    "FDA_BULK_NDC": os.path.join(
        DATA_DIRS["bronze_fda"],
        "drug-ndc-0001-of-0001.json"
    ),
    "FDA_NDC": os.path.join(
        DATA_DIRS["bronze_fda"],
        "fda_ndc_raw.json"
    ),
    "FDA_ENFORCEMENT": os.path.join(
        DATA_DIRS["bronze_fda"],
        "fda_enforcement_raw.json"
    ),
    "FDA_SAFETY_CORPUS": os.path.join(
        DATA_DIRS["bronze_rag_corpus"],
        "fda_safety_corpus.json"
    ),
    "RXNAV": os.path.join(
        DATA_DIRS["bronze_rxnav"],
        "rxnav_concepts_raw.json"
    ),
    "WHO": os.path.join(
        DATA_DIRS["bronze_who"],
        "who_eml_raw.json"
    )
}

loaded_data = {}

for source_name, path in audit_files.items():

    data = load_json_file(path)
    loaded_data[source_name] = data

    print(f"\n[{source_name}]")

    if not isinstance(data, dict):
        print("Status: INVALID_OR_UNAVAILABLE")
        continue

    records = data.get("results", [])

    if not isinstance(records, list):
        records = []

    print(f"Record Count: {len(records)}")
    print(f"Metadata: {data.get('metadata', {})}")

    if records and isinstance(records[0], dict):
        print(f"Record Fields: {list(records[0].keys())[:25]}")
        print("First Record Sample:")

        sample = {
            k: str(v)[:100]
            for k, v in list(records[0].items())[:8]
        }

        print(sample)


# --------------------------------------------------
# 3. Compare FDA Enforcement and Safety Corpus
# --------------------------------------------------

print("\n" + "-" * 75)
print("[FDA ENFORCEMENT vs FDA SAFETY CORPUS]")

enforcement = loaded_data.get("FDA_ENFORCEMENT")
safety = loaded_data.get("FDA_SAFETY_CORPUS")

if isinstance(enforcement, dict) and isinstance(safety, dict):

    enforcement_records = enforcement.get("results", [])
    safety_records = safety.get("results", [])

    def canonical_hash(record):

        raw = json.dumps(
            record,
            sort_keys=True,
            ensure_ascii=False,
            default=str
        )

        return hashlib.sha256(
            raw.encode("utf-8")
        ).hexdigest()

    enforcement_hashes = {
        canonical_hash(x)
        for x in enforcement_records
        if isinstance(x, dict)
    }

    safety_hashes = {
        canonical_hash(x)
        for x in safety_records
        if isinstance(x, dict)
    }

    overlap = enforcement_hashes.intersection(safety_hashes)

    print(f"Enforcement Records: {len(enforcement_records)}")
    print(f"Safety Corpus Records: {len(safety_records)}")
    print(f"Exact Matching Records: {len(overlap)}")

    if enforcement_hashes:
        print(
            f"Overlap vs Enforcement: "
            f"{len(overlap) / len(enforcement_hashes) * 100:.2f}%"
        )

    if safety_hashes:
        print(
            f"Overlap vs Safety Corpus: "
            f"{len(overlap) / len(safety_hashes) * 100:.2f}%"
        )

else:
    print("Comparison unavailable.")


# --------------------------------------------------
# 4. Check known synthetic WHO indicators
# --------------------------------------------------

print("\n" + "-" * 75)
print("[WHO SYNTHETIC INDICATOR CHECK]")

who_data = loaded_data.get("WHO")
who_records = (
    who_data.get("results", [])
    if isinstance(who_data, dict)
    else []
)

known_names = {
    "amoxicillin",
    "ibuprofen",
    "metformin",
    "paracetamol",
    "amlodipine",
    "omeprazole",
    "azithromycin",
    "atorvastatin"
}

matches = []

for record in who_records:

    if not isinstance(record, dict):
        continue

    name = str(record.get("medicine_name", "")).strip().lower()
    eml_id = str(record.get("eml_id", "")).strip().upper()

    if name in known_names or eml_id.startswith("WHO-EML-"):
        matches.append({
            "medicine_name": name,
            "eml_id": eml_id
        })

print(f"WHO Records: {len(who_records)}")
print(f"Known Synthetic Indicators: {len(matches)}")

for item in matches:
    print(item)


# --------------------------------------------------
# 5. Final audit summary
# --------------------------------------------------

print("\n" + "=" * 75)
print("CELL 10 AUDIT SUMMARY")
print("=" * 75)

for source_name, data in loaded_data.items():

    if isinstance(data, dict):
        records = data.get("results", [])
        count = len(records) if isinstance(records, list) else 0
        print(f"{source_name}: {count} records")
    else:
        print(f"{source_name}: UNAVAILABLE")

print("\nExact FDA Dataset Overlap: Review results above")
print("WHO Source Verification: NOT ESTABLISHED BY THIS AUDIT")

print("=" * 75)
print("✅ CELL 10.1 EXECUTED SUCCESSFULLY (SOURCE INTEGRITY AUDIT)")

CELL 10 - SOURCE INTEGRITY & PROVENANCE AUDIT

[FDA_BULK_NDC]
Record Count: 138173
Metadata: {}
Record Fields: ['product_ndc', 'generic_name', 'labeler_name', 'brand_name', 'active_ingredients', 'finished', 'packaging', 'listing_expiration_date', 'openfda', 'marketing_category', 'dosage_form', 'spl_id', 'product_type', 'route', 'marketing_start_date', 'product_id', 'application_number', 'brand_name_base', 'pharm_class']
First Record Sample:
{'product_ndc': '71101-029', 'generic_name': 'Lidocaine HCI', 'labeler_name': 'Veridian Healthcare', 'brand_name': 'HealthWise Roll-On Lidocaine Pain Relief Liquid Lavender Scent', 'active_ingredients': "[{'name': 'LIDOCAINE', 'strength': '4 g/mL'}]", 'finished': 'True', 'packaging': "[{'package_ndc': '71101-029-01', 'description': '1 BOTTLE, WITH APPLICATOR in 1 CARTON (71101-029-01", 'listing_expiration_date': '20261231'}

[FDA_NDC]
Record Count: 300
Metadata: {'ingestion_timestamp_utc': '2026-11-02T15:49:04.614456+00:00', 'source_endpoint': 'http

In [16]:

# ============================================================================
# CELL 10.2 - ADVANCED SOURCE INTEGRITY & LINEAGE VALIDATION
# ============================================================================
# Objectives:
# 1. Validate FDA Bulk NDC structure and data quality.
# 2. Compare FDA Enforcement with FDA Safety Corpus using recall identifiers.
# 3. Validate source metadata and record consistency.
# 4. Generate a machine-readable audit report.
#
# IMPORTANT:
# - Read-only audit.
# - Does NOT modify Bronze, Silver, or Gold source files.
# ============================================================================

import os
import json
import hashlib
from datetime import datetime, timezone
from collections import Counter

# ----------------------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------------------

BASE_DIR = "/content"

AUDIT_DIR = os.path.join(BASE_DIR, "reports")
os.makedirs(AUDIT_DIR, exist_ok=True)

SOURCE_FILES = {
    "FDA_BULK_NDC": "drug-ndc-0001-of-0001.json",
    "FDA_NDC": "fda_ndc_raw.json",
    "FDA_ENFORCEMENT": "fda_enforcement_raw.json",
    "FDA_SAFETY_CORPUS": "fda_safety_corpus.json",
    "RXNAV": "rxnav_concepts_raw.json",
    "WHO": "who_eml_raw.json"
}

# ----------------------------------------------------------------------------
# 2. FILE DISCOVERY
# ----------------------------------------------------------------------------

def find_source_file(filename):
    search_roots = [
        os.path.join(BASE_DIR, "data", "bronze"),
        os.path.join(BASE_DIR, "data"),
        BASE_DIR
    ]

    for root in search_roots:
        if not os.path.exists(root):
            continue

        for current_root, dirs, files in os.walk(root):
            # Avoid searching unnecessary folders
            dirs[:] = [
                d for d in dirs
                if d not in ["reports", "silver", "gold",
                             "silver_parquet", "gold_parquet"]
            ]

            if filename in files:
                return os.path.join(current_root, filename)

    return None


# ----------------------------------------------------------------------------
# 3. SAFE JSON LOADER
# ----------------------------------------------------------------------------

def load_json_source(source_name, filename):

    path = find_source_file(filename)

    if path is None:
        return {
            "source_name": source_name,
            "file_path": None,
            "file_found": False,
            "valid_json": False,
            "records": [],
            "metadata": {},
            "error": "SOURCE_FILE_NOT_FOUND"
        }

    try:
        with open(path, "r", encoding="utf-8") as file:
            payload = json.load(file)

        if not isinstance(payload, dict):
            raise ValueError("JSON root must be an object")

        records = payload.get("results", [])

        if records is None:
            records = []

        if not isinstance(records, list):
            raise ValueError("'results' must be a list")

        metadata = payload.get("metadata", {})

        if not isinstance(metadata, dict):
            metadata = {}

        return {
            "source_name": source_name,
            "file_path": path,
            "file_found": True,
            "valid_json": True,
            "records": records,
            "metadata": metadata,
            "error": None
        }

    except Exception as error:
        return {
            "source_name": source_name,
            "file_path": path,
            "file_found": True,
            "valid_json": False,
            "records": [],
            "metadata": {},
            "error": str(error)
        }


# ----------------------------------------------------------------------------
# 4. LOAD ALL SOURCES
# ----------------------------------------------------------------------------

sources = {}

for source_name, filename in SOURCE_FILES.items():
    sources[source_name] = load_json_source(source_name, filename)


# ----------------------------------------------------------------------------
# 5. GENERAL SOURCE INTEGRITY REPORT
# ----------------------------------------------------------------------------

source_integrity = {}

for source_name, source in sources.items():

    records = source["records"]
    metadata = source["metadata"]

    source_integrity[source_name] = {
        "file_found": source["file_found"],
        "valid_json": source["valid_json"],
        "record_count": len(records),
        "metadata_available": bool(metadata),
        "metadata_record_count": metadata.get(
            "total_records",
            metadata.get("retrieved_count")
        ),
        "file_path": source["file_path"],
        "error": source["error"]
    }


# ----------------------------------------------------------------------------
# 6. FDA BULK NDC DATA QUALITY
# ----------------------------------------------------------------------------

bulk_records = sources["FDA_BULK_NDC"]["records"]

required_bulk_fields = [
    "product_ndc",
    "generic_name",
    "labeler_name",
    "product_type"
]

bulk_quality = {
    "total_records": len(bulk_records),
    "required_fields": required_bulk_fields,
    "missing_field_counts": {},
    "missing_product_ndc": 0,
    "duplicate_product_ndc": 0,
    "unique_product_ndc": 0,
    "invalid_record_types": 0
}

bulk_ndc_values = []

for record in bulk_records:

    if not isinstance(record, dict):
        bulk_quality["invalid_record_types"] += 1
        continue

    for field in required_bulk_fields:
        value = record.get(field)

        if value is None or str(value).strip() == "":
            bulk_quality["missing_field_counts"][field] = (
                bulk_quality["missing_field_counts"].get(field, 0) + 1
            )

    product_ndc = record.get("product_ndc")

    if product_ndc is None or str(product_ndc).strip() == "":
        bulk_quality["missing_product_ndc"] += 1
    else:
        bulk_ndc_values.append(str(product_ndc).strip())

bulk_ndc_counter = Counter(bulk_ndc_values)

bulk_quality["unique_product_ndc"] = len(bulk_ndc_counter)

bulk_quality["duplicate_product_ndc"] = sum(
    count - 1
    for count in bulk_ndc_counter.values()
    if count > 1
)


# ----------------------------------------------------------------------------
# 7. FDA ENFORCEMENT vs SAFETY CORPUS LINEAGE
# ----------------------------------------------------------------------------

enforcement_records = sources["FDA_ENFORCEMENT"]["records"]
safety_records = sources["FDA_SAFETY_CORPUS"]["records"]

def normalize_identifier(value):
    if value is None:
        return None

    value = str(value).strip().upper()

    return value if value else None


enforcement_recall_ids = set()
safety_recall_ids = set()

for record in enforcement_records:
    if isinstance(record, dict):
        identifier = normalize_identifier(
            record.get("recall_number")
        )

        if identifier:
            enforcement_recall_ids.add(identifier)


for record in safety_records:
    if isinstance(record, dict):
        identifier = normalize_identifier(
            record.get("source_recall_number")
        )

        if identifier:
            safety_recall_ids.add(identifier)


matched_recall_ids = (
    enforcement_recall_ids.intersection(safety_recall_ids)
)

unmatched_safety_ids = (
    safety_recall_ids - enforcement_recall_ids
)

unmatched_enforcement_ids = (
    enforcement_recall_ids - safety_recall_ids
)

if safety_recall_ids:
    safety_lineage_match_percentage = (
        len(matched_recall_ids) / len(safety_recall_ids)
    ) * 100
else:
    safety_lineage_match_percentage = None

if enforcement_recall_ids:
    enforcement_coverage_percentage = (
        len(matched_recall_ids) / len(enforcement_recall_ids)
    ) * 100
else:
    enforcement_coverage_percentage = None


lineage_report = {
    "enforcement_record_count": len(enforcement_records),
    "safety_corpus_record_count": len(safety_records),
    "enforcement_unique_recall_ids": len(enforcement_recall_ids),
    "safety_unique_source_recall_ids": len(safety_recall_ids),
    "matched_recall_ids": len(matched_recall_ids),
    "unmatched_safety_ids_count": len(unmatched_safety_ids),
    "unmatched_enforcement_ids_count": len(unmatched_enforcement_ids),
    "safety_lineage_match_percentage": safety_lineage_match_percentage,
    "enforcement_coverage_percentage": enforcement_coverage_percentage,
    "matched_recall_ids_sample": sorted(list(matched_recall_ids))[:10],
    "unmatched_safety_ids_sample": sorted(list(unmatched_safety_ids))[:10]
}


# ----------------------------------------------------------------------------
# 8. SOURCE METADATA CONSISTENCY
# ----------------------------------------------------------------------------

metadata_checks = {}

for source_name, source in sources.items():

    records = source["records"]
    metadata = source["metadata"]

    declared_count = metadata.get(
        "total_records",
        metadata.get("retrieved_count")
    )

    if declared_count is None:
        count_status = "NOT_DECLARED"
    else:
        try:
            count_status = (
                "MATCH"
                if int(declared_count) == len(records)
                else "MISMATCH"
            )
        except (TypeError, ValueError):
            count_status = "INVALID_METADATA_COUNT"

    metadata_checks[source_name] = {
        "declared_record_count": declared_count,
        "actual_record_count": len(records),
        "record_count_status": count_status,
        "source_metadata": metadata
    }


# ----------------------------------------------------------------------------
# 9. WHO SOURCE VERIFICATION STATUS
# ----------------------------------------------------------------------------

who_metadata = sources["WHO"]["metadata"]
who_records = sources["WHO"]["records"]

who_verification_status = who_metadata.get(
    "verification_status",
    "NOT_AVAILABLE"
)

if not who_records:
    who_audit_status = "NO_RECORDS_NOT_VERIFIABLE"
elif who_verification_status != "VERIFIED":
    who_audit_status = "SOURCE_NOT_VERIFIED"
else:
    who_audit_status = "REQUIRES_INDEPENDENT_PROVENANCE_CHECK"


# ----------------------------------------------------------------------------
# 10. AUDIT VERDICT
# ----------------------------------------------------------------------------

critical_issues = []
warnings = []

for source_name, source in sources.items():

    if not source["file_found"]:
        critical_issues.append(
            f"{source_name}: SOURCE_FILE_NOT_FOUND"
        )

    elif not source["valid_json"]:
        critical_issues.append(
            f"{source_name}: INVALID_JSON"
        )

    elif len(source["records"]) == 0:
        warnings.append(
            f"{source_name}: EMPTY_SOURCE"
        )


if bulk_quality["missing_product_ndc"] > 0:
    warnings.append("FDA_BULK_NDC: MISSING_PRODUCT_NDC_VALUES")

if bulk_quality["duplicate_product_ndc"] > 0:
    warnings.append("FDA_BULK_NDC: DUPLICATE_PRODUCT_NDC_VALUES")

if bulk_quality["invalid_record_types"] > 0:
    warnings.append("FDA_BULK_NDC: INVALID_RECORD_TYPES")

if lineage_report["matched_recall_ids"] == 0 and safety_records:
    warnings.append(
        "FDA_SAFETY_CORPUS: NO_RECALL_IDENTIFIER_MATCHES"
    )

if who_audit_status != "VERIFIED":
    warnings.append(
        f"WHO: {who_audit_status}"
    )

for source_name, check in metadata_checks.items():
    if check["record_count_status"] == "MISMATCH":
        warnings.append(
            f"{source_name}: METADATA_RECORD_COUNT_MISMATCH"
        )

if critical_issues:
    overall_status = "FAILED"
elif warnings:
    overall_status = "PARTIAL"
else:
    overall_status = "PASSED"


# ----------------------------------------------------------------------------
# 11. BUILD FINAL AUDIT REPORT
# ----------------------------------------------------------------------------

audit_report = {
    "audit_name": "Advanced Source Integrity & Lineage Validation",
    "cell": "10.2",
    "execution_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "overall_status": overall_status,

    "source_integrity": source_integrity,

    "fda_bulk_ndc_quality": bulk_quality,

    "fda_enforcement_safety_lineage": lineage_report,

    "metadata_consistency": metadata_checks,

    "who_source_audit": {
        "record_count": len(who_records),
        "verification_status": who_verification_status,
        "audit_status": who_audit_status
    },

    "critical_issues": critical_issues,
    "warnings": warnings,

    "read_only_audit": True
}


# ----------------------------------------------------------------------------
# 12. SAVE REPORT
# ----------------------------------------------------------------------------

report_path = os.path.join(
    AUDIT_DIR,
    "cell_10_2_source_integrity_report.json"
)

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(
        audit_report,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ----------------------------------------------------------------------------
# 13. EXECUTION SUMMARY
# ----------------------------------------------------------------------------

print("\n" + "=" * 75)
print("CELL 10.2 - ADVANCED SOURCE INTEGRITY & LINEAGE AUDIT")
print("=" * 75)

print(f"\nOverall Audit Status: {overall_status}")

print("\nSOURCE RECORD COUNTS")

for source_name, source in sources.items():
    print(
        f"{source_name}: "
        f"{len(source['records'])} records"
    )

print("\nFDA BULK NDC QUALITY")

print(
    "Unique Product NDC:",
    bulk_quality["unique_product_ndc"]
)

print(
    "Duplicate Product NDC:",
    bulk_quality["duplicate_product_ndc"]
)

print(
    "Missing Product NDC:",
    bulk_quality["missing_product_ndc"]
)

print(
    "Invalid Record Types:",
    bulk_quality["invalid_record_types"]
)

print("\nFDA ENFORCEMENT / SAFETY CORPUS LINEAGE")

print(
    "Matched Recall IDs:",
    lineage_report["matched_recall_ids"]
)

print(
    "Safety Corpus Lineage Match:",
    (
        f"{safety_lineage_match_percentage:.2f}%"
        if safety_lineage_match_percentage is not None
        else "NOT CALCULATED"
    )
)

print(
    "Enforcement Coverage:",
    (
        f"{enforcement_coverage_percentage:.2f}%"
        if enforcement_coverage_percentage is not None
        else "NOT CALCULATED"
    )
)

print("\nWHO SOURCE AUDIT")

print("Records:", len(who_records))
print("Verification:", who_verification_status)
print("Audit Status:", who_audit_status)

print("\nCRITICAL ISSUES:", len(critical_issues))

for issue in critical_issues:
    print(" -", issue)

print("\nWARNINGS:", len(warnings))

for warning in warnings:
    print(" -", warning)

print("\nAUDIT REPORT:")
print(report_path)

print("\n" + "=" * 75)
print("CELL 10.2 EXECUTION COMPLETED")
print("Read-only audit: TRUE")
print("=" * 75)



CELL 10.2 - ADVANCED SOURCE INTEGRITY & LINEAGE AUDIT

Overall Audit Status: PARTIAL

SOURCE RECORD COUNTS
FDA_BULK_NDC: 138173 records
FDA_NDC: 300 records
FDA_ENFORCEMENT: 150 records
FDA_SAFETY_CORPUS: 150 records
RXNAV: 30 records
WHO: 0 records

FDA BULK NDC QUALITY
Unique Product NDC: 135724
Duplicate Product NDC: 2449
Missing Product NDC: 0
Invalid Record Types: 0

FDA ENFORCEMENT / SAFETY CORPUS LINEAGE
Matched Recall IDs: 150
Safety Corpus Lineage Match: 100.00%
Enforcement Coverage: 100.00%

WHO SOURCE AUDIT
Records: 0
Verification: NOT_VERIFIED
Audit Status: NO_RECORDS_NOT_VERIFIABLE

CRITICAL ISSUES: 0

WARNINGS: 3
 - WHO: EMPTY_SOURCE
 - FDA_BULK_NDC: DUPLICATE_PRODUCT_NDC_VALUES
 - WHO: NO_RECORDS_NOT_VERIFIABLE

AUDIT REPORT:
/content/reports/cell_10_2_source_integrity_report.json

CELL 10.2 EXECUTION COMPLETED
Read-only audit: TRUE


In [17]:

# ============================================================================
# CELL 10.3 - FDA BULK NDC DUPLICATE DEEP INSPECTION
# ============================================================================

import os
import json
import hashlib
from collections import defaultdict
from datetime import datetime, timezone

BASE_DIR = "/content"
REPORT_DIR = os.path.join(BASE_DIR, "reports")

os.makedirs(REPORT_DIR, exist_ok=True)

FILE_PATH = (
    "/content/data/bronze/openfda/"
    "drug-ndc-0001-of-0001.json"
)

# --------------------------------------------------------------------------
# 1. LOAD SOURCE
# --------------------------------------------------------------------------

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"FDA Bulk NDC file not found: {FILE_PATH}"
    )

with open(FILE_PATH, "r", encoding="utf-8") as file:
    payload = json.load(file)

records = payload.get("results", [])

if not isinstance(records, list):
    raise ValueError("Invalid results structure")

# --------------------------------------------------------------------------
# 2. GROUP RECORDS BY PRODUCT NDC
# --------------------------------------------------------------------------

ndc_groups = defaultdict(list)

for record in records:

    if not isinstance(record, dict):
        continue

    ndc = record.get("product_ndc")

    if ndc is not None and str(ndc).strip():
        ndc_groups[str(ndc).strip()].append(record)

duplicate_groups = {
    ndc: group
    for ndc, group in ndc_groups.items()
    if len(group) > 1
}

# --------------------------------------------------------------------------
# 3. EXACT RECORD DUPLICATES
# --------------------------------------------------------------------------

def record_hash(record):

    serialized = json.dumps(
        record,
        sort_keys=True,
        ensure_ascii=False,
        default=str
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


exact_duplicate_groups = 0
exact_duplicate_extra_records = 0

conflicting_groups = 0
conflicting_examples = []

for ndc, group in duplicate_groups.items():

    hashes = [record_hash(record) for record in group]

    unique_hashes = set(hashes)

    if len(unique_hashes) == 1:
        exact_duplicate_groups += 1
        exact_duplicate_extra_records += len(group) - 1

    else:
        conflicting_groups += 1

        if len(conflicting_examples) < 10:

            differing_fields = []

            all_fields = set()

            for record in group:
                all_fields.update(record.keys())

            for field in sorted(all_fields):

                values = {
                    json.dumps(
                        record.get(field),
                        sort_keys=True,
                        ensure_ascii=False,
                        default=str
                    )
                    for record in group
                }

                if len(values) > 1:
                    differing_fields.append(field)

            conflicting_examples.append({
                "product_ndc": ndc,
                "records_count": len(group),
                "differing_fields": differing_fields
            })

# --------------------------------------------------------------------------
# 4. DISTRIBUTION
# --------------------------------------------------------------------------

duplicate_extra_records = sum(
    len(group) - 1
    for group in duplicate_groups.values()
)

duplicate_distribution = {
    "2_records": 0,
    "3_records": 0,
    "4_or_more_records": 0
}

for group in duplicate_groups.values():

    count = len(group)

    if count == 2:
        duplicate_distribution["2_records"] += 1

    elif count == 3:
        duplicate_distribution["3_records"] += 1

    else:
        duplicate_distribution["4_or_more_records"] += 1

# --------------------------------------------------------------------------
# 5. FINAL REPORT
# --------------------------------------------------------------------------

total_records = len(records)
unique_ndc_count = len(ndc_groups)

duplicate_percentage = (
    duplicate_extra_records / total_records * 100
    if total_records else 0
)

report = {
    "audit_name": "FDA Bulk NDC Duplicate Deep Inspection",
    "cell": "10.3",
    "execution_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "total_records": total_records,
    "unique_product_ndc": unique_ndc_count,

    "duplicate_groups": len(duplicate_groups),
    "duplicate_extra_records": duplicate_extra_records,
    "duplicate_percentage": round(duplicate_percentage, 4),

    "exact_duplicate_groups": exact_duplicate_groups,
    "exact_duplicate_extra_records": exact_duplicate_extra_records,

    "conflicting_groups": conflicting_groups,

    "duplicate_distribution": duplicate_distribution,

    "conflicting_examples": conflicting_examples,

    "read_only_audit": True
}

REPORT_PATH = os.path.join(
    REPORT_DIR,
    "cell_10_3_fda_ndc_duplicate_report.json"
)

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(
        report,
        file,
        indent=4,
        ensure_ascii=False
    )

# --------------------------------------------------------------------------
# 6. EXECUTION SUMMARY
# --------------------------------------------------------------------------

print("\n" + "=" * 75)
print("CELL 10.3 - FDA BULK NDC DUPLICATE DEEP INSPECTION")
print("=" * 75)

print("Total Records:", total_records)
print("Unique Product NDC:", unique_ndc_count)

print("\nDUPLICATE ANALYSIS")

print("Duplicate Groups:", len(duplicate_groups))
print("Duplicate Extra Records:", duplicate_extra_records)
print("Duplicate Percentage:", f"{duplicate_percentage:.2f}%")

print("\nEXACT VS CONFLICTING")

print("Exact Duplicate Groups:", exact_duplicate_groups)
print("Exact Duplicate Extra Records:", exact_duplicate_extra_records)
print("Conflicting Groups:", conflicting_groups)

print("\nDUPLICATE GROUP DISTRIBUTION")

for category, count in duplicate_distribution.items():
    print(f"{category}: {count}")

print("\nCONFLICTING EXAMPLES")

for example in conflicting_examples:
    print(example)

print("\nREPORT:", REPORT_PATH)

print("\nREAD-ONLY AUDIT: TRUE")
print("CELL 10.3 EXECUTION COMPLETED")
print("=" * 75)



CELL 10.3 - FDA BULK NDC DUPLICATE DEEP INSPECTION
Total Records: 138173
Unique Product NDC: 135724

DUPLICATE ANALYSIS
Duplicate Groups: 1910
Duplicate Extra Records: 2449
Duplicate Percentage: 1.77%

EXACT VS CONFLICTING
Exact Duplicate Groups: 0
Exact Duplicate Extra Records: 0
Conflicting Groups: 1910

DUPLICATE GROUP DISTRIBUTION
2_records: 1646
3_records: 166
4_or_more_records: 98

CONFLICTING EXAMPLES
{'product_ndc': '71288-117', 'records_count': 2, 'differing_fields': ['openfda', 'packaging', 'product_id', 'spl_id']}
{'product_ndc': '0409-1144', 'records_count': 2, 'differing_fields': ['marketing_start_date', 'openfda', 'packaging', 'product_id', 'spl_id']}
{'product_ndc': '0409-1159', 'records_count': 2, 'differing_fields': ['brand_name', 'brand_name_base', 'generic_name', 'marketing_start_date', 'openfda', 'packaging', 'product_id', 'spl_id']}
{'product_ndc': '71288-427', 'records_count': 2, 'differing_fields': ['openfda', 'packaging', 'product_id', 'spl_id']}
{'product_ndc'

In [18]:

# ============================================================================
# CELL 10.4 - FDA BULK NDC COMPOSITE KEY & DUPLICATE ANALYSIS
# ============================================================================
# Read-only diagnostic audit.
# No Bronze, Silver, or Gold data will be modified.
# ============================================================================

import os
import json
import hashlib
from collections import defaultdict, Counter
from datetime import datetime, timezone

# ----------------------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------------------

BASE_DIR = "/content"

FILE_PATH = (
    "/content/data/bronze/openfda/"
    "drug-ndc-0001-of-0001.json"
)

REPORT_DIR = os.path.join(BASE_DIR, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(FILE_PATH)

# ----------------------------------------------------------------------------
# 2. LOAD FDA BULK NDC
# ----------------------------------------------------------------------------

with open(FILE_PATH, "r", encoding="utf-8") as file:
    payload = json.load(file)

records = payload.get("results", [])

if not isinstance(records, list):
    raise ValueError("Invalid FDA Bulk NDC results structure")

valid_records = [
    record for record in records
    if isinstance(record, dict)
]

# ----------------------------------------------------------------------------
# 3. GENERIC GROUPING FUNCTION
# ----------------------------------------------------------------------------

def group_by_key(data, key_fields):

    groups = defaultdict(list)

    for record in data:

        values = tuple(
            str(record.get(field)).strip()
            if record.get(field) is not None
            else None
            for field in key_fields
        )

        if any(value is None or value == "" for value in values):
            continue

        groups[values].append(record)

    return groups


# ----------------------------------------------------------------------------
# 4. PRODUCT NDC ANALYSIS
# ----------------------------------------------------------------------------

ndc_groups = group_by_key(
    valid_records,
    ["product_ndc"]
)

duplicate_ndc_groups = {
    key: group
    for key, group in ndc_groups.items()
    if len(group) > 1
}

# ----------------------------------------------------------------------------
# 5. PRODUCT ID ANALYSIS
# ----------------------------------------------------------------------------

product_id_groups = group_by_key(
    valid_records,
    ["product_id"]
)

duplicate_product_id_groups = {
    key: group
    for key, group in product_id_groups.items()
    if len(group) > 1
}

# ----------------------------------------------------------------------------
# 6. COMPOSITE KEY ANALYSIS
# ----------------------------------------------------------------------------

composite_groups = group_by_key(
    valid_records,
    ["product_ndc", "product_id"]
)

duplicate_composite_groups = {
    key: group
    for key, group in composite_groups.items()
    if len(group) > 1
}

# ----------------------------------------------------------------------------
# 7. RECORD HASH ANALYSIS
# ----------------------------------------------------------------------------

def calculate_hash(record):

    content = json.dumps(
        record,
        sort_keys=True,
        ensure_ascii=False,
        default=str
    )

    return hashlib.sha256(
        content.encode("utf-8")
    ).hexdigest()


hash_counts = Counter(
    calculate_hash(record)
    for record in valid_records
)

exact_duplicate_hash_groups = sum(
    1 for count in hash_counts.values()
    if count > 1
)

exact_duplicate_extra_records = sum(
    count - 1
    for count in hash_counts.values()
    if count > 1
)

# ----------------------------------------------------------------------------
# 8. DIFFERENCE ANALYSIS
# ----------------------------------------------------------------------------

fields_to_inspect = [
    "product_ndc",
    "product_id",
    "generic_name",
    "brand_name",
    "brand_name_base",
    "labeler_name",
    "product_type",
    "dosage_form",
    "route",
    "marketing_category",
    "marketing_start_date",
    "listing_expiration_date",
    "spl_id",
    "packaging",
    "openfda"
]

field_difference_frequency = Counter()

for key, group in duplicate_ndc_groups.items():

    for field in fields_to_inspect:

        distinct_values = {
            json.dumps(
                record.get(field),
                sort_keys=True,
                ensure_ascii=False,
                default=str
            )
            for record in group
        }

        if len(distinct_values) > 1:
            field_difference_frequency[field] += 1

# ----------------------------------------------------------------------------
# 9. COMPOSITE KEY DUPLICATE SAMPLES
# ----------------------------------------------------------------------------

composite_duplicate_samples = []

for key, group in list(duplicate_composite_groups.items())[:10]:

    composite_duplicate_samples.append({
        "product_ndc": key[0],
        "product_id": key[1],
        "record_count": len(group)
    })

# ----------------------------------------------------------------------------
# 10. PRODUCT NDC DUPLICATE SAMPLES
# ----------------------------------------------------------------------------

ndc_duplicate_samples = []

for key, group in list(duplicate_ndc_groups.items())[:10]:

    ndc_duplicate_samples.append({
        "product_ndc": key[0],
        "record_count": len(group),
        "distinct_product_ids": len({
            str(record.get("product_id"))
            for record in group
            if record.get("product_id") is not None
        }),
        "distinct_spl_ids": len({
            str(record.get("spl_id"))
            for record in group
            if record.get("spl_id") is not None
        })
    })

# ----------------------------------------------------------------------------
# 11. MISSING KEY ANALYSIS
# ----------------------------------------------------------------------------

missing_product_ndc = sum(
    1 for record in valid_records
    if not record.get("product_ndc")
)

missing_product_id = sum(
    1 for record in valid_records
    if not record.get("product_id")
)

missing_spl_id = sum(
    1 for record in valid_records
    if not record.get("spl_id")
)

# ----------------------------------------------------------------------------
# 12. AUDIT REPORT
# ----------------------------------------------------------------------------

report = {

    "audit_name": "FDA Bulk NDC Composite Key Analysis",
    "cell": "10.4",

    "execution_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "total_records": len(valid_records),

    "product_ndc_analysis": {
        "unique_keys": len(ndc_groups),
        "duplicate_groups": len(duplicate_ndc_groups),
        "duplicate_extra_records": sum(
            len(group) - 1
            for group in duplicate_ndc_groups.values()
        )
    },

    "product_id_analysis": {
        "unique_keys": len(product_id_groups),
        "duplicate_groups": len(duplicate_product_id_groups),
        "duplicate_extra_records": sum(
            len(group) - 1
            for group in duplicate_product_id_groups.values()
        )
    },

    "composite_key_analysis": {
        "key_fields": ["product_ndc", "product_id"],
        "unique_keys": len(composite_groups),
        "duplicate_groups": len(duplicate_composite_groups),
        "duplicate_extra_records": sum(
            len(group) - 1
            for group in duplicate_composite_groups.values()
        )
    },

    "exact_record_duplicate_analysis": {
        "duplicate_hash_groups": exact_duplicate_hash_groups,
        "duplicate_extra_records": exact_duplicate_extra_records
    },

    "missing_key_values": {
        "product_ndc": missing_product_ndc,
        "product_id": missing_product_id,
        "spl_id": missing_spl_id
    },

    "field_difference_frequency": dict(
        field_difference_frequency.most_common()
    ),

    "product_ndc_duplicate_samples": ndc_duplicate_samples,

    "composite_key_duplicate_samples": composite_duplicate_samples,

    "read_only_audit": True
}

# ----------------------------------------------------------------------------
# 13. SAVE REPORT
# ----------------------------------------------------------------------------

REPORT_PATH = os.path.join(
    REPORT_DIR,
    "cell_10_4_composite_key_analysis.json"
)

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(
        report,
        file,
        indent=4,
        ensure_ascii=False
    )

# ----------------------------------------------------------------------------
# 14. EXECUTION SUMMARY
# ----------------------------------------------------------------------------

print("\n" + "=" * 75)
print("CELL 10.4 - FDA BULK NDC COMPOSITE KEY ANALYSIS")
print("=" * 75)

print("\nTOTAL RECORDS:", len(valid_records))

print("\nPRODUCT NDC")
print("Unique Keys:", len(ndc_groups))
print("Duplicate Groups:", len(duplicate_ndc_groups))

print("\nPRODUCT ID")
print("Unique Keys:", len(product_id_groups))
print("Duplicate Groups:", len(duplicate_product_id_groups))

print("\nCOMPOSITE KEY: product_ndc + product_id")
print("Unique Keys:", len(composite_groups))
print("Duplicate Groups:", len(duplicate_composite_groups))

print("\nEXACT RECORD DUPLICATES")
print("Duplicate Hash Groups:", exact_duplicate_hash_groups)
print("Extra Records:", exact_duplicate_extra_records)

print("\nMISSING KEY VALUES")
print("Missing product_ndc:", missing_product_ndc)
print("Missing product_id:", missing_product_id)
print("Missing spl_id:", missing_spl_id)

print("\nFIELD DIFFERENCE FREQUENCY")

for field, count in field_difference_frequency.most_common():
    print(f"{field}: {count}")

print("\nPRODUCT NDC DUPLICATE SAMPLES")

for sample in ndc_duplicate_samples:
    print(sample)

print("\nCOMPOSITE KEY DUPLICATE SAMPLES")

for sample in composite_duplicate_samples:
    print(sample)

print("\nREPORT:", REPORT_PATH)
print("READ-ONLY AUDIT: TRUE")
print("CELL 10.4 EXECUTED SUCCESSFULLY")
print("=" * 75)



CELL 10.4 - FDA BULK NDC COMPOSITE KEY ANALYSIS

TOTAL RECORDS: 138173

PRODUCT NDC
Unique Keys: 135724
Duplicate Groups: 1910

PRODUCT ID
Unique Keys: 138173
Duplicate Groups: 0

COMPOSITE KEY: product_ndc + product_id
Unique Keys: 138173
Duplicate Groups: 0

EXACT RECORD DUPLICATES
Duplicate Hash Groups: 0
Extra Records: 0

MISSING KEY VALUES
Missing product_ndc: 0
Missing product_id: 0
Missing spl_id: 0

FIELD DIFFERENCE FREQUENCY
product_id: 1910
spl_id: 1910
packaging: 1909
openfda: 1549
brand_name: 782
brand_name_base: 782
route: 779
marketing_category: 776
product_type: 746
marketing_start_date: 672
listing_expiration_date: 220
generic_name: 156
labeler_name: 145
dosage_form: 4

PRODUCT NDC DUPLICATE SAMPLES
{'product_ndc': '71288-117', 'record_count': 2, 'distinct_product_ids': 2, 'distinct_spl_ids': 2}
{'product_ndc': '0409-1144', 'record_count': 2, 'distinct_product_ids': 2, 'distinct_spl_ids': 2}
{'product_ndc': '0409-1159', 'record_count': 2, 'distinct_product_ids': 2, 'di

In [19]:

# ============================================================================
# CELL 10.5 - COMPREHENSIVE SOURCE INTEGRITY & DATA QUALITY REMEDIATION
# ============================================================================
# Scope:
#   1. FDA Bulk NDC structural and business-rule validation
#   2. Primary-key candidate validation
#   3. Duplicate and conflicting-record classification
#   4. Date and field-value validation
#   5. FDA metadata consistency checks
#   6. FDA Enforcement <-> Safety Corpus lineage validation
#   7. RxNav and WHO source status validation
#   8. Non-destructive Silver candidate generation
#   9. Quarantine and remediation reports
#
# IMPORTANT:
#   - Bronze files are READ-ONLY.
#   - No source records are silently deleted.
#   - product_id is evaluated as a candidate key, not assumed to be
#     a permanent business key.
#   - Missing external sources are reported, never fabricated.
# ============================================================================

import os
import json
import csv
import hashlib
from collections import defaultdict, Counter
from datetime import datetime, timezone

# ============================================================================
# 1. CONFIGURATION
# ============================================================================

BASE_DIR = "/content"

BRONZE_ROOT = os.path.join(BASE_DIR, "data", "bronze")
REPORT_DIR = os.path.join(BASE_DIR, "reports")
REMEDIATION_DIR = os.path.join(BASE_DIR, "data", "remediation")

os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(REMEDIATION_DIR, exist_ok=True)

SOURCE_FILES = {
    "FDA_BULK_NDC": "drug-ndc-0001-of-0001.json",
    "FDA_NDC": "fda_ndc_raw.json",
    "FDA_ENFORCEMENT": "fda_enforcement_raw.json",
    "FDA_SAFETY_CORPUS": "fda_safety_corpus.json",
    "RXNAV": "rxnav_concepts_raw.json",
    "WHO": "who_eml_raw.json"
}

REQUIRED_FDA_FIELDS = [
    "product_id",
    "product_ndc",
    "generic_name",
    "labeler_name",
    "product_type"
]

DATE_FIELDS = [
    "marketing_start_date",
    "listing_expiration_date"
]

# ============================================================================
# 2. HELPERS
# ============================================================================

def utc_now():
    return datetime.now(timezone.utc).isoformat()


def find_file(filename):
    search_roots = [
        BRONZE_ROOT,
        os.path.join(BASE_DIR, "data"),
        BASE_DIR
    ]

    excluded_dirs = {
        "reports",
        "remediation",
        "silver",
        "gold",
        "silver_parquet",
        "gold_parquet",
        ".ipynb_checkpoints"
    }

    for root in search_roots:
        if not os.path.isdir(root):
            continue

        for current_root, dirs, files in os.walk(root):
            dirs[:] = [
                directory for directory in dirs
                if directory not in excluded_dirs
            ]

            if filename in files:
                return os.path.join(current_root, filename)

    return None


def load_json_source(source_name, filename):

    path = find_file(filename)

    result = {
        "source_name": source_name,
        "path": path,
        "file_found": path is not None,
        "valid_json": False,
        "records": [],
        "metadata": {},
        "error": None
    }

    if path is None:
        result["error"] = "SOURCE_FILE_NOT_FOUND"
        return result

    try:
        with open(path, "r", encoding="utf-8") as file:
            payload = json.load(file)

        if not isinstance(payload, dict):
            raise ValueError("JSON root is not an object")

        records = payload.get("results", [])
        metadata = payload.get("metadata", {})

        if not isinstance(records, list):
            raise ValueError("results is not a list")

        if not isinstance(metadata, dict):
            metadata = {}

        result.update({
            "valid_json": True,
            "records": records,
            "metadata": metadata
        })

    except Exception as error:
        result["error"] = str(error)

    return result


def normalize_text(value):
    if value is None:
        return None

    text_value = str(value).strip()

    if not text_value:
        return None

    return text_value


def normalize_identifier(value):
    value = normalize_text(value)

    if value is None:
        return None

    return value.upper()


def parse_yyyymmdd(value):

    value = normalize_text(value)

    if value is None:
        return None, "MISSING"

    if len(value) != 8 or not value.isdigit():
        return None, "INVALID_FORMAT"

    try:
        parsed = datetime.strptime(value, "%Y%m%d").date()
        return parsed, "VALID"

    except ValueError:
        return None, "INVALID_DATE"


def json_hash(record):
    serialized = json.dumps(
        record,
        sort_keys=True,
        ensure_ascii=False,
        default=str
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


def write_json(path, content):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(
            content,
            file,
            indent=4,
            ensure_ascii=False,
            default=str
        )


# ============================================================================
# 3. LOAD ALL BRONZE SOURCES
# ============================================================================

sources = {}

for source_name, filename in SOURCE_FILES.items():
    sources[source_name] = load_json_source(
        source_name,
        filename
    )

# ============================================================================
# 4. SOURCE INVENTORY AND METADATA VALIDATION
# ============================================================================

source_inventory = {}
metadata_issues = []

for source_name, source in sources.items():

    records = source["records"]
    metadata = source["metadata"]

    declared_count = metadata.get(
        "total_records",
        metadata.get("retrieved_count")
    )

    if declared_count is None:
        count_status = "NOT_DECLARED"

    else:
        try:
            count_status = (
                "MATCH"
                if int(declared_count) == len(records)
                else "MISMATCH"
            )
        except (TypeError, ValueError):
            count_status = "INVALID_DECLARED_COUNT"

    if count_status in [
        "MISMATCH",
        "INVALID_DECLARED_COUNT"
    ]:
        metadata_issues.append({
            "source": source_name,
            "issue": count_status,
            "declared_count": declared_count,
            "actual_count": len(records)
        })

    source_inventory[source_name] = {
        "file_found": source["file_found"],
        "valid_json": source["valid_json"],
        "path": source["path"],
        "record_count": len(records),
        "metadata_available": bool(metadata),
        "declared_record_count": declared_count,
        "record_count_status": count_status,
        "error": source["error"]
    }

# ============================================================================
# 5. FDA BULK NDC STRUCTURAL VALIDATION
# ============================================================================

bulk_source = sources["FDA_BULK_NDC"]
bulk_records = bulk_source["records"]

if not bulk_source["valid_json"]:
    raise RuntimeError(
        "FDA Bulk NDC could not be loaded: "
        + str(bulk_source["error"])
    )

valid_dict_records = [
    record for record in bulk_records
    if isinstance(record, dict)
]

non_dict_record_count = (
    len(bulk_records) - len(valid_dict_records)
)

# ============================================================================
# 6. PRIMARY KEY AND DUPLICATE ANALYSIS
# ============================================================================

product_id_groups = defaultdict(list)
product_ndc_groups = defaultdict(list)
composite_groups = defaultdict(list)
record_hash_counts = Counter()

missing_product_id = 0
missing_product_ndc = 0

for index, record in enumerate(valid_dict_records):

    product_id = normalize_identifier(
        record.get("product_id")
    )

    product_ndc = normalize_identifier(
        record.get("product_ndc")
    )

    if product_id is None:
        missing_product_id += 1
    else:
        product_id_groups[product_id].append(index)

    if product_ndc is None:
        missing_product_ndc += 1
    else:
        product_ndc_groups[product_ndc].append(index)

    if product_id is not None and product_ndc is not None:
        composite_groups[
            (product_ndc, product_id)
        ].append(index)

    record_hash_counts[
        json_hash(record)
    ] += 1


duplicate_product_id_groups = {
    key: value
    for key, value in product_id_groups.items()
    if len(value) > 1
}

duplicate_product_ndc_groups = {
    key: value
    for key, value in product_ndc_groups.items()
    if len(value) > 1
}

duplicate_composite_groups = {
    key: value
    for key, value in composite_groups.items()
    if len(value) > 1
}

exact_duplicate_extra_records = sum(
    count - 1
    for count in record_hash_counts.values()
    if count > 1
)

product_id_is_unique = (
    missing_product_id == 0
    and len(duplicate_product_id_groups) == 0
    and len(product_id_groups) == len(valid_dict_records)
)

# ============================================================================
# 7. FDA BUSINESS-RULE VALIDATION
# ============================================================================

allowed_finished_values = {
    "TRUE",
    "FALSE"
}

allowed_product_types = {
    "HUMAN PRESCRIPTION DRUG",
    "HUMAN OTC DRUG",
    "HUMAN COMPOUNDED DRUG",
    "ANIMAL PRESCRIPTION DRUG",
    "ANIMAL OTC DRUG",
    "ANIMAL COMPOUNDED DRUG",
    "NON-STANDARDIZED ALLERGENIC",
    "STANDARDIZED ALLERGENIC",
    "PLASMA DERIVATIVE",
    "VACCINE",
    "CELLULAR THERAPY",
    "GENE THERAPY",
    "TISSUE ENGINEERING"
}

field_missing_counts = Counter()
finished_value_counts = Counter()
product_type_counts = Counter()

invalid_finished_count = 0
unexpected_product_type_count = 0

invalid_dates = []
date_order_issues = []

record_issues = defaultdict(list)

for index, record in enumerate(valid_dict_records):

    for field in REQUIRED_FDA_FIELDS:

        value = normalize_text(record.get(field))

        if value is None:
            field_missing_counts[field] += 1
            record_issues[index].append(
                f"MISSING_REQUIRED_FIELD:{field}"
            )

    # finished validation
    finished_value = normalize_identifier(
        record.get("finished")
    )

    if finished_value is not None:
        finished_value_counts[finished_value] += 1

        if finished_value not in allowed_finished_values:
            invalid_finished_count += 1
            record_issues[index].append(
                "UNEXPECTED_FINISHED_VALUE"
            )
    else:
        record_issues[index].append(
            "MISSING_FINISHED_VALUE"
        )

    # product_type inventory
    product_type = normalize_text(
        record.get("product_type")
    )

    if product_type:
        normalized_type = product_type.upper()
        product_type_counts[normalized_type] += 1

        # Report values not in the known list, but do not
        # automatically classify them as invalid.
        if normalized_type not in allowed_product_types:
            unexpected_product_type_count += 1

    # date validation
    parsed_dates = {}

    for field in DATE_FIELDS:

        raw_value = record.get(field)

        parsed_date, date_status = parse_yyyymmdd(
            raw_value
        )

        parsed_dates[field] = parsed_date

        if date_status in [
            "INVALID_FORMAT",
            "INVALID_DATE"
        ]:
            invalid_dates.append({
                "record_index": index,
                "product_id": record.get("product_id"),
                "field": field,
                "value": raw_value,
                "issue": date_status
            })

            record_issues[index].append(
                f"{date_status}:{field}"
            )

    start_date = parsed_dates.get(
        "marketing_start_date"
    )

    expiration_date = parsed_dates.get(
        "listing_expiration_date"
    )

    if (
        start_date is not None
        and expiration_date is not None
        and expiration_date < start_date
    ):
        date_order_issues.append({
            "record_index": index,
            "product_id": record.get("product_id"),
            "marketing_start_date": str(start_date),
            "listing_expiration_date": str(expiration_date)
        })

        record_issues[index].append(
            "EXPIRATION_BEFORE_MARKETING_START"
        )

# ============================================================================
# 8. DUPLICATE NDC FIELD DIFFERENCE ANALYSIS
# ============================================================================

fields_to_compare = [
    "product_id",
    "spl_id",
    "generic_name",
    "brand_name",
    "brand_name_base",
    "labeler_name",
    "product_type",
    "dosage_form",
    "route",
    "marketing_category",
    "marketing_start_date",
    "listing_expiration_date",
    "packaging",
    "openfda"
]

field_difference_frequency = Counter()

for ndc, indexes in duplicate_product_ndc_groups.items():

    group = [
        valid_dict_records[index]
        for index in indexes
    ]

    for field in fields_to_compare:

        distinct_values = {
            json.dumps(
                record.get(field),
                sort_keys=True,
                ensure_ascii=False,
                default=str
            )
            for record in group
        }

        if len(distinct_values) > 1:
            field_difference_frequency[field] += 1

# ============================================================================
# 9. FDA ENFORCEMENT <-> SAFETY CORPUS LINEAGE
# ============================================================================

enforcement_records = sources[
    "FDA_ENFORCEMENT"
]["records"]

safety_records = sources[
    "FDA_SAFETY_CORPUS"
]["records"]

enforcement_ids = set()
safety_ids = set()

for record in enforcement_records:

    if isinstance(record, dict):
        value = normalize_identifier(
            record.get("recall_number")
        )

        if value:
            enforcement_ids.add(value)

for record in safety_records:

    if isinstance(record, dict):
        value = normalize_identifier(
            record.get("source_recall_number")
        )

        if value:
            safety_ids.add(value)

matched_ids = enforcement_ids.intersection(
    safety_ids
)

unmatched_safety_ids = safety_ids - enforcement_ids
unmatched_enforcement_ids = enforcement_ids - safety_ids

safety_lineage_percentage = (
    len(matched_ids) / len(safety_ids) * 100
    if safety_ids else None
)

enforcement_coverage_percentage = (
    len(matched_ids) / len(enforcement_ids) * 100
    if enforcement_ids else None
)

# ============================================================================
# 10. RXNAV VALIDATION
# ============================================================================

rxnav_records = sources["RXNAV"]["records"]

rxnav_status_counts = Counter()
rxnav_missing_rxcui = 0
rxnav_duplicate_queries = Counter()

for record in rxnav_records:

    if not isinstance(record, dict):
        continue

    query_name = normalize_identifier(
        record.get("query_drug_name")
    )

    rxcui = normalize_text(
        record.get("rxcui")
    )

    status = normalize_identifier(
        record.get("status")
    )

    if query_name:
        rxnav_duplicate_queries[query_name] += 1

    if not rxcui:
        rxnav_missing_rxcui += 1

    if status:
        rxnav_status_counts[status] += 1

duplicate_rxnav_queries = {
    query: count
    for query, count in rxnav_duplicate_queries.items()
    if count > 1
}

# ============================================================================
# 11. WHO SOURCE STATUS
# ============================================================================

who_source = sources["WHO"]

who_metadata = who_source["metadata"]
who_records = who_source["records"]

who_verification = who_metadata.get(
    "verification_status",
    "NOT_AVAILABLE"
)

if not who_records:
    who_status = "UNAVAILABLE_NO_RECORDS"
elif who_verification != "VERIFIED":
    who_status = "RECORDS_PRESENT_BUT_NOT_VERIFIED"
else:
    who_status = "DECLARED_VERIFIED_REQUIRES_EXTERNAL_PROVENANCE_CHECK"

# ============================================================================
# 12. BUILD NON-DESTRUCTIVE SILVER CANDIDATE
# ============================================================================
# Every valid dictionary record is retained.
# Records with quality issues are flagged, not silently deleted.
# Original values are preserved.
# Additional fields are prefixed with _dq_.

candidate_path = os.path.join(
    REMEDIATION_DIR,
    "fda_bulk_ndc_silver_candidate.jsonl"
)

quarantine_path = os.path.join(
    REMEDIATION_DIR,
    "fda_bulk_ndc_quarantine.jsonl"
)

candidate_count = 0
quarantine_count = 0
clean_count = 0

with open(
    candidate_path,
    "w",
    encoding="utf-8"
) as candidate_file, open(
    quarantine_path,
    "w",
    encoding="utf-8"
) as quarantine_file:

    for index, original_record in enumerate(valid_dict_records):

        record = dict(original_record)

        issues = list(record_issues.get(index, []))

        product_id = normalize_identifier(
            record.get("product_id")
        )

        product_ndc = normalize_identifier(
            record.get("product_ndc")
        )

        # A missing or duplicated product_id prevents safe
        # use as a unique row identifier.
        if product_id is None:
            issues.append("NO_CANDIDATE_PRIMARY_KEY")

        elif product_id in duplicate_product_id_groups:
            issues.append("DUPLICATE_CANDIDATE_PRIMARY_KEY")

        # Retain repeated NDCs, but explicitly classify them.
        if product_ndc in duplicate_product_ndc_groups:
            issues.append("REPEATED_PRODUCT_NDC_NOT_AUTO_ERROR")

        record["_dq_record_id"] = product_id
        record["_dq_product_ndc_normalized"] = product_ndc
        record["_dq_quality_status"] = (
            "REVIEW_REQUIRED" if issues else "PASS"
        )
        record["_dq_issue_count"] = len(issues)
        record["_dq_issues"] = sorted(set(issues))

        candidate_file.write(
            json.dumps(
                record,
                ensure_ascii=False,
                default=str
            ) + "\n"
        )

        candidate_count += 1

        if issues:
            quarantine_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    default=str
                ) + "\n"
            )

            quarantine_count += 1
        else:
            clean_count += 1

# ============================================================================
# 13. NON-DICTIONARY RECORD QUARANTINE
# ============================================================================

non_dict_quarantine_path = os.path.join(
    REMEDIATION_DIR,
    "fda_bulk_ndc_non_dictionary_records.jsonl"
)

if non_dict_record_count:

    with open(
        non_dict_quarantine_path,
        "w",
        encoding="utf-8"
    ) as file:

        for index, record in enumerate(bulk_records):

            if not isinstance(record, dict):
                file.write(
                    json.dumps({
                        "source_index": index,
                        "record_value": str(record),
                        "issue": "NON_DICTIONARY_RECORD"
                    }, ensure_ascii=False) + "\n"
                )

else:
    non_dict_quarantine_path = None

# ============================================================================
# 14. REMEDIATION RECOMMENDATIONS
# ============================================================================

recommendations = []

if product_id_is_unique:
    recommendations.append({
        "area": "PRIMARY_KEY",
        "status": "CANDIDATE_PASSED_CURRENT_DATASET",
        "recommendation": (
            "product_id is unique and non-null in this snapshot. "
            "Use it as a candidate row key after validating stability "
            "across future source snapshots."
        )
    })
else:
    recommendations.append({
        "area": "PRIMARY_KEY",
        "status": "FAILED",
        "recommendation": (
            "Do not use product_id as a unique primary key until "
            "missing and duplicate identifiers are resolved."
        )
    })

if duplicate_product_ndc_groups:
    recommendations.append({
        "area": "PRODUCT_NDC",
        "status": "REPEATED_BUSINESS_IDENTIFIER",
        "recommendation": (
            "Preserve repeated product_ndc values. Do not deduplicate "
            "on product_ndc alone. Investigate record-level differences."
        )
    })

if invalid_dates or date_order_issues:
    recommendations.append({
        "area": "DATE_QUALITY",
        "status": "REVIEW_REQUIRED",
        "recommendation": (
            "Review invalid date formats and date-order violations. "
            "Original date values were preserved."
        )
    })

if who_status != "DECLARED_VERIFIED_REQUIRES_EXTERNAL_PROVENANCE_CHECK":
    recommendations.append({
        "area": "WHO",
        "status": who_status,
        "recommendation": (
            "Obtain a genuine WHO EML source file and independently "
            "document its official source, version, retrieval date, "
            "and integrity hash. Do not fabricate records."
        )
    })

if metadata_issues:
    recommendations.append({
        "area": "SOURCE_METADATA",
        "status": "REVIEW_REQUIRED",
        "recommendation": (
            "Resolve source metadata count mismatches. Missing metadata "
            "must remain explicitly undocumented, not inferred."
        )
    })

# ============================================================================
# 15. OVERALL STATUS
# ============================================================================

critical_issues = []

if not bulk_source["file_found"] or not bulk_source["valid_json"]:
    critical_issues.append("FDA_BULK_NDC_SOURCE_UNAVAILABLE_OR_INVALID")

if non_dict_record_count:
    critical_issues.append("NON_DICTIONARY_RECORDS_FOUND")

if missing_product_id:
    critical_issues.append("MISSING_PRODUCT_ID")

if duplicate_product_id_groups:
    critical_issues.append("DUPLICATE_PRODUCT_ID")

if critical_issues:
    overall_status = "FAILED"
elif (
    invalid_dates
    or date_order_issues
    or invalid_finished_count
    or metadata_issues
    or who_status != "DECLARED_VERIFIED_REQUIRES_EXTERNAL_PROVENANCE_CHECK"
):
    overall_status = "PARTIAL"
else:
    overall_status = "PASSED"

# ============================================================================
# 16. FINAL AUDIT REPORT
# ============================================================================

audit_report = {

    "audit_name": (
        "Comprehensive FDA Source Integrity, "
        "Data Quality and Remediation Audit"
    ),

    "cell": "10.5",
    "execution_timestamp_utc": utc_now(),

    "overall_status": overall_status,

    "read_only_bronze": True,
    "source_records_deleted": 0,

    "source_inventory": source_inventory,

    "fda_bulk_ndc": {
        "total_source_records": len(bulk_records),
        "valid_dictionary_records": len(valid_dict_records),
        "non_dictionary_records": non_dict_record_count,

        "primary_key": {
            "candidate_field": "product_id",
            "unique": product_id_is_unique,
            "unique_values": len(product_id_groups),
            "missing_values": missing_product_id,
            "duplicate_groups": len(
                duplicate_product_id_groups
            )
        },

        "product_ndc": {
            "unique_values": len(product_ndc_groups),
            "duplicate_groups": len(
                duplicate_product_ndc_groups
            ),
            "duplicate_extra_records": sum(
                len(group) - 1
                for group in duplicate_product_ndc_groups.values()
            )
        },

        "composite_key": {
            "fields": ["product_ndc", "product_id"],
            "unique_values": len(composite_groups),
            "duplicate_groups": len(duplicate_composite_groups)
        },

        "exact_duplicate_extra_records": (
            exact_duplicate_extra_records
        ),

        "missing_required_fields": dict(
            field_missing_counts
        ),

        "finished_value_distribution": dict(
            finished_value_counts
        ),

        "invalid_finished_values": invalid_finished_count,

        "product_type_distribution": dict(
            product_type_counts
        ),

        "product_types_outside_known_list": (
            unexpected_product_type_count
        ),

        "invalid_dates_count": len(invalid_dates),
        "date_order_issues_count": len(date_order_issues),

        "field_difference_frequency": dict(
            field_difference_frequency.most_common()
        )
    },

    "fda_lineage": {
        "enforcement_records": len(enforcement_records),
        "safety_corpus_records": len(safety_records),
        "matched_recall_ids": len(matched_ids),
        "unmatched_safety_ids": len(unmatched_safety_ids),
        "unmatched_enforcement_ids": len(
            unmatched_enforcement_ids
        ),
        "safety_lineage_match_percentage": (
            safety_lineage_percentage
        ),
        "enforcement_coverage_percentage": (
            enforcement_coverage_percentage
        ),
        "safety_corpus_is_independent_primary_source": False
    },

    "rxnav": {
        "records": len(rxnav_records),
        "status_distribution": dict(rxnav_status_counts),
        "missing_rxcui": rxnav_missing_rxcui,
        "duplicate_query_names": len(duplicate_rxnav_queries)
    },

    "who": {
        "records": len(who_records),
        "verification_status": who_verification,
        "audit_status": who_status
    },

    "metadata_issues": metadata_issues,

    "remediation_outputs": {
        "silver_candidate_jsonl": candidate_path,
        "silver_candidate_records": candidate_count,
        "candidate_pass_records": clean_count,
        "candidate_review_records": quarantine_count,
        "quarantine_jsonl": quarantine_path,
        "non_dictionary_quarantine": non_dict_quarantine_path
    },

    "critical_issues": critical_issues,
    "recommendations": recommendations
}

# ============================================================================
# 17. SAVE REPORTS
# ============================================================================

report_path = os.path.join(
    REPORT_DIR,
    "cell_10_5_comprehensive_integrity_report.json"
)

write_json(report_path, audit_report)

invalid_dates_path = os.path.join(
    REPORT_DIR,
    "cell_10_5_invalid_dates.json"
)

write_json(invalid_dates_path, invalid_dates)

date_order_path = os.path.join(
    REPORT_DIR,
    "cell_10_5_date_order_issues.json"
)

write_json(date_order_path, date_order_issues)

# ============================================================================
# 18. EXECUTION SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("CELL 10.5 - COMPREHENSIVE SOURCE INTEGRITY & DATA QUALITY AUDIT")
print("=" * 80)

print("\nOVERALL STATUS:", overall_status)

print("\nFDA BULK NDC")
print("Source Records:", len(bulk_records))
print("Valid Dictionary Records:", len(valid_dict_records))
print("Non-Dictionary Records:", non_dict_record_count)

print("\nPRIMARY KEY ANALYSIS")
print("Candidate Key: product_id")
print("Unique:", product_id_is_unique)
print("Unique Values:", len(product_id_groups))
print("Missing:", missing_product_id)
print("Duplicate Groups:", len(duplicate_product_id_groups))

print("\nPRODUCT NDC ANALYSIS")
print("Unique Values:", len(product_ndc_groups))
print("Duplicate Groups:", len(duplicate_product_ndc_groups))

print("\nCOMPOSITE KEY")
print("Duplicate Groups:", len(duplicate_composite_groups))

print("\nEXACT DUPLICATES")
print("Extra Exact Duplicate Records:", exact_duplicate_extra_records)

print("\nBUSINESS RULE VALIDATION")
print("Missing Required Fields:", dict(field_missing_counts))
print("Finished Values:", dict(finished_value_counts))
print("Unexpected Finished Values:", invalid_finished_count)
print("Product Types Outside Known List:", unexpected_product_type_count)
print("Invalid Dates:", len(invalid_dates))
print("Date Order Issues:", len(date_order_issues))

print("\nFDA LINEAGE")
print("Matched Recall IDs:", len(matched_ids))
print("Safety Lineage Match:", safety_lineage_percentage)
print("Enforcement Coverage:", enforcement_coverage_percentage)

print("\nRXNAV")
print("Records:", len(rxnav_records))
print("Status Distribution:", dict(rxnav_status_counts))
print("Missing RxCUI:", rxnav_missing_rxcui)

print("\nWHO")
print("Records:", len(who_records))
print("Verification:", who_verification)
print("Status:", who_status)

print("\nREMEDIATION OUTPUTS")
print("Candidate Records:", candidate_count)
print("Candidate PASS:", clean_count)
print("Candidate REVIEW:", quarantine_count)
print("Quarantine:", quarantine_path)

print("\nCRITICAL ISSUES:", len(critical_issues))

for issue in critical_issues:
    print(" -", issue)

print("\nRECOMMENDATIONS")

for item in recommendations:
    print(
        f"- {item['area']}: "
        f"{item['status']}"
    )
    print(" ", item["recommendation"])

print("\nREPORT FILES")
print("Main Report:", report_path)
print("Invalid Dates:", invalid_dates_path)
print("Date Order Issues:", date_order_path)
print("Candidate:", candidate_path)

print("\nBRONZE MODIFIED: FALSE")
print("SOURCE RECORDS DELETED: 0")
print("CELL 10.5 EXECUTION COMPLETED")
print("=" * 80)


CELL 10.5 - COMPREHENSIVE SOURCE INTEGRITY & DATA QUALITY AUDIT

OVERALL STATUS: PARTIAL

FDA BULK NDC
Source Records: 138173
Valid Dictionary Records: 138173
Non-Dictionary Records: 0

PRIMARY KEY ANALYSIS
Candidate Key: product_id
Unique: True
Unique Values: 138173
Missing: 0
Duplicate Groups: 0

PRODUCT NDC ANALYSIS
Unique Values: 135724
Duplicate Groups: 1910

COMPOSITE KEY
Duplicate Groups: 0

EXACT DUPLICATES
Extra Exact Duplicate Records: 0

BUSINESS RULE VALIDATION
Missing Required Fields: {'generic_name': 3}
Finished Values: {'TRUE': 116528, 'FALSE': 21645}
Unexpected Finished Values: 0
Product Types Outside Known List: 21428
Invalid Dates: 0
Date Order Issues: 0

FDA LINEAGE
Matched Recall IDs: 150
Safety Lineage Match: 100.0
Enforcement Coverage: 100.0

RXNAV
Records: 30
Status Distribution: {'MATCHED': 26, 'NO_RXCUI_MATCH': 4}
Missing RxCUI: 4

WHO
Records: 0
Verification: NOT_VERIFIED
Status: UNAVAILABLE_NO_RECORDS

REMEDIATION OUTPUTS
Candidate Records: 138173
Candidate 

In [20]:
# =============================================================================
# CELL 10.6 — ENTERPRISE OBSERVABILITY, TELEMETRY, SLA & EXECUTION GOVERNANCE
# =============================================================================

import os
import sys
import json
import time
import uuid
import shutil
import zipfile
import psutil
import logging
import platform
import hashlib
import tempfile
import traceback

from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONFIGURATION & ENVIRONMENT
# =============================================================================

CELL_ID = "10.6"
CELL_NAME = "ENTERPRISE OBSERVABILITY & EXECUTION GOVERNANCE"

BASE_DIR = Path(globals().get("BASE_DIR", "/content")).resolve()

# Prefer the project's existing DATA_DIRS configuration.
_existing_dirs = globals().get("DATA_DIRS", {})

REPORTS_DIR = Path(
    _existing_dirs.get("reports", BASE_DIR / "reports")
).resolve()

DATA_DIR = Path(
    _existing_dirs.get("data", BASE_DIR / "data")
).resolve()

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

HISTORY_FILE = REPORTS_DIR / "pipeline_execution_history.json"
OBSERVABILITY_REPORT = REPORTS_DIR / "cell_10_6_observability_report.json"

MASTER_ZIP = BASE_DIR / "enterprise_master_production_bundle.zip"

# SLA thresholds — configurable in one place.
SLA_CONFIG = {
    "memory_warning_mb": 2000,
    "disk_free_critical_percent": 5,
    "disk_free_warning_percent": 10,
    "max_history_runs": 50,
    "cpu_sample_seconds": 0.25,
}

# Use the notebook logger if available; otherwise create a local logger.
if "logger" not in globals():
    logger = logging.getLogger("pipeline_observability")
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(
            logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
        )
        logger.addHandler(handler)
    logger.setLevel(logging.INFO)

logger.info(
    f"Executing CELL {CELL_ID}: {CELL_NAME}"
)

RUN_ID = (
    f"RUN-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}-"
    f"{uuid.uuid4().hex[:10].upper()}"
)

START_TIME_UTC = datetime.now(timezone.utc)


# =============================================================================
# 1. SAFE FILE & JSON UTILITIES
# =============================================================================

def atomic_write_json(file_path: Path, payload: dict) -> None:
    """Write JSON atomically to avoid partially written reports."""

    file_path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = file_path.with_name(
        f".{file_path.name}.{uuid.uuid4().hex}.tmp"
    )

    try:
        with open(temp_path, "w", encoding="utf-8") as file:
            json.dump(
                payload,
                file,
                ensure_ascii=False,
                indent=2,
                default=str
            )
            file.flush()
            os.fsync(file.fileno())

        os.replace(temp_path, file_path)

    finally:
        if temp_path.exists():
            temp_path.unlink(missing_ok=True)


def safe_read_json(file_path: Path):
    """Return parsed JSON or a descriptive read status."""

    if not file_path.exists():
        return None, "NOT_FOUND"

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            return json.load(file), "READ_SUCCESS"

    except Exception as exc:
        return None, f"READ_ERROR: {type(exc).__name__}: {exc}"


def sha256_file(file_path: Path) -> str:
    """Calculate a file SHA-256 without loading it entirely into memory."""

    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# =============================================================================
# 2. SYSTEM TELEMETRY & RESOURCE PROFILING
# =============================================================================

def get_directory_size_mb(directory: Path) -> dict:
    """Measure directory footprint while tolerating inaccessible files."""

    total_bytes = 0
    scanned_files = 0
    unreadable_files = 0

    if not directory.exists():
        return {
            "size_mb": 0,
            "scanned_files": 0,
            "unreadable_files": 0,
            "directory_exists": False,
        }

    for root, _, filenames in os.walk(directory):
        for filename in filenames:
            file_path = Path(root) / filename

            try:
                if file_path.is_file():
                    total_bytes += file_path.stat().st_size
                    scanned_files += 1
            except (OSError, PermissionError):
                unreadable_files += 1

    return {
        "size_mb": round(total_bytes / (1024 ** 2), 2),
        "scanned_files": scanned_files,
        "unreadable_files": unreadable_files,
        "directory_exists": True,
    }


def capture_system_telemetry() -> dict:
    """Capture process, system, storage and runtime metrics."""

    process = psutil.Process(os.getpid())

    # Prime CPU measurement, then take a short interval sample.
    process.cpu_percent(interval=None)
    process_cpu_percent = process.cpu_percent(
        interval=SLA_CONFIG["cpu_sample_seconds"]
    )

    memory = process.memory_info()
    virtual_memory = psutil.virtual_memory()
    disk = psutil.disk_usage(str(BASE_DIR))

    data_footprint = get_directory_size_mb(DATA_DIR)

    try:
        process_threads = process.num_threads()
    except (psutil.Error, OSError):
        process_threads = None

    try:
        open_files_count = len(process.open_files())
    except (psutil.Error, OSError):
        open_files_count = None

    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),

        "process": {
            "pid": os.getpid(),
            "memory_rss_mb": round(memory.rss / (1024 ** 2), 2),
            "memory_vms_mb": round(memory.vms / (1024 ** 2), 2),
            "cpu_percent": round(process_cpu_percent, 2),
            "threads": process_threads,
            "open_files": open_files_count,
        },

        "system": {
            "cpu_logical_cores": psutil.cpu_count(logical=True),
            "memory_total_mb": round(virtual_memory.total / (1024 ** 2), 2),
            "memory_available_mb": round(
                virtual_memory.available / (1024 ** 2), 2
            ),
            "memory_used_percent": virtual_memory.percent,
            "platform": platform.platform(),
            "python_version": sys.version.split()[0],
        },

        "storage": {
            "base_directory": str(BASE_DIR),
            "disk_total_gb": round(disk.total / (1024 ** 3), 2),
            "disk_used_gb": round(disk.used / (1024 ** 3), 2),
            "disk_free_gb": round(disk.free / (1024 ** 3), 2),
            "disk_free_percent": round(
                (disk.free / disk.total) * 100, 2
            ) if disk.total else None,
            "data_directory": str(DATA_DIR),
            "data_directory_size_mb": data_footprint["size_mb"],
            "data_files_scanned": data_footprint["scanned_files"],
            "unreadable_data_files": data_footprint["unreadable_files"],
            "data_directory_exists": data_footprint["directory_exists"],
        },
    }


# =============================================================================
# 3. PIPELINE AUDIT DISCOVERY
# =============================================================================

def discover_pipeline_audits() -> dict:
    """Load available audit reports without assuming missing reports passed."""

    candidate_files = {
        "cell_10_5": REPORTS_DIR / "cell_10_5_comprehensive_integrity_report.json",
        "cell_10_2": REPORTS_DIR / "cell_10_2_source_integrity_report.json",
        "master_audit": REPORTS_DIR / "master_pipeline_audit_report.json",
    }

    discovered = {}

    for audit_name, audit_path in candidate_files.items():
        audit_data, read_status = safe_read_json(audit_path)

        discovered[audit_name] = {
            "path": str(audit_path),
            "read_status": read_status,
            "available": audit_data is not None,
            "data": audit_data,
        }

    return discovered


def extract_audit_verdict(audit_data: dict):
    """Extract a verdict across known report field conventions."""

    if not isinstance(audit_data, dict):
        return "UNKNOWN"

    possible_fields = [
        "overall_status",
        "overall_audit_status",
        "overall_pipeline_verdict",
        "pipeline_status",
        "status",
    ]

    for field in possible_fields:
        value = audit_data.get(field)

        if isinstance(value, str) and value.strip():
            return value.strip().upper()

    return "UNKNOWN"


# =============================================================================
# 4. SLA EVALUATION & ALERT MANAGEMENT
# =============================================================================

def evaluate_sla_and_generate_alerts(
    telemetry: dict,
    audit_inventory: dict
) -> dict:

    alerts = []
    warnings = []
    critical_count = 0

    def add_alert(severity, alert_type, message, source=None):
        nonlocal critical_count

        alert = {
            "severity": severity,
            "alert_type": alert_type,
            "message": message,
            "source": source,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        alerts.append(alert)

        if severity == "CRITICAL":
            critical_count += 1

    # -------------------------------------------------------------------------
    # SLA 1 — Process Memory
    # -------------------------------------------------------------------------

    rss_mb = telemetry["process"]["memory_rss_mb"]

    if rss_mb > SLA_CONFIG["memory_warning_mb"]:
        add_alert(
            "WARNING",
            "HIGH_PROCESS_MEMORY",
            f"Process RSS is {rss_mb} MB; configured threshold is "
            f"{SLA_CONFIG['memory_warning_mb']} MB.",
            "system_telemetry"
        )

    # -------------------------------------------------------------------------
    # SLA 2 — Disk Capacity
    # -------------------------------------------------------------------------

    disk_free_percent = telemetry["storage"]["disk_free_percent"]

    if disk_free_percent is not None:

        if disk_free_percent < SLA_CONFIG["disk_free_critical_percent"]:
            add_alert(
                "CRITICAL",
                "CRITICAL_DISK_CAPACITY",
                f"Free disk capacity is {disk_free_percent}%, below the "
                f"{SLA_CONFIG['disk_free_critical_percent']}% critical threshold.",
                "system_telemetry"
            )

        elif disk_free_percent < SLA_CONFIG["disk_free_warning_percent"]:
            add_alert(
                "WARNING",
                "LOW_DISK_CAPACITY",
                f"Free disk capacity is {disk_free_percent}%, below the "
                f"{SLA_CONFIG['disk_free_warning_percent']}% warning threshold.",
                "system_telemetry"
            )

    # -------------------------------------------------------------------------
    # SLA 3 — Latest Comprehensive Integrity Audit
    # -------------------------------------------------------------------------

    cell_10_5 = audit_inventory["cell_10_5"]

    if not cell_10_5["available"]:

        add_alert(
            "WARNING",
            "COMPREHENSIVE_AUDIT_UNAVAILABLE",
            "CELL 10.5 report is missing or unreadable. "
            "Pipeline data quality cannot be fully evaluated.",
            cell_10_5["path"]
        )

    else:
        verdict = extract_audit_verdict(cell_10_5["data"])

        if verdict in {"FAILED", "CRITICAL", "FAIL"}:
            add_alert(
                "CRITICAL",
                "DATA_INTEGRITY_AUDIT_FAILED",
                f"CELL 10.5 returned verdict: {verdict}.",
                cell_10_5["path"]
            )

        elif verdict in {"PARTIAL", "WARNING", "DEGRADED"}:
            add_alert(
                "WARNING",
                "DATA_INTEGRITY_AUDIT_PARTIAL",
                f"CELL 10.5 returned verdict: {verdict}. "
                "Review its warnings and unresolved source dependencies.",
                cell_10_5["path"]
            )

        elif verdict in {"PASSED", "SUCCESS"}:
            pass

        else:
            add_alert(
                "WARNING",
                "DATA_INTEGRITY_VERDICT_UNKNOWN",
                f"CELL 10.5 verdict could not be interpreted: {verdict}.",
                cell_10_5["path"]
            )

        # Critical issue count, if present in the report.
        report_data = cell_10_5["data"]

        reported_critical = report_data.get(
            "critical_issues",
            report_data.get("critical_issue_count")
        )

        if isinstance(reported_critical, int) and reported_critical > 0:
            add_alert(
                "CRITICAL",
                "CRITICAL_DATA_QUALITY_ISSUES",
                f"CEL 10.5 reports {reported_critical} critical issue(s).",
                cell_10_5["path"]
            )

        # WHO verification is a source dependency, not an automatic pipeline failure.
        who_status = str(
            report_data.get("who_verification_status", "")
        ).upper()

        if (
            who_status == "NOT_VERIFIED"
            or report_data.get("who_records") == 0
        ):
            add_alert(
                "WARNING",
                "WHO_SOURCE_NOT_VERIFIED",
                "WHO source is unavailable or not independently verified. "
                "Do not treat WHO coverage as complete.",
                cell_10_5["path"]
            )

    # -------------------------------------------------------------------------
    # SLA 4 — Master Audit Report
    # -------------------------------------------------------------------------

    master_audit = audit_inventory["master_audit"]

    if not master_audit["available"]:
        add_alert(
            "WARNING",
            "MASTER_AUDIT_UNAVAILABLE",
            "Master pipeline audit report is unavailable. "
            "No successful master verdict is assumed.",
            master_audit["path"]
        )

    else:
        master_verdict = extract_audit_verdict(master_audit["data"])

        if master_verdict in {"FAILED", "CRITICAL", "FAIL"}:
            add_alert(
                "CRITICAL",
                "MASTER_PIPELINE_AUDIT_FAILED",
                f"Master pipeline audit verdict: {master_verdict}.",
                master_audit["path"]
            )

        elif master_verdict in {"PARTIAL", "WARNING", "DEGRADED"}:
            add_alert(
                "WARNING",
                "MASTER_PIPELINE_AUDIT_PARTIAL",
                f"Master pipeline audit verdict: {master_verdict}.",
                master_audit["path"]
            )

        elif master_verdict == "UNKNOWN":
            add_alert(
                "WARNING",
                "MASTER_PIPELINE_VERDICT_UNKNOWN",
                "Master audit exists but its verdict field is not recognized.",
                master_audit["path"]
            )

    # -------------------------------------------------------------------------
    # Overall SLA Status
    # -------------------------------------------------------------------------

    warning_count = sum(
        alert["severity"] == "WARNING" for alert in alerts
    )

    if critical_count > 0:
        overall_status = "CRITICAL_SLA_BREACH"

    elif warning_count > 0:
        overall_status = "WARNING_SLA_BREACH"

    else:
        overall_status = "MEETS_SLA"

    return {
        "overall_sla_status": overall_status,
        "total_alerts_raised": len(alerts),
        "critical_alerts": critical_count,
        "warning_alerts": warning_count,
        "alerts_list": alerts,
        "thresholds": SLA_CONFIG,
        "evaluation_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }


# =============================================================================
# 5. EXECUTION HISTORY MANAGEMENT
# =============================================================================

def load_execution_history(history_path: Path) -> list:
    """Load and validate the previous execution history."""

    if not history_path.exists():
        return []

    if history_path.stat().st_size == 0:
        return []

    with open(history_path, "r", encoding="utf-8") as file:
        history = json.load(file)

    if not isinstance(history, list):
        raise ValueError(
            "Execution history must be a JSON list. "
            "Original file was preserved; manual review is required."
        )

    return history


def append_execution_run_history(
    telemetry: dict,
    sla_summary: dict,
    audit_inventory: dict,
    history_path: Path
) -> dict:

    history_records = load_execution_history(history_path)

    run_entry = {
        "run_id": RUN_ID,
        "cell_id": CELL_ID,
        "cell_name": CELL_NAME,
        "started_at_utc": START_TIME_UTC.isoformat(),
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "execution_status": "COMPLETED",
        "telemetry": telemetry,
        "sla_summary": sla_summary,
        "audit_sources": {
            name: {
                "available": details["available"],
                "read_status": details["read_status"],
                "path": details["path"],
            }
            for name, details in audit_inventory.items()
        },
    }

    history_records.append(run_entry)

    history_records = history_records[
        -SLA_CONFIG["max_history_runs"]:
    ]

    atomic_write_json(
        history_path,
        history_records
    )

    logger.info(
        f"Execution history updated: {history_path}"
    )

    return {
        "run_entry": run_entry,
        "total_retained_runs": len(history_records),
    }


# =============================================================================
# 6. OBSERVABILITY REPORT
# =============================================================================

def build_observability_report(
    telemetry: dict,
    sla_summary: dict,
    audit_inventory: dict,
    history_summary: dict
) -> dict:

    return {
        "report_metadata": {
            "cell_id": CELL_ID,
            "cell_name": CELL_NAME,
            "run_id": RUN_ID,
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "started_at_utc": START_TIME_UTC.isoformat(),
            "project_base_directory": str(BASE_DIR),
        },

        "execution": {
            "status": "COMPLETED",
            "observability_status": "ACTIVE",
            "history_runs_retained": history_summary["total_retained_runs"],
        },

        "system_telemetry": telemetry,

        "sla_evaluation": sla_summary,

        "audit_inventory": {
            name: {
                "path": details["path"],
                "available": details["available"],
                "read_status": details["read_status"],
                "verdict": (
                    extract_audit_verdict(details["data"])
                    if details["available"]
                    else "UNAVAILABLE"
                ),
            }
            for name, details in audit_inventory.items()
        },

        "artifacts": {
            "execution_history": str(HISTORY_FILE),
            "observability_report": str(OBSERVABILITY_REPORT),
            "master_bundle": str(MASTER_ZIP),
        },

        "governance_notes": [
            "Telemetry measures the current notebook process and runtime environment.",
            "A PARTIAL audit is reported as a warning, not silently treated as a pass.",
            "Missing audit reports are never interpreted as successful validation.",
            "Observability does not independently establish external source provenance.",
            "Telemetry and audit reports are operational evidence, not proof of production readiness.",
        ],
    }


# =============================================================================
# 7. SAFE MASTER ZIP REFRESH — NO DUPLICATE ARCHIVE ENTRIES
# =============================================================================

def update_master_bundle(
    zip_path: Path,
    files_to_add: list
) -> dict:
    """
    Rebuild the ZIP atomically.
    Existing entries are preserved, while updated target entries are replaced.
    """

    if not zip_path.exists():
        return {
            "status": "NOT_UPDATED",
            "reason": "MASTER_BUNDLE_NOT_FOUND",
            "path": str(zip_path),
            "files_added": [],
        }

    temp_zip = zip_path.with_name(
        f".{zip_path.stem}.{uuid.uuid4().hex}.tmp.zip"
    )

    replacement_map = {}

    for file_path in files_to_add:
        file_path = Path(file_path)

        if file_path.exists() and file_path.is_file():
            archive_name = file_path.relative_to(BASE_DIR).as_posix()
            replacement_map[archive_name] = file_path

    try:
        # Preserve existing entries except the files being refreshed.
        with zipfile.ZipFile(zip_path, "r") as source_zip:
            bad_entry = source_zip.testzip()

            if bad_entry is not None:
                raise RuntimeError(
                    f"Existing master ZIP failed integrity check at: {bad_entry}"
                )

            with zipfile.ZipFile(
                temp_zip,
                "w",
                compression=zipfile.ZIP_DEFLATED,
                compresslevel=6
            ) as target_zip:

                for item in source_zip.infolist():

                    if item.filename in replacement_map:
                        continue

                    # Preserve the existing archive entry.
                    target_zip.writestr(
                        item,
                        source_zip.read(item.filename)
                    )

                # Add the latest versions exactly once.
                for archive_name, file_path in replacement_map.items():
                    target_zip.write(
                        file_path,
                        arcname=archive_name
                    )

        # Validate the rebuilt archive before replacing the original.
        with zipfile.ZipFile(temp_zip, "r") as validation_zip:
            bad_entry = validation_zip.testzip()

            if bad_entry is not None:
                raise RuntimeError(
                    f"Rebuilt ZIP integrity validation failed: {bad_entry}"
                )

            archive_names = validation_zip.namelist()

            duplicate_names = (
                len(archive_names) - len(set(archive_names))
            )

            if duplicate_names > 0:
                raise RuntimeError(
                    f"Rebuilt ZIP contains {duplicate_names} duplicate entries."
                )

        os.replace(temp_zip, zip_path)

        return {
            "status": "UPDATED",
            "path": str(zip_path),
            "files_added": list(replacement_map.keys()),
            "duplicate_entries": 0,
            "integrity_check": "PASSED",
            "size_mb": round(zip_path.stat().st_size / (1024 ** 2), 2),
        }

    finally:
        if temp_zip.exists():
            temp_zip.unlink(missing_ok=True)


# =============================================================================
# 8. EXECUTE OBSERVABILITY ENGINE
# =============================================================================

observability_execution_error = None

try:

    logger.info("Step 1/7 — Capturing system telemetry...")
    telemetry_data = capture_system_telemetry()

    logger.info("Step 2/7 — Discovering pipeline audit reports...")
    audit_inventory = discover_pipeline_audits()

    logger.info("Step 3/7 — Evaluating SLA rules and alerts...")
    sla_summary_data = evaluate_sla_and_generate_alerts(
        telemetry_data,
        audit_inventory
    )

    logger.info("Step 4/7 — Updating persistent execution history...")
    history_summary = append_execution_run_history(
        telemetry_data,
        sla_summary_data,
        audit_inventory,
        HISTORY_FILE
    )

    logger.info("Step 5/7 — Building observability report...")
    observability_report = build_observability_report(
        telemetry_data,
        sla_summary_data,
        audit_inventory,
        history_summary
    )

    # Save the report before packaging so it can be included in the bundle.
    atomic_write_json(
        OBSERVABILITY_REPORT,
        observability_report
    )

    logger.info("Step 6/7 — Refreshing master production bundle...")
    bundle_summary = update_master_bundle(
        MASTER_ZIP,
        [
            HISTORY_FILE,
            OBSERVABILITY_REPORT,
        ]
    )

    # Add package status to the report after the ZIP operation.
    observability_report["artifacts"]["master_bundle_update"] = (
        bundle_summary
    )

    atomic_write_json(
        OBSERVABILITY_REPORT,
        observability_report
    )

    # Refresh the report entry once more so the ZIP contains its final version.
    if bundle_summary["status"] == "UPDATED":
        bundle_summary = update_master_bundle(
            MASTER_ZIP,
            [
                HISTORY_FILE,
                OBSERVABILITY_REPORT,
            ]
        )

    logger.info("Step 7/7 — Finalizing observability execution...")

except Exception as exc:

    observability_execution_error = {
        "error_type": type(exc).__name__,
        "error_message": str(exc),
        "traceback": traceback.format_exc(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }

    logger.exception(
        f"CELL {CELL_ID} encountered an execution error."
    )

    # Preserve an explicit failure report if enough context exists.
    failure_report = {
        "cell_id": CELL_ID,
        "run_id": RUN_ID,
        "execution_status": "FAILED",
        "error": observability_execution_error,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }

    atomic_write_json(
        OBSERVABILITY_REPORT,
        failure_report
    )

    raise


# =============================================================================
# 9. FINAL EXECUTION SUMMARY
# =============================================================================

print("\n" + "=" * 88)
print(f"CELL {CELL_ID} — ENTERPRISE OBSERVABILITY EXECUTION SUMMARY")
print("=" * 88)

print(f"Run ID:                    {RUN_ID}")
print(f"Execution Status:          COMPLETED")

print("\nSYSTEM TELEMETRY")
print("-" * 88)
print(
    f"Process RSS Memory:        "
    f"{telemetry_data['process']['memory_rss_mb']} MB"
)
print(
    f"Process CPU:               "
    f"{telemetry_data['process']['cpu_percent']}%"
)
print(
    f"Data Directory Footprint:  "
    f"{telemetry_data['storage']['data_directory_size_mb']} MB"
)
print(
    f"Disk Free:                 "
    f"{telemetry_data['storage']['disk_free_percent']}%"
)

print("\nSLA EVALUATION")
print("-" * 88)
print(f"Overall SLA Status:        {sla_summary_data['overall_sla_status']}")
print(f"Total Alerts:              {sla_summary_data['total_alerts_raised']}")
print(f"Critical Alerts:           {sla_summary_data['critical_alerts']}")
print(f"Warning Alerts:            {sla_summary_data['warning_alerts']}")

if sla_summary_data["alerts_list"]:
    print("\nALERT DETAILS")

    for alert in sla_summary_data["alerts_list"]:
        print(
            f"[{alert['severity']}] "
            f"{alert['alert_type']}: {alert['message']}"
        )
else:
    print("No SLA alerts were raised.")

print("\nAUDIT REPORT AVAILABILITY")
print("-" * 88)

for audit_name, audit_details in audit_inventory.items():
    if audit_details["available"]:
        verdict = extract_audit_verdict(audit_details["data"])
        print(f"{audit_name}: AVAILABLE — {verdict}")
    else:
        print(
            f"{audit_name}: UNAVAILABLE — "
            f"{audit_details['read_status']}"
        )

print("\nPERSISTENT ARTIFACTS")
print("-" * 88)
print(f"Execution History:         {HISTORY_FILE}")
print(f"Observability Report:      {OBSERVABILITY_REPORT}")
print(f"Master Bundle:             {bundle_summary['status']}")

if bundle_summary["status"] == "UPDATED":
    print(f"Master ZIP Size:           {bundle_summary['size_mb']} MB")
    print(f"ZIP Integrity:             {bundle_summary['integrity_check']}")
    print(f"Duplicate ZIP Entries:     {bundle_summary['duplicate_entries']}")
else:
    print(f"Master ZIP Note:           {bundle_summary['reason']}")

print("\n" + "=" * 88)
print(f"CELL {CELL_ID} EXECUTION COMPLETED")
print("=" * 88)


CELL 10.6 — ENTERPRISE OBSERVABILITY EXECUTION SUMMARY
Run ID:                    RUN-20261102T155006-167DD033B3
Execution Status:          COMPLETED

SYSTEM TELEMETRY
----------------------------------------------------------------------------------------
Process RSS Memory:        2566.74 MB
Process CPU:               0.0%
Data Directory Footprint:  429.91 MB
Disk Free:                 99.49%

SLA EVALUATION
----------------------------------------------------------------------------------------
Overall SLA Status:        WARNING_SLA_BREACH
Total Alerts:              2
Critical Alerts:           0
Warning Alerts:            2

ALERT DETAILS
[WARNING] HIGH_PROCESS_MEMORY: Process RSS is 2566.74 MB; configured threshold is 2000 MB.
[WARNING] DATA_INTEGRITY_AUDIT_PARTIAL: CELL 10.5 returned verdict: PARTIAL. Review its warnings and unresolved source dependencies.

AUDIT REPORT AVAILABILITY
----------------------------------------------------------------------------------------
cell_10_

In [21]:

# =============================================================================
# CELL 11 — FINAL ENTERPRISE INTEGRATION VERIFICATION
# RELEASE READINESS & GOVERNANCE AUDIT
# =============================================================================

import os
import sys
import json
import csv
import hashlib
import zipfile
import logging
import traceback

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter


# =============================================================================
# 0. CONFIGURATION
# =============================================================================

CELL_ID = "11"
CELL_NAME = "FINAL ENTERPRISE RELEASE READINESS AUDIT"

BASE_DIR = Path(globals().get("BASE_DIR", "/content")).resolve()

_existing_dirs = globals().get("DATA_DIRS", {})

DATA_DIR = Path(
    _existing_dirs.get("data", BASE_DIR / "data")
).resolve()

REPORTS_DIR = Path(
    _existing_dirs.get("reports", BASE_DIR / "reports")
).resolve()

BRONZE_DIR = Path(
    _existing_dirs.get("bronze", DATA_DIR / "bronze")
).resolve()

SILVER_DIR = Path(
    _existing_dirs.get("silver", DATA_DIR / "silver")
).resolve()

GOLD_DIR = Path(
    _existing_dirs.get("gold", DATA_DIR / "gold")
).resolve()

SILVER_PARQUET_DIR = DATA_DIR / "silver_parquet"
GOLD_PARQUET_DIR = DATA_DIR / "gold_parquet"

MASTER_ZIP = BASE_DIR / "enterprise_master_production_bundle.zip"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

CERTIFICATE_FILE = (
    REPORTS_DIR / "final_release_readiness_certificate.json"
)

AUDIT_REPORT_FILE = (
    REPORTS_DIR / "cell_11_final_integration_audit.json"
)

# Logger fallback
if "logger" not in globals():
    logger = logging.getLogger("cell_11_release_audit")
    if not logger.handlers:
        handler = logging.StreamHandler()
        logger.addHandler(handler)
    logger.setLevel(logging.INFO)

logger.info(
    f"Executing CELL {CELL_ID}: {CELL_NAME}"
)

RUN_ID = (
    "RELEASE-AUDIT-"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
)


# =============================================================================
# 1. GENERAL UTILITIES
# =============================================================================

checks = []
critical_issues = []
high_issues = []
medium_issues = []
low_issues = []


def add_check(
    check_id,
    category,
    name,
    status,
    severity,
    details,
    evidence=None
):
    """Register one auditable verification result."""

    record = {
        "check_id": check_id,
        "category": category,
        "check_name": name,
        "status": status,
        "severity": severity,
        "details": str(details),
        "evidence": evidence or {},
        "timestamp_utc": datetime.now(timezone.utc).isoformat()
    }

    checks.append(record)

    if status in {"FAILED", "ERROR"}:
        issue = {
            "check_id": check_id,
            "category": category,
            "severity": severity,
            "name": name,
            "details": str(details)
        }

        if severity == "CRITICAL":
            critical_issues.append(issue)
        elif severity == "HIGH":
            high_issues.append(issue)
        elif severity == "MEDIUM":
            medium_issues.append(issue)
        else:
            low_issues.append(issue)

    elif status in {"WARNING", "PARTIAL", "NOT_VERIFIED"}:
        issue = {
            "check_id": check_id,
            "category": category,
            "severity": severity,
            "name": name,
            "details": str(details)
        }

        if severity == "HIGH":
            high_issues.append(issue)
        elif severity == "MEDIUM":
            medium_issues.append(issue)
        else:
            low_issues.append(issue)

    return record


def read_json(path):
    """Safely read a JSON file."""

    path = Path(path)

    if not path.exists():
        return None, "NOT_FOUND"

    try:
        with open(path, "r", encoding="utf-8") as file:
            return json.load(file), "SUCCESS"

    except Exception as exc:
        return None, f"READ_ERROR: {type(exc).__name__}: {exc}"


def file_is_valid(path):
    path = Path(path)
    return path.exists() and path.is_file() and path.stat().st_size > 0


def count_files(directory, extensions=None):
    """Count files recursively, optionally filtering extensions."""

    directory = Path(directory)

    if not directory.exists():
        return 0

    total = 0

    for path in directory.rglob("*"):
        if not path.is_file():
            continue

        if extensions is None or path.suffix.lower() in extensions:
            total += 1

    return total


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def extract_verdict(report):
    """Support multiple audit report field conventions."""

    if not isinstance(report, dict):
        return "UNKNOWN"

    fields = [
        "overall_status",
        "overall_audit_status",
        "overall_pipeline_verdict",
        "final_sign_off_status",
        "status"
    ]

    for field in fields:
        value = report.get(field)

        if isinstance(value, str) and value.strip():
            return value.strip().upper()

    return "UNKNOWN"


def atomic_write_json(path, payload):
    """Write a JSON artifact atomically."""

    path = Path(path)
    temp_path = path.with_name(path.name + ".tmp")

    try:
        with open(temp_path, "w", encoding="utf-8") as file:
            json.dump(
                payload,
                file,
                ensure_ascii=False,
                indent=2,
                default=str
            )
            file.flush()
            os.fsync(file.fileno())

        os.replace(temp_path, path)

    finally:
        if temp_path.exists():
            temp_path.unlink(missing_ok=True)


# =============================================================================
# 2. SOURCE INTEGRITY & PROVENANCE REPORT REVIEW
# =============================================================================

logger.info("Step 1/8 — Reviewing source integrity reports...")

cell_10_5_path = (
    REPORTS_DIR / "cell_10_5_comprehensive_integrity_report.json"
)

cell_10_2_path = (
    REPORTS_DIR / "cell_10_2_source_integrity_report.json"
)

cell_10_5, status_10_5 = read_json(cell_10_5_path)
cell_10_2, status_10_2 = read_json(cell_10_2_path)

if cell_10_5 is None:
    add_check(
        "REL-001",
        "SOURCE_INTEGRITY",
        "CELL 10.5 Comprehensive Audit",
        "FAILED",
        "CRITICAL",
        f"Comprehensive integrity report unavailable: {status_10_5}",
        {"path": str(cell_10_5_path)}
    )
else:
    verdict = extract_verdict(cell_10_5)

    if verdict in {"FAILED", "FAIL", "CRITICAL"}:
        result = "FAILED"
    elif verdict in {"PARTIAL", "WARNING", "DEGRADED"}:
        result = "WARNING"
    elif verdict in {"PASSED", "SUCCESS"}:
        result = "PASSED"
    else:
        result = "NOT_VERIFIED"

    add_check(
        "REL-001",
        "SOURCE_INTEGRITY",
        "CELL 10.5 Comprehensive Audit",
        result,
        "HIGH",
        f"CELL 10.5 audit verdict: {verdict}",
        {
            "path": str(cell_10_5_path),
            "verdict": verdict
        }
    )


if cell_10_2 is None:
    add_check(
        "REL-002",
        "SOURCE_INTEGRITY",
        "CELL 10.2 Advanced Source Audit",
        "WARNING",
        "MEDIUM",
        f"Advanced source audit unavailable: {status_10_2}",
        {"path": str(cell_10_2_path)}
    )
else:
    verdict = extract_verdict(cell_10_2)

    result = (
        "WARNING"
        if verdict in {"PARTIAL", "WARNING", "DEGRADED"}
        else "FAILED"
        if verdict in {"FAILED", "FAIL", "CRITICAL"}
        else "PASSED"
        if verdict in {"PASSED", "SUCCESS"}
        else "NOT_VERIFIED"
    )

    add_check(
        "REL-002",
        "SOURCE_INTEGRITY",
        "CELL 10.2 Advanced Source Audit",
        result,
        "MEDIUM",
        f"CELL 10.2 audit verdict: {verdict}",
        {
            "path": str(cell_10_2_path),
            "verdict": verdict
        }
    )


# WHO provenance check
who_path = (
    BRONZE_DIR / "bronze_who" / "who_eml_raw.json"
)

who_data, who_read_status = read_json(who_path)

who_metadata = (
    who_data.get("metadata", {})
    if isinstance(who_data, dict)
    else {}
)

who_records = (
    who_data.get("results", [])
    if isinstance(who_data, dict)
    else []
)

who_verification = str(
    who_metadata.get("verification_status", "NOT_VERIFIED")
).upper()

who_synthetic_flag = who_metadata.get(
    "has_synthetic_data",
    None
)

if who_data is None:
    who_result = "WARNING"
    who_details = "WHO raw source file is unavailable."

elif who_synthetic_flag is True:
    who_result = "FAILED"
    who_details = "WHO metadata explicitly flags synthetic data."

elif who_verification != "VERIFIED":
    who_result = "NOT_VERIFIED"
    who_details = (
        f"WHO verification status is {who_verification}; "
        f"records={len(who_records) if isinstance(who_records, list) else 'UNKNOWN'}."
    )

else:
    who_result = "PASSED"
    who_details = "WHO source metadata reports VERIFIED."

add_check(
    "REL-003",
    "SOURCE_INTEGRITY",
    "WHO Source Provenance",
    who_result,
    "MEDIUM",
    who_details,
    {
        "path": str(who_path),
        "read_status": who_read_status,
        "verification_status": who_verification,
        "records": len(who_records) if isinstance(who_records, list) else None,
        "synthetic_flag": who_synthetic_flag
    }
)


# =============================================================================
# 3. BRONZE, SILVER & GOLD LAYER VERIFICATION
# =============================================================================

logger.info("Step 2/8 — Validating Medallion Architecture layers...")

layer_directories = {
    "BRONZE": BRONZE_DIR,
    "SILVER": SILVER_DIR,
    "GOLD": GOLD_DIR
}

layer_inventory = {}

for layer_name, directory in layer_directories.items():

    exists = directory.exists()

    csv_count = count_files(directory, {".csv"})
    json_count = count_files(directory, {".json"})
    parquet_count = count_files(directory, {".parquet"})

    total_files = count_files(directory)

    layer_inventory[layer_name] = {
        "directory": str(directory),
        "directory_exists": exists,
        "total_files": total_files,
        "csv_files": csv_count,
        "json_files": json_count,
        "parquet_files": parquet_count
    }

    if not exists:
        result = "FAILED"
        details = f"{layer_name} directory does not exist."

    elif total_files == 0:
        result = "FAILED"
        details = f"{layer_name} directory exists but contains no files."

    else:
        result = "PASSED"
        details = (
            f"{layer_name} contains {total_files} files "
            f"({csv_count} CSV, {json_count} JSON, {parquet_count} Parquet)."
        )

    add_check(
        f"REL-LAYER-{layer_name}",
        "MEDALLION_ARCHITECTURE",
        f"{layer_name} Layer Inventory",
        result,
        "CRITICAL",
        details,
        layer_inventory[layer_name]
    )


# =============================================================================
# 4. SILVER CSV & GOLD CONTENT VALIDATION
# =============================================================================

logger.info("Step 3/8 — Validating Silver CSV and Gold datasets...")

csv_inventory = {}

for directory_name, directory in [
    ("SILVER", SILVER_DIR),
    ("GOLD", GOLD_DIR)
]:

    csv_files = (
        list(directory.rglob("*.csv"))
        if directory.exists()
        else []
    )

    for csv_path in csv_files:

        relative_name = str(
            csv_path.relative_to(DATA_DIR)
        )

        try:
            if csv_path.stat().st_size == 0:
                csv_inventory[relative_name] = {
                    "status": "EMPTY",
                    "rows": 0,
                    "columns": 0
                }
                continue

            with open(
                csv_path,
                "r",
                encoding="utf-8-sig",
                newline=""
            ) as file:

                reader = csv.reader(file)
                header = next(reader, None)

                if not header:
                    csv_inventory[relative_name] = {
                        "status": "NO_HEADER",
                        "rows": 0,
                        "columns": 0
                    }
                    continue

                row_count = sum(1 for _ in reader)

            csv_inventory[relative_name] = {
                "status": "READABLE_NONEMPTY" if row_count > 0 else "HEADER_ONLY",
                "rows": row_count,
                "columns": len(header),
                "headers": header
            }

        except Exception as exc:
            csv_inventory[relative_name] = {
                "status": "READ_ERROR",
                "error": f"{type(exc).__name__}: {exc}"
            }


readable_csv_count = sum(
    item["status"] == "READABLE_NONEMPTY"
    for item in csv_inventory.values()
)

empty_csv_count = sum(
    item["status"] in {"EMPTY", "NO_HEADER", "HEADER_ONLY"}
    for item in csv_inventory.values()
)

csv_error_count = sum(
    item["status"] == "READ_ERROR"
    for item in csv_inventory.values()
)

if csv_error_count > 0:
    csv_result = "FAILED"
elif empty_csv_count > 0:
    csv_result = "WARNING"
else:
    csv_result = "PASSED"

add_check(
    "REL-004",
    "DATASET_VALIDATION",
    "Silver and Gold CSV Readability",
    csv_result,
    "HIGH",
    (
        f"Readable nonempty CSVs: {readable_csv_count}; "
        f"empty/header-only CSVs: {empty_csv_count}; "
        f"read errors: {csv_error_count}."
    ),
    {
        "datasets": csv_inventory,
        "readable_nonempty": readable_csv_count,
        "empty_or_header_only": empty_csv_count,
        "read_errors": csv_error_count
    }
)


# Required Gold analytics dataset
gold_analytics_path = GOLD_DIR / "gold_pharma_analytics.csv"

if not file_is_valid(gold_analytics_path):
    gold_result = "FAILED"
    gold_details = "Gold pharmaceutical analytics dataset is missing or empty."
    gold_rows = 0

else:
    gold_info = csv_inventory.get(
        str(gold_analytics_path.relative_to(DATA_DIR)),
        {}
    )

    gold_rows = gold_info.get("rows", 0)

    if gold_info.get("status") == "READABLE_NONEMPTY":
        gold_result = "PASSED"
        gold_details = f"Gold analytics contains {gold_rows} data rows."
    else:
        gold_result = "FAILED"
        gold_details = (
            f"Gold analytics is not readable as a nonempty CSV: "
            f"{gold_info.get('status', 'UNKNOWN')}."
        )

add_check(
    "REL-005",
    "GOLD_LAYER",
    "Gold Analytics Content",
    gold_result,
    "CRITICAL",
    gold_details,
    {
        "path": str(gold_analytics_path),
        "rows": gold_rows
    }
)


# RAG corpus is optional but its absence is visible.
gold_rag_path = GOLD_DIR / "gold_rag_knowledge_corpus.csv"

if file_is_valid(gold_rag_path):
    rag_info = csv_inventory.get(
        str(gold_rag_path.relative_to(DATA_DIR)),
        {}
    )

    rag_rows = rag_info.get("rows", 0)

    rag_result = (
        "PASSED"
        if rag_info.get("status") == "READABLE_NONEMPTY"
        else "WARNING"
    )

    rag_details = f"RAG corpus contains {rag_rows} data rows."

else:
    rag_rows = 0
    rag_result = "WARNING"
    rag_details = "Gold RAG knowledge corpus is missing or empty."

add_check(
    "REL-006",
    "GOLD_LAYER",
    "Gold RAG Knowledge Corpus",
    rag_result,
    "MEDIUM",
    rag_details,
    {
        "path": str(gold_rag_path),
        "rows": rag_rows
    }
)


# =============================================================================
# 5. PARQUET VALIDATION
# =============================================================================

logger.info("Step 4/8 — Validating Parquet storage...")

parquet_inventory = {}

parquet_directories = {
    "SILVER_PARQUET": SILVER_PARQUET_DIR,
    "GOLD_PARQUET": GOLD_PARQUET_DIR
}

for name, directory in parquet_directories.items():

    parquet_files = (
        list(directory.rglob("*.parquet"))
        if directory.exists()
        else []
    )

    readable = 0
    unreadable = 0
    file_details = []

    for parquet_file in parquet_files:

        try:
            if parquet_file.stat().st_size == 0:
                raise ValueError("Zero-byte Parquet file.")

            # Prefer PyArrow for actual metadata/schema inspection.
            try:
                import pyarrow.parquet as pq

                parquet_metadata = pq.read_metadata(str(parquet_file))

                rows = parquet_metadata.num_rows
                columns = parquet_metadata.num_columns

                # Read one row group where available to verify actual data access.
                if rows > 0:
                    parquet_table = pq.read_table(
                        str(parquet_file)
                    )

                file_status = "READABLE"

            except ImportError:
                # Without PyArrow, only the presence and size are verified.
                rows = None
                columns = None
                file_status = "NOT_FULLY_VERIFIED"

            if file_status == "READABLE":
                readable += 1
            else:
                unreadable += 1

            file_details.append({
                "path": str(parquet_file),
                "status": file_status,
                "rows": rows,
                "columns": columns,
                "size_bytes": parquet_file.stat().st_size
            })

        except Exception as exc:
            unreadable += 1

            file_details.append({
                "path": str(parquet_file),
                "status": "READ_ERROR",
                "error": f"{type(exc).__name__}: {exc}"
            })

    parquet_inventory[name] = {
        "directory": str(directory),
        "files_found": len(parquet_files),
        "readable": readable,
        "not_fully_verified_or_unreadable": unreadable,
        "files": file_details
    }

    if not parquet_files:
        result = "WARNING"
        details = f"No Parquet files found in {name}."

    elif unreadable > 0:
        result = "WARNING"
        details = (
            f"{readable} files fully readable; "
            f"{unreadable} not fully verified or unreadable."
        )

    else:
        result = "PASSED"
        details = f"All {readable} Parquet files passed the available checks."

    add_check(
        f"REL-PARQUET-{name}",
        "PARQUET_STORAGE",
        f"{name} Validation",
        result,
        "MEDIUM",
        details,
        parquet_inventory[name]
    )


# =============================================================================
# 6. CATALOG, LINEAGE, DASHBOARD & OBSERVABILITY
# =============================================================================

logger.info("Step 5/8 — Reviewing governance and observability artifacts...")

required_artifacts = {
    "DATA_CATALOG": REPORTS_DIR / "DATA_CATALOG_DICTIONARY.md",
    "LINEAGE_DAG": REPORTS_DIR / "data_lineage_dag.html",
    "AUDIT_DASHBOARD": REPORTS_DIR / "enterprise_audit_dashboard.html",
    "OBSERVABILITY_REPORT": REPORTS_DIR / "cell_10_6_observability_report.json",
    "EXECUTION_HISTORY": REPORTS_DIR / "pipeline_execution_history.json",
}

artifact_inventory = {}

for name, path in required_artifacts.items():

    exists = file_is_valid(path)

    artifact_inventory[name] = {
        "path": str(path),
        "exists": exists,
        "size_bytes": path.stat().st_size if exists else 0,
        "sha256": sha256_file(path) if exists else None
    }

    add_check(
        f"REL-ARTIFACT-{name}",
        "GOVERNANCE_ARTIFACTS",
        f"{name} Availability",
        "PASSED" if exists else "WARNING",
        "MEDIUM",
        (
            "Artifact exists and is nonempty."
            if exists
            else "Artifact is missing or empty."
        ),
        artifact_inventory[name]
    )


# Observability report verdict and SLA
observability_path = (
    REPORTS_DIR / "cell_10_6_observability_report.json"
)

observability_data, obs_read_status = read_json(observability_path)

if observability_data is None:
    add_check(
        "REL-OBS-001",
        "OBSERVABILITY",
        "Observability Report Readability",
        "FAILED",
        "HIGH",
        f"Observability report unavailable: {obs_read_status}"
    )

else:
    sla = observability_data.get("sla_evaluation", {})
    sla_status = str(
        sla.get("overall_sla_status", "UNKNOWN")
    ).upper()

    if sla_status == "CRITICAL_SLA_BREACH":
        obs_result = "FAILED"
    elif sla_status == "WARNING_SLA_BREACH":
        obs_result = "WARNING"
    elif sla_status == "MEETS_SLA":
        obs_result = "PASSED"
    else:
        obs_result = "NOT_VERIFIED"

    add_check(
        "REL-OBS-002",
        "OBSERVABILITY",
        "SLA Evaluation",
        obs_result,
        "MEDIUM",
        f"Latest recorded SLA status: {sla_status}.",
        {
            "sla_status": sla_status,
            "total_alerts": sla.get("total_alerts_raised"),
            "critical_alerts": sla.get("critical_alerts"),
            "warning_alerts": sla.get("warning_alerts")
        }
    )


# Execution history structure
history_path = REPORTS_DIR / "pipeline_execution_history.json"

history_data, history_status = read_json(history_path)

if isinstance(history_data, list) and len(history_data) > 0:

    history_result = "PASSED"
    history_details = (
        f"Execution history contains {len(history_data)} run records."
    )

else:
    history_result = "WARNING"
    history_details = (
        f"Execution history missing, empty or invalid: {history_status}."
    )

add_check(
    "REL-OBS-003",
    "OBSERVABILITY",
    "Execution History Integrity",
    history_result,
    "MEDIUM",
    history_details,
    {
        "path": str(history_path),
        "records": len(history_data) if isinstance(history_data, list) else 0
    }
)


# =============================================================================
# 7. MASTER BUNDLE INTEGRITY
# =============================================================================

logger.info("Step 6/8 — Validating master production bundle...")

bundle_details = {
    "path": str(MASTER_ZIP),
    "exists": MASTER_ZIP.exists(),
    "integrity": "NOT_CHECKED",
    "duplicate_entries": None,
    "entry_count": 0,
    "required_entries": {},
    "size_mb": None
}

if not MASTER_ZIP.exists():

    bundle_result = "FAILED"
    bundle_message = "Master production ZIP does not exist."

else:

    try:
        with zipfile.ZipFile(MASTER_ZIP, "r") as archive:

            bad_entry = archive.testzip()
            names = archive.namelist()

            duplicate_names = {
                name: count
                for name, count in Counter(names).items()
                if count > 1
            }

            required_bundle_files = [
                "reports/pipeline_execution_history.json",
                "reports/cell_10_6_observability_report.json"
            ]

            required_status = {
                item: item in names
                for item in required_bundle_files
            }

            bundle_details.update({
                "integrity": "PASSED" if bad_entry is None else "FAILED",
                "corrupt_entry": bad_entry,
                "entry_count": len(names),
                "duplicate_entries": len(duplicate_names),
                "duplicate_entry_names": duplicate_names,
                "required_entries": required_status,
                "size_mb": round(
                    MASTER_ZIP.stat().st_size / (1024 ** 2), 2
                )
            })

            if bad_entry is not None:
                bundle_result = "FAILED"
                bundle_message = f"ZIP corruption detected: {bad_entry}"

            elif duplicate_names:
                bundle_result = "FAILED"
                bundle_message = (
                    f"ZIP contains {len(duplicate_names)} duplicate entry names."
                )

            elif not all(required_status.values()):
                bundle_result = "WARNING"
                bundle_message = (
                    "ZIP is readable, but one or more latest observability "
                    "artifacts are missing."
                )

            else:
                bundle_result = "PASSED"
                bundle_message = (
                    f"ZIP integrity passed; {len(names)} unique entries."
                )

    except Exception as exc:
        bundle_result = "FAILED"
        bundle_message = f"{type(exc).__name__}: {exc}"

add_check(
    "REL-ZIP-001",
    "MASTER_BUNDLE",
    "Master ZIP Integrity & Uniqueness",
    bundle_result,
    "CRITICAL",
    bundle_message,
    bundle_details
)


# =============================================================================
# 8. RELEASE READINESS DECISION
# =============================================================================

logger.info("Step 7/8 — Calculating final release readiness...")

status_counts = Counter(
    item["status"] for item in checks
)

# A critical failure blocks release.
if critical_issues:
    release_status = "NOT_READY"

# High-severity failures also block release.
elif high_issues:
    release_status = "NOT_READY"

# Warnings and unverified evidence prevent an unconditional READY verdict.
elif any(
    item["status"] in {
        "WARNING",
        "PARTIAL",
        "NOT_VERIFIED"
    }
    for item in checks
):
    release_status = "READY_WITH_WARNINGS"

else:
    release_status = "READY"


if release_status == "READY":
    release_recommendation = (
        "The evaluated release checks passed. "
        "This result does not independently certify regulatory compliance "
        "or production performance under real workloads."
    )

elif release_status == "READY_WITH_WARNINGS":
    release_recommendation = (
        "The evaluated artifacts are available, but one or more warnings "
        "or unverified dependencies remain. Review them before production release."
    )

else:
    release_recommendation = (
        "Release is blocked by critical or high-severity findings. "
        "Resolve the listed issues and rerun the audit."
    )


# =============================================================================
# 9. FINAL AUDIT REPORT & RELEASE CERTIFICATE
# =============================================================================

logger.info("Step 8/8 — Generating final release artifacts...")

final_payload = {
    "metadata": {
        "cell_id": CELL_ID,
        "cell_name": CELL_NAME,
        "run_id": RUN_ID,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "project_directory": str(BASE_DIR),
        "audit_type": "FINAL_RELEASE_READINESS_ASSESSMENT",
        "read_only_audit": True
    },

    "release_decision": {
        "status": release_status,
        "recommendation": release_recommendation,
        "production_grade_100_percent_claim": False
    },

    "summary": {
        "total_checks": len(checks),
        "passed": status_counts.get("PASSED", 0),
        "failed": status_counts.get("FAILED", 0) + status_counts.get("ERROR", 0),
        "warnings": status_counts.get("WARNING", 0),
        "partial": status_counts.get("PARTIAL", 0),
        "not_verified": status_counts.get("NOT_VERIFIED", 0),
        "critical_issues": len(critical_issues),
        "high_issues": len(high_issues),
        "medium_issues": len(medium_issues),
        "low_issues": len(low_issues)
    },

    "layer_inventory": layer_inventory,
    "csv_inventory": csv_inventory,
    "parquet_inventory": parquet_inventory,
    "artifact_inventory": artifact_inventory,
    "master_bundle": bundle_details,

    "issues": {
        "critical": critical_issues,
        "high": high_issues,
        "medium": medium_issues,
        "low": low_issues
    },

    "checks": checks,

    "limitations": [
        "File existence does not prove source authenticity or business correctness.",
        "WHO provenance requires independent verification against an official source.",
        "A unique product_id in one snapshot does not prove longitudinal key stability.",
        "Repeated product_ndc values are not automatically treated as erroneous.",
        "CSV readability does not prove semantic data quality.",
        "Parquet files are fully read-tested only when PyArrow is available.",
        "The audit does not independently test production-scale throughput or availability.",
        "The audit does not certify HIPAA, GDPR or other regulatory compliance."
    ]
}

# Save both audit and certificate.
atomic_write_json(
    AUDIT_REPORT_FILE,
    final_payload
)

atomic_write_json(
    CERTIFICATE_FILE,
    final_payload
)


# =============================================================================
# 10. UPDATE MASTER ZIP SAFELY — REPLACE ENTRIES, DO NOT APPEND DUPLICATES
# =============================================================================

zip_update_status = "NOT_UPDATED"
zip_update_message = "Master ZIP was not updated."

if MASTER_ZIP.exists():

    temp_zip = MASTER_ZIP.with_name(
        MASTER_ZIP.stem + "_cell11_temp.zip"
    )

    files_to_add = [
        AUDIT_REPORT_FILE,
        CERTIFICATE_FILE
    ]

    replacement_map = {
        path.relative_to(BASE_DIR).as_posix(): path
        for path in files_to_add
        if path.exists()
    }

    try:

        with zipfile.ZipFile(MASTER_ZIP, "r") as source_zip:

            if source_zip.testzip() is not None:
                raise RuntimeError(
                    "Original Master ZIP failed integrity validation."
                )

            with zipfile.ZipFile(
                temp_zip,
                "w",
                compression=zipfile.ZIP_DEFLATED,
                compresslevel=6
            ) as target_zip:

                for item in source_zip.infolist():

                    if item.filename in replacement_map:
                        continue

                    target_zip.writestr(
                        item,
                        source_zip.read(item.filename)
                    )

                for archive_name, file_path in replacement_map.items():
                    target_zip.write(
                        file_path,
                        arcname=archive_name
                    )

        # Validate before replacing original.
        with zipfile.ZipFile(temp_zip, "r") as validation_zip:

            if validation_zip.testzip() is not None:
                raise RuntimeError(
                    "Updated Master ZIP failed integrity validation."
                )

            names = validation_zip.namelist()

            if len(names) != len(set(names)):
                raise RuntimeError(
                    "Duplicate entries detected in updated Master ZIP."
                )

        os.replace(temp_zip, MASTER_ZIP)

        zip_update_status = "UPDATED"
        zip_update_message = (
            "Master ZIP updated and validated without duplicate entries."
        )

    except Exception as exc:

        zip_update_status = "FAILED"
        zip_update_message = (
            f"{type(exc).__name__}: {exc}"
        )

        if temp_zip.exists():
            temp_zip.unlink(missing_ok=True)

else:
    zip_update_status = "NOT_FOUND"
    zip_update_message = "Master ZIP does not exist; certificate saved separately."


# =============================================================================
# 11. FINAL EXECUTION SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("CELL 11 — FINAL ENTERPRISE RELEASE READINESS AUDIT")
print("=" * 90)

print(f"Run ID:                    {RUN_ID}")
print(f"Execution Status:          COMPLETED")
print(f"Release Decision:          {release_status}")

print("\nAUDIT SUMMARY")
print("-" * 90)

print(f"Total Checks:               {len(checks)}")
print(f"Passed:                     {status_counts.get('PASSED', 0)}")
print(f"Failed:                     {status_counts.get('FAILED', 0)}")
print(f"Warnings:                   {status_counts.get('WARNING', 0)}")
print(f"Not Verified:               {status_counts.get('NOT_VERIFIED', 0)}")

print("\nISSUE SEVERITY")
print("-" * 90)

print(f"Critical:                   {len(critical_issues)}")
print(f"High:                       {len(high_issues)}")
print(f"Medium:                     {len(medium_issues)}")
print(f"Low:                        {len(low_issues)}")

print("\nMEDALLION INVENTORY")
print("-" * 90)

for layer, info in layer_inventory.items():
    print(
        f"{layer}: {info['total_files']} files "
        f"(CSV={info['csv_files']}, "
        f"JSON={info['json_files']}, "
        f"Parquet={info['parquet_files']})"
    )

print("\nMASTER BUNDLE")
print("-" * 90)
print(f"ZIP Update Status:          {zip_update_status}")
print(f"ZIP Details:                {zip_update_message}")

print("\nRELEASE RECOMMENDATION")
print("-" * 90)
print(release_recommendation)

print("\nGENERATED ARTIFACTS")
print("-" * 90)
print(f"Final Certificate:          {CERTIFICATE_FILE}")
print(f"Final Audit Report:         {AUDIT_REPORT_FILE}")

print("\n" + "=" * 90)
print("CELL 11 EXECUTION COMPLETED")
print("=" * 90)



CELL 11 — FINAL ENTERPRISE RELEASE READINESS AUDIT
Run ID:                    RELEASE-AUDIT-20261102T155019
Execution Status:          COMPLETED
Release Decision:          NOT_READY

AUDIT SUMMARY
------------------------------------------------------------------------------------------
Total Checks:               19
Passed:                     14
Failed:                     0
Warnings:                   5
Not Verified:               0

ISSUE SEVERITY
------------------------------------------------------------------------------------------
Critical:                   0
High:                       2
Medium:                     3
Low:                        0

MEDALLION INVENTORY
------------------------------------------------------------------------------------------
BRONZE: 11 files (CSV=0, JSON=11, Parquet=0)
SILVER: 6 files (CSV=6, JSON=0, Parquet=0)
GOLD: 2 files (CSV=2, JSON=0, Parquet=0)

MASTER BUNDLE
----------------------------------------------------------------------------

In [22]:
import json
from pathlib import Path

report_path = Path(
    "/content/reports/cell_11_final_integration_audit.json"
)

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print("=" * 80)
print("CELL 11 — DETAILED ISSUE DIAGNOSTICS")
print("=" * 80)

print("\nRELEASE DECISION:")
print(report["release_decision"]["status"])

print("\nSUMMARY:")
print(json.dumps(
    report["summary"],
    ensure_ascii=False,
    indent=2
))

print("\nALL NON-PASSED CHECKS:")
for check in report["checks"]:
    if check["status"] != "PASSED":
        print("-" * 80)
        print("ID:", check["check_id"])
        print("Category:", check["category"])
        print("Name:", check["check_name"])
        print("Status:", check["status"])
        print("Severity:", check["severity"])
        print("Details:", check["details"])

print("\nHIGH SEVERITY ISSUES:")
print(json.dumps(
    report["issues"]["high"],
    ensure_ascii=False,
    indent=2
))

print("\nMEDIUM SEVERITY ISSUES:")
print(json.dumps(
    report["issues"]["medium"],
    ensure_ascii=False,
    indent=2
))

print("=" * 80)

CELL 11 — DETAILED ISSUE DIAGNOSTICS

RELEASE DECISION:
NOT_READY

SUMMARY:
{
  "total_checks": 19,
  "passed": 14,
  "failed": 0,
  "warnings": 5,
  "partial": 0,
  "not_verified": 0,
  "critical_issues": 0,
  "high_issues": 2,
  "medium_issues": 3,
  "low_issues": 0
}

ALL NON-PASSED CHECKS:
--------------------------------------------------------------------------------
ID: REL-001
Category: SOURCE_INTEGRITY
Name: CELL 10.5 Comprehensive Audit
Status: WARNING
Severity: HIGH
Details: CELL 10.5 audit verdict: PARTIAL
--------------------------------------------------------------------------------
ID: REL-002
Category: SOURCE_INTEGRITY
Name: CELL 10.2 Advanced Source Audit
Status: WARNING
Severity: MEDIUM
Details: CELL 10.2 audit verdict: PARTIAL
--------------------------------------------------------------------------------
ID: REL-003
Category: SOURCE_INTEGRITY
Name: WHO Source Provenance
Status: WARNING
Severity: MEDIUM
Details: WHO raw source file is unavailable.
-----------------

In [23]:
from pathlib import Path
import pandas as pd
import json

BASE = Path("/content")

print("=" * 90)
print("CELL 11 — EMPTY DATASETS & RAG DIAGNOSTICS")
print("=" * 90)

# --------------------------------------------------
# 1. Inspect all Silver and Gold CSV files
# --------------------------------------------------

for layer in ["silver", "gold"]:
    folder = BASE / "data" / layer

    print(f"\n{'=' * 70}")
    print(f"{layer.upper()} CSV INVENTORY")
    print("=" * 70)

    if not folder.exists():
        print("Folder not found:", folder)
        continue

    for path in sorted(folder.glob("*.csv")):
        try:
            df = pd.read_csv(path)

            print(f"\nFile: {path.name}")
            print("Rows:", len(df))
            print("Columns:", list(df.columns))
            print("Empty:", df.empty)

            if not df.empty:
                print("Sample:")
                print(df.head(2).to_string(index=False))

        except Exception as e:
            print(f"\nFile: {path.name}")
            print("READ ERROR:", str(e))


# --------------------------------------------------
# 2. Locate and inspect RAG-related files
# --------------------------------------------------

print("\n" + "=" * 70)
print("RAG CORPUS FILES")
print("=" * 70)

search_dirs = [
    BASE / "data" / "gold",
    BASE / "data" / "silver",
    BASE / "data" / "bronze",
]

for folder in search_dirs:
    if not folder.exists():
        continue

    for path in sorted(folder.rglob("*")):
        if not path.is_file():
            continue

        if "rag" not in path.name.lower():
            continue

        print(f"\nFile: {path}")
        print("Size:", path.stat().st_size, "bytes")

        try:
            if path.suffix.lower() == ".csv":
                df = pd.read_csv(path)
                print("Rows:", len(df))
                print("Columns:", list(df.columns))
                print(df.head(3).to_string(index=False))

            elif path.suffix.lower() == ".json":
                with open(path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                if isinstance(data, list):
                    print("Records:", len(data))
                    print("Sample:", data[:2])
                elif isinstance(data, dict):
                    print("Root keys:", list(data.keys()))
                    results = data.get("results")
                    if isinstance(results, list):
                        print("Results:", len(results))
                        print("Sample:", results[:2])
                    else:
                        print("Content preview:", str(data)[:1000])

        except Exception as e:
            print("INSPECTION ERROR:", str(e))


print("\n" + "=" * 90)
print("DIAGNOSTICS COMPLETED — NO DATA MODIFIED")
print("=" * 90)

CELL 11 — EMPTY DATASETS & RAG DIAGNOSTICS

SILVER CSV INVENTORY

File: silver_eda_alerts.csv
Rows: 0
Columns: ['alert_id', 'alert_title']
Empty: True

File: silver_fda_ndc.csv
Rows: 300
Columns: ['product_id', 'product_ndc', 'brand_name', 'generic_name', 'active_ingredients_all', 'dosage_form', 'labeler_name', 'lineage_hash']
Empty: False
Sample:
                                    product_id product_ndc           brand_name                     generic_name                                           active_ingredients_all dosage_form                                    labeler_name                                                     lineage_hash
76125-667_39b47b35-753a-47ca-aa93-3ec34e28b599   76125-667                KOATE    antihemophilic factor (human)                                                      UNSPECIFIED         KIT                         KEDRION BIOPHARMA, INC. 604512f0c159cf955b5e7dfd11657c34d5b0b730c664b0d91b5fba0c4bc3a2cc
76176-135_5e2e190d-83ae-79ea-e063-6394a90ada

In [24]:
from pathlib import Path
import json

BASE = Path("/content")

print("=" * 90)
print("BRONZE SOURCE & RAG INPUT DIAGNOSTICS")
print("=" * 90)

# --------------------------------------------------
# 1. Inspect Bronze files
# --------------------------------------------------

bronze = BASE / "data" / "bronze"

print("\nBRONZE FILE INVENTORY")
print("-" * 70)

if bronze.exists():
    for path in sorted(bronze.rglob("*")):
        if path.is_file():
            print(
                f"{path.relative_to(bronze)} | "
                f"{path.stat().st_size:,} bytes"
            )
else:
    print("Bronze directory not found.")


# --------------------------------------------------
# 2. Inspect source metadata JSON files
# --------------------------------------------------

print("\nSOURCE METADATA INSPECTION")
print("-" * 70)

metadata_candidates = []

for folder in [BASE / "data", BASE / "reports"]:
    if folder.exists():
        metadata_candidates.extend(
            p for p in folder.rglob("*.json")
            if any(
                word in p.name.lower()
                for word in ["metadata", "source", "ingestion", "manifest"]
            )
        )

for path in sorted(set(metadata_candidates)):
    print(f"\nFILE: {path}")

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, dict):
            print(json.dumps(data, ensure_ascii=False, indent=2)[:3000])
        elif isinstance(data, list):
            print("Records:", len(data))
            print(json.dumps(data[:2], ensure_ascii=False, indent=2)[:2000])

    except Exception as e:
        print("READ ERROR:", str(e))


# --------------------------------------------------
# 3. Locate RAG-related artifacts
# --------------------------------------------------

print("\nRAG-RELATED ARTIFACTS")
print("-" * 70)

for folder in [BASE / "data", BASE / "reports"]:
    if not folder.exists():
        continue

    for path in sorted(folder.rglob("*")):
        if path.is_file() and "rag" in path.name.lower():
            print(
                f"{path} | "
                f"{path.stat().st_size:,} bytes"
            )


print("\n" + "=" * 90)
print("READ-ONLY DIAGNOSTICS COMPLETED")
print("=" * 90)

BRONZE SOURCE & RAG INPUT DIAGNOSTICS

BRONZE FILE INVENTORY
----------------------------------------------------------------------
eda/eda_alerts_raw.json | 288 bytes
kaggle/kaggle_medicines_raw.json | 319 bytes
openfda/drug-ndc-0001-of-0001.json | 243,337,316 bytes
openfda/fda_bulk_download_raw.json | 343 bytes
openfda/fda_enforcement_raw.json | 249,752 bytes
openfda/fda_ndc_raw.json | 559,670 bytes
openfda/fda_shortages_raw.json | 368 bytes
rag_corpus/drugs_com_pricing_raw.json | 267 bytes
rag_corpus/fda_safety_corpus.json | 82,659 bytes
rxnav/rxnav_concepts_raw.json | 7,423 bytes
who/who_eml_raw.json | 336 bytes

SOURCE METADATA INSPECTION
----------------------------------------------------------------------

FILE: /content/reports/MASTER_BUNDLE_MANIFEST.json
{
  "bundle_name": "Pharmaceutical Data Engineering Pipeline",
  "generated_at_utc": "2026-11-02T15:49:16.240629+00:00",
  "base_directory": "/content",
  "included_directories": [
    "data/bronze/openfda",
    "data/bronze/

In [25]:
from pathlib import Path
import json

BASE = Path("/content/data/bronze")

FILES_TO_INSPECT = [
    "eda/eda_alerts_raw.json",
    "kaggle/kaggle_medicines_raw.json",
    "openfda/fda_enforcement_raw.json",
    "openfda/fda_shortages_raw.json",
    "openfda/fda_bulk_download_raw.json",
    "rag_corpus/drugs_com_pricing_raw.json",
    "rag_corpus/fda_safety_corpus.json",
    "rxnav/rxnav_concepts_raw.json",
    "who/who_eml_raw.json",
]

print("=" * 90)
print("BRONZE RAW RESPONSE CONTENT INSPECTION")
print("=" * 90)

for relative_path in FILES_TO_INSPECT:
    path = BASE / relative_path

    print("\n" + "-" * 80)
    print("FILE:", relative_path)

    if not path.exists():
        print("STATUS: FILE NOT FOUND")
        continue

    print("SIZE:", path.stat().st_size, "bytes")

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print("JSON VALID: YES")
        print("ROOT TYPE:", type(data).__name__)

        if isinstance(data, dict):
            print("ROOT KEYS:", list(data.keys()))

            for key in [
                "status",
                "message",
                "error",
                "results",
                "meta",
                "source",
                "record_count",
                "total_records",
                "verification_status",
            ]:
                if key in data:
                    value = data[key]

                    if isinstance(value, (dict, list)):
                        print(f"{key}: {json.dumps(value, ensure_ascii=False)[:1500]}")
                    else:
                        print(f"{key}: {value}")

            if not any(
                key in data
                for key in ["status", "message", "error", "results", "meta"]
            ):
                print("CONTENT PREVIEW:")
                print(json.dumps(data, ensure_ascii=False, indent=2)[:2000])

        elif isinstance(data, list):
            print("RECORD COUNT:", len(data))
            print("SAMPLE:")
            print(json.dumps(data[:2], ensure_ascii=False, indent=2)[:2000])

        else:
            print("VALUE:", str(data)[:1000])

    except Exception as e:
        print("JSON VALID: NO")
        print("ERROR:", str(e))


print("\n" + "=" * 90)
print("INSPECTION COMPLETE — NO FILES MODIFIED")
print("=" * 90)

BRONZE RAW RESPONSE CONTENT INSPECTION

--------------------------------------------------------------------------------
FILE: eda/eda_alerts_raw.json
SIZE: 288 bytes
JSON VALID: YES
ROOT TYPE: dict
ROOT KEYS: ['metadata', 'results']
results: []

--------------------------------------------------------------------------------
FILE: kaggle/kaggle_medicines_raw.json
SIZE: 319 bytes
JSON VALID: YES
ROOT TYPE: dict
ROOT KEYS: ['metadata', 'results']
results: []

--------------------------------------------------------------------------------
FILE: openfda/fda_enforcement_raw.json
SIZE: 249752 bytes
JSON VALID: YES
ROOT TYPE: dict
ROOT KEYS: ['metadata', 'results']
results: [{"status": "Ongoing", "city": "San Juan Del Rio", "state": "N/A", "country": "Mexico", "classification": "Class II", "openfda": {}, "product_type": "Drugs", "event_id": "86162", "recalling_firm": "Albek De Mexico S.A. De C.V.", "address_1": "Oriente Cuatro No. 11 Nuevo", "address_2": "N/A", "postal_code": "N/A", "volunt

In [26]:
from pathlib import Path
import json

BASE = Path("/content/data/bronze")

FILES = [
    "eda/eda_alerts_raw.json",
    "kaggle/kaggle_medicines_raw.json",
    "openfda/fda_enforcement_raw.json",
    "openfda/fda_shortages_raw.json",
    "rag_corpus/drugs_com_pricing_raw.json",
    "rag_corpus/fda_safety_corpus.json",
    "who/who_eml_raw.json",
    "rxnav/rxnav_concepts_raw.json",
    "openfda/fda_ndc_raw.json",
]

print("=" * 90)
print("INGESTION METADATA AUDIT")
print("=" * 90)

for relative_path in FILES:
    path = BASE / relative_path

    print("\n" + "-" * 80)
    print("SOURCE:", relative_path)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        metadata = data.get("metadata", {})
        results = data.get("results", [])

        print("RECORD COUNT:", len(results) if isinstance(results, list) else "NOT A LIST")
        print("METADATA:")

        print(json.dumps(
            metadata,
            ensure_ascii=False,
            indent=2
        ))

    except Exception as e:
        print("ERROR:", str(e))

print("\n" + "=" * 90)
print("AUDIT COMPLETED — READ ONLY")
print("=" * 90)


INGESTION METADATA AUDIT

--------------------------------------------------------------------------------
SOURCE: eda/eda_alerts_raw.json
RECORD COUNT: 0
METADATA:
{
  "ingestion_timestamp_utc": "2026-11-02T15:49:11.348393+00:00",
  "source": "Egyptian Drug Authority (EDA)",
  "total_records": 0,
  "local_pdfs_processed": 0,
  "has_synthetic_data": false,
  "execution_status": "NO_LOCAL_DATA_FOUND"
}

--------------------------------------------------------------------------------
SOURCE: kaggle/kaggle_medicines_raw.json
RECORD COUNT: 0
METADATA:
{
  "ingestion_timestamp_utc": "2026-11-02T15:49:15.740839+00:00",
  "sources": [
    "Kaggle 1mg Medicine Data",
    "Kaggle 11,000+ Medicine Details"
  ],
  "discovered_files_count": 0,
  "total_records_ingested": 0,
  "execution_status": "NO_KAGGLE_FILES_FOUND"
}

--------------------------------------------------------------------------------
SOURCE: openfda/fda_enforcement_raw.json
RECORD COUNT: 150
METADATA:
{
  "ingestion_timestamp_utc

In [27]:

import os
import json
import hashlib
import zipfile
import tempfile
import shutil
import pandas as pd
from datetime import datetime, timezone

print("Executing CELL 12: Data Governance, Drift Detection & API Payloads...")

# ============================================================
# 1. PATHS
# ============================================================

SILVER_DIR = DATA_DIRS["silver"]
GOLD_DIR = os.path.join(BASE_DIR, "data", "gold")
REPORTS_DIR = DATA_DIRS["reports"]
API_DIR = os.path.join(BASE_DIR, "data", "serving_api_payloads")

os.makedirs(SILVER_DIR, exist_ok=True)
os.makedirs(GOLD_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(API_DIR, exist_ok=True)

silver_fda_path = os.path.join(SILVER_DIR, "silver_fda_ndc.csv")
gold_analytics_path = os.path.join(GOLD_DIR, "gold_pharma_analytics.csv")
gold_rag_path = os.path.join(GOLD_DIR, "gold_rag_knowledge_corpus.csv")

# ============================================================
# 2. SAFE CSV READER
# ============================================================

def read_csv_safely(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        print(f"WARNING: CSV missing or empty: {path}")
        return pd.DataFrame()

    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        print(f"WARNING: CSV has no columns: {path}")
        return pd.DataFrame()
    except Exception as exc:
        print(f"ERROR reading {path}: {exc}")
        return pd.DataFrame()

# ============================================================
# 3. GOVERNANCE / HASH MASKING
# ============================================================

def hash_sensitive_columns(df, sensitive_cols):
    if df.empty:
        return df.copy()

    result = df.copy()

    for col in sensitive_cols:
        if col in result.columns:
            result[col] = result[col].apply(
                lambda value: (
                    hashlib.sha256(str(value).encode("utf-8")).hexdigest()[:16]
                    if pd.notna(value) and str(value).strip()
                    else None
                )
            )

    return result


df_silver_fda = read_csv_safely(silver_fda_path)
df_gold_analytics = read_csv_safely(gold_analytics_path)
df_gold_rag = read_csv_safely(gold_rag_path)

df_silver_fda_gov = hash_sensitive_columns(
    df_silver_fda,
    sensitive_cols=["labeler_name"]
)

gov_silver_path = os.path.join(
    SILVER_DIR,
    "silver_fda_ndc_governed.csv"
)

df_silver_fda_gov.to_csv(gov_silver_path, index=False)

# ============================================================
# 4. DRIFT DETECTION WITH PERSISTENT BASELINE
# ============================================================

baseline_path = os.path.join(
    REPORTS_DIR,
    "silver_fda_ndc_drift_baseline.json"
)

drift_report_path = os.path.join(
    REPORTS_DIR,
    "data_drift_audit_report.json"
)

def calculate_dataset_profile(df):
    return {
        "columns": list(df.columns),
        "row_count": int(len(df)),
        "null_percentages": {
            col: round(float(df[col].isna().mean() * 100), 2)
            for col in df.columns
        }
    }

current_profile = calculate_dataset_profile(df_silver_fda)

drift_results = {
    "dataset": "openfda_ndc_catalog",
    "audit_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "baseline_file": baseline_path,
    "schema_drift_detected": False,
    "null_drift_detected": False,
    "new_columns_added": [],
    "missing_columns": [],
    "null_percentage_diff": {},
    "status": None
}

if df_silver_fda.empty:
    drift_results["status"] = "INSUFFICIENT_CURRENT_DATA"

elif not os.path.exists(baseline_path):
    with open(baseline_path, "w", encoding="utf-8") as f:
        json.dump(current_profile, f, ensure_ascii=False, indent=2)

    drift_results["status"] = "BASELINE_CREATED_FIRST_RUN"

else:
    try:
        with open(baseline_path, "r", encoding="utf-8") as f:
            baseline_profile = json.load(f)

        baseline_cols = set(baseline_profile["columns"])
        current_cols = set(current_profile["columns"])

        new_cols = sorted(current_cols - baseline_cols)
        missing_cols = sorted(baseline_cols - current_cols)

        drift_results["new_columns_added"] = new_cols
        drift_results["missing_columns"] = missing_cols
        drift_results["schema_drift_detected"] = bool(
            new_cols or missing_cols
        )

        common_cols = baseline_cols.intersection(current_cols)

        for col in sorted(common_cols):
            base_pct = baseline_profile["null_percentages"].get(col, 0)
            curr_pct = current_profile["null_percentages"].get(col, 0)
            diff = round(curr_pct - base_pct, 2)

            if abs(diff) > 15:
                drift_results["null_drift_detected"] = True
                drift_results["null_percentage_diff"][col] = {
                    "baseline_null_pct": base_pct,
                    "current_null_pct": curr_pct,
                    "difference": diff
                }

        if (
            drift_results["schema_drift_detected"]
            or drift_results["null_drift_detected"]
        ):
            drift_results["status"] = "DRIFT_DETECTED"
        else:
            drift_results["status"] = "NO_DRIFT_DETECTED"

    except Exception as exc:
        drift_results["status"] = "BASELINE_READ_ERROR"
        drift_results["error"] = str(exc)

with open(drift_report_path, "w", encoding="utf-8") as f:
    json.dump(drift_results, f, ensure_ascii=False, indent=2)

# ============================================================
# 5. API ANALYTICS PAYLOAD
# ============================================================

def dataframe_to_records(df):
    if df.empty:
        return []

    clean_df = df.astype(object).where(pd.notna(df), None)
    return clean_df.to_dict(orient="records")

analytics_payload = {
    "endpoint": "/api/v1/pharmaceuticals/analytics",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "READY" if not df_gold_analytics.empty else "NO_DATA",
    "total_records": int(len(df_gold_analytics)),
    "data": dataframe_to_records(df_gold_analytics)
}

api_analytics_path = os.path.join(
    API_DIR,
    "api_gold_analytics_endpoint.json"
)

with open(api_analytics_path, "w", encoding="utf-8") as f:
    json.dump(
        analytics_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )

# ============================================================
# 6. VECTOR DB PAYLOAD
# ============================================================

if df_gold_rag.empty:
    rag_status = "BLOCKED_NO_VALID_DOCUMENTS"
else:
    rag_status = "PAYLOAD_GENERATED"

vector_payload = {
    "namespace": "pharma-safety-knowledge-base",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": rag_status,
    "documents_count": int(len(df_gold_rag)),
    "vector_documents": dataframe_to_records(df_gold_rag)
}

vector_db_path = os.path.join(
    API_DIR,
    "api_vector_db_import_payload.json"
)

with open(vector_db_path, "w", encoding="utf-8") as f:
    json.dump(
        vector_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )

# ============================================================
# 7. SAVE CELL 12 EXECUTION REPORT
# ============================================================

cell12_report = {
    "cell": "12",
    "execution_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "governance": {
        "source_rows": int(len(df_silver_fda)),
        "governed_rows": int(len(df_silver_fda_gov)),
        "masked_columns": (
            ["labeler_name"]
            if "labeler_name" in df_silver_fda.columns
            else []
        ),
        "output_file": gov_silver_path
    },
    "drift_detection": drift_results,
    "analytics_api": {
        "status": analytics_payload["status"],
        "records": analytics_payload["total_records"],
        "output_file": api_analytics_path
    },
    "vector_db": {
        "status": rag_status,
        "documents": int(len(df_gold_rag)),
        "output_file": vector_db_path
    }
}

cell12_report_path = os.path.join(
    REPORTS_DIR,
    "cell_12_execution_report.json"
)

with open(cell12_report_path, "w", encoding="utf-8") as f:
    json.dump(cell12_report, f, ensure_ascii=False, indent=2, default=str)

# ============================================================
# 8. SAFE MASTER ZIP UPDATE (NO DUPLICATE ENTRIES)
# ============================================================

master_zip = os.path.join(
    BASE_DIR,
    "enterprise_master_production_bundle.zip"
)

files_to_add = [
    drift_report_path,
    baseline_path,
    gov_silver_path,
    api_analytics_path,
    vector_db_path,
    cell12_report_path
]

zip_status = "SKIPPED"
zip_details = ""

if os.path.exists(master_zip):
    temp_zip = master_zip + ".tmp"

    try:
        replacement_entries = {
            os.path.relpath(path, BASE_DIR): path
            for path in files_to_add
            if os.path.exists(path)
        }

        with zipfile.ZipFile(master_zip, "r") as source_zip:
            with zipfile.ZipFile(
                temp_zip, "w", compression=zipfile.ZIP_DEFLATED
            ) as target_zip:

                for item in source_zip.infolist():
                    if item.filename not in replacement_entries:
                        target_zip.writestr(
                            item,
                            source_zip.read(item.filename)
                        )

                for arcname, filepath in replacement_entries.items():
                    target_zip.write(filepath, arcname=arcname)

        with zipfile.ZipFile(temp_zip, "r") as check_zip:
            names = check_zip.namelist()

            if len(names) != len(set(names)):
                raise RuntimeError("Duplicate ZIP entries detected")

            bad_file = check_zip.testzip()
            if bad_file is not None:
                raise RuntimeError(f"ZIP integrity error: {bad_file}")

        os.replace(temp_zip, master_zip)
        zip_status = "UPDATED"
        zip_details = "ZIP updated and integrity validated."

    except Exception as exc:
        if os.path.exists(temp_zip):
            os.remove(temp_zip)

        zip_status = "ERROR"
        zip_details = str(exc)

else:
    zip_details = "Master ZIP file not found; no ZIP update performed."

# ============================================================
# 9. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 75)
print("CELL 12 EXECUTION SUMMARY")
print("=" * 75)

print(f"FDA Silver rows:             {len(df_silver_fda)}")
print(f"Governed Silver rows:        {len(df_silver_fda_gov)}")
print(f"Drift status:                {drift_results['status']}")
print(f"Analytics API status:        {analytics_payload['status']}")
print(f"Analytics records:           {analytics_payload['total_records']}")
print(f"RAG status:                  {rag_status}")
print(f"RAG documents:               {len(df_gold_rag)}")
print(f"Master ZIP status:           {zip_status}")
print(f"Master ZIP details:          {zip_details}")
print(f"Execution report:            {cell12_report_path}")
print("=" * 75)
print("CELL 12 EXECUTION COMPLETED")


Executing CELL 12: Data Governance, Drift Detection & API Payloads...

CELL 12 EXECUTION SUMMARY
FDA Silver rows:             300
Governed Silver rows:        300
Drift status:                BASELINE_CREATED_FIRST_RUN
Analytics API status:        READY
Analytics records:           227
RAG status:                  PAYLOAD_GENERATED
RAG documents:               150
Master ZIP status:           UPDATED
Master ZIP details:          ZIP updated and integrity validated.
Execution report:            /content/reports/cell_12_execution_report.json
CELL 12 EXECUTION COMPLETED


In [28]:

import os
import json
import sqlite3
import zipfile
import pandas as pd
from datetime import datetime, timezone

logger.info("Executing Cell 13: SQL Data Warehouse Indexing...")

# ============================================================
# 1. SAFE CSV READER
# ============================================================

def read_csv_safely_cell13(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()
    except Exception as exc:
        logger.warning(f"CSV read failed: {path} | {exc}")
        return pd.DataFrame()

# ============================================================
# 2. SQL DATA WAREHOUSE INDEXER
# ============================================================

def index_data_lake_to_sqlite_dw(db_path, base_dir):
    os.makedirs(os.path.dirname(db_path), exist_ok=True)

    conn = sqlite3.connect(db_path)
    indexed_tables = []
    skipped_files = []
    errors = []

    try:
        for layer in ["silver", "gold"]:
            layer_dir = os.path.join(base_dir, "data", layer)

            if not os.path.isdir(layer_dir):
                continue

            for filename in sorted(os.listdir(layer_dir)):
                if not filename.endswith(".csv"):
                    continue

                file_path = os.path.join(layer_dir, filename)
                table_name = os.path.splitext(filename)[0]

                try:
                    df = read_csv_safely_cell13(file_path)

                    if df.empty:
                        skipped_files.append({
                            "file": filename,
                            "reason": "EMPTY_DATASET"
                        })
                        continue

                    df.to_sql(
                        table_name,
                        conn,
                        if_exists="replace",
                        index=False
                    )

                    indexed_tables.append({
                        "table": table_name,
                        "rows": int(len(df)),
                        "layer": layer
                    })

                except Exception as exc:
                    errors.append({
                        "file": filename,
                        "error": str(exc)
                    })
                    logger.warning(
                        f"Failed to index {table_name}: {exc}"
                    )

        cursor = conn.cursor()

        cursor.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name
        """)

        tables_in_db = [row[0] for row in cursor.fetchall()]

        conn.commit()

    finally:
        conn.close()

    return {
        "dw_path": db_path,
        "indexed_tables_count": len(indexed_tables),
        "indexed_tables": indexed_tables,
        "tables": tables_in_db,
        "skipped_files": skipped_files,
        "errors": errors
    }

# ============================================================
# 3. EXECUTE WAREHOUSE INDEXING
# ============================================================

sqlite_dw_file = os.path.join(
    BASE_DIR,
    "data",
    "pharma_enterprise_warehouse.db"
)

dw_summary = index_data_lake_to_sqlite_dw(
    sqlite_dw_file,
    BASE_DIR
)

# ============================================================
# 4. SAVE EXECUTION REPORT
# ============================================================

report_path = os.path.join(
    DATA_DIRS["reports"],
    "cell_13_warehouse_report.json"
)

os.makedirs(os.path.dirname(report_path), exist_ok=True)

report = {
    "cell": "13",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "warehouse_path": sqlite_dw_file,
    "indexed_tables_count": dw_summary["indexed_tables_count"],
    "indexed_tables": dw_summary["indexed_tables"],
    "skipped_files": dw_summary["skipped_files"],
    "errors": dw_summary["errors"],
    "database_tables": dw_summary["tables"]
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

# ============================================================
# 5. SAFE MASTER ZIP UPDATE
# ============================================================

master_zip = os.path.join(
    BASE_DIR,
    "enterprise_master_production_bundle.zip"
)

zip_status = "SKIPPED"

if os.path.exists(master_zip):
    temp_zip = master_zip + ".tmp"

    try:
        arcname = os.path.relpath(sqlite_dw_file, BASE_DIR)

        with zipfile.ZipFile(master_zip, "r") as source:
            with zipfile.ZipFile(
                temp_zip,
                "w",
                compression=zipfile.ZIP_DEFLATED
            ) as target:

                for item in source.infolist():
                    if item.filename != arcname:
                        target.writestr(
                            item,
                            source.read(item.filename)
                        )

                target.write(sqlite_dw_file, arcname=arcname)

        with zipfile.ZipFile(temp_zip, "r") as check:
            names = check.namelist()

            if len(names) != len(set(names)):
                raise RuntimeError("Duplicate ZIP entries detected")

            if check.testzip() is not None:
                raise RuntimeError("ZIP integrity check failed")

        os.replace(temp_zip, master_zip)
        zip_status = "UPDATED"

    except Exception as exc:
        if os.path.exists(temp_zip):
            os.remove(temp_zip)

        zip_status = f"ERROR: {exc}"

else:
    zip_status = "MASTER_ZIP_NOT_FOUND"

# ============================================================
# 6. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 75)
print("CELL 13 — SQL DATA WAREHOUSE SUMMARY")
print("=" * 75)

print(f"Database:              {sqlite_dw_file}")
print(f"Indexed Tables:        {dw_summary['indexed_tables_count']}")
print(f"Database Tables:       {dw_summary['tables']}")
print(f"Skipped Files:         {len(dw_summary['skipped_files'])}")
print(f"Indexing Errors:       {len(dw_summary['errors'])}")
print(f"ZIP Status:            {zip_status}")
print(f"Report:                {report_path}")

print("\nINDEXED TABLE DETAILS:")
for item in dw_summary["indexed_tables"]:
    print(
        f"- {item['table']} | "
        f"Rows: {item['rows']} | "
        f"Layer: {item['layer']}"
    )

print("=" * 75)
print("CELL 13 EXECUTION COMPLETED")
print("=" * 75)



CELL 13 — SQL DATA WAREHOUSE SUMMARY
Database:              /content/data/pharma_enterprise_warehouse.db
Indexed Tables:        4
Database Tables:       ['gold_pharma_analytics', 'gold_rag_knowledge_corpus', 'silver_fda_ndc', 'silver_fda_ndc_governed']
Skipped Files:         5
Indexing Errors:       0
ZIP Status:            UPDATED
Report:                /content/reports/cell_13_warehouse_report.json

INDEXED TABLE DETAILS:
- silver_fda_ndc | Rows: 300 | Layer: silver
- silver_fda_ndc_governed | Rows: 300 | Layer: silver
- gold_pharma_analytics | Rows: 227 | Layer: gold
- gold_rag_knowledge_corpus | Rows: 150 | Layer: gold
CELL 13 EXECUTION COMPLETED


In [29]:
import os
import json
import zipfile
import tempfile
import pandas as pd
from datetime import datetime, timezone

logger.info(
    "Executing Cell 14: CI/CD Readiness & Airflow DAG Manifest..."
)

# ============================================================
# 1. PATHS
# ============================================================

reports_dir = DATA_DIRS["reports"]
gold_dir = os.path.join(BASE_DIR, "data", "gold")
silver_dir = os.path.join(BASE_DIR, "data", "silver")

os.makedirs(reports_dir, exist_ok=True)

airflow_dag_file = os.path.join(
    reports_dir,
    "airflow_pharma_pipeline_dag.py"
)

master_zip_file = os.path.join(
    BASE_DIR,
    "enterprise_master_production_bundle.zip"
)

# ============================================================
# 2. GENERATE AIRFLOW DAG TEMPLATE
# ============================================================

def generate_apache_airflow_dag_manifest(output_path):

    airflow_dag_code = '''"""
Apache Airflow DAG Template
Enterprise Pharmaceutical Medallion Pipeline

IMPORTANT:
Replace placeholder Python callables with actual pipeline
entry points before deploying this DAG.
"""

from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.python import PythonOperator


default_args = {
    "owner": "data_engineering_team",
    "depends_on_past": False,
    "start_date": datetime(2026, 1, 1),
    "email_on_failure": False,
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
}


def bronze_ingestion():
    raise NotImplementedError(
        "Connect this task to the actual Bronze ingestion pipeline."
    )


def silver_transformation():
    raise NotImplementedError(
        "Connect this task to the actual Silver transformation pipeline."
    )


def gold_aggregation():
    raise NotImplementedError(
        "Connect this task to the actual Gold aggregation pipeline."
    )


def quality_and_observability():
    raise NotImplementedError(
        "Connect this task to the actual quality and audit pipeline."
    )


with DAG(
    dag_id="enterprise_pharma_medallion_pipeline",
    default_args=default_args,
    description="Pharmaceutical Medallion Pipeline Orchestration",
    schedule="@daily",
    catchup=False,
    tags=["pharma", "data-engineering", "medallion"],
) as dag:

    bronze = PythonOperator(
        task_id="bronze_ingestion",
        python_callable=bronze_ingestion,
    )

    silver = PythonOperator(
        task_id="silver_transformation",
        python_callable=silver_transformation,
    )

    gold = PythonOperator(
        task_id="gold_aggregation",
        python_callable=gold_aggregation,
    )

    quality = PythonOperator(
        task_id="quality_and_observability",
        python_callable=quality_and_observability,
    )

    bronze >> silver >> gold >> quality
'''

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(airflow_dag_code)

    logger.info(f"Airflow DAG template generated: {output_path}")


generate_apache_airflow_dag_manifest(airflow_dag_file)

# ============================================================
# 3. ARTIFACT VALIDATION
# ============================================================

def inspect_csv(path):

    result = {
        "exists": os.path.exists(path),
        "readable": False,
        "rows": 0,
        "columns": 0,
        "error": None
    }

    if not result["exists"]:
        return result

    try:
        if os.path.getsize(path) == 0:
            return result

        df = pd.read_csv(path)

        result["readable"] = True
        result["rows"] = int(len(df))
        result["columns"] = int(len(df.columns))

    except Exception as exc:
        result["error"] = str(exc)

    return result


warehouse_path = os.path.join(
    BASE_DIR,
    "data",
    "pharma_enterprise_warehouse.db"
)

analytics_path = os.path.join(
    gold_dir,
    "gold_pharma_analytics.csv"
)

rag_path = os.path.join(
    gold_dir,
    "gold_rag_knowledge_corpus.csv"
)

master_audit_path = os.path.join(
    reports_dir,
    "master_pipeline_audit_report.json"
)

dashboard_path = os.path.join(
    reports_dir,
    "enterprise_audit_dashboard.html"
)

analytics_check = inspect_csv(analytics_path)
rag_check = inspect_csv(rag_path)

required_checks = {
    "warehouse_database": (
        os.path.exists(warehouse_path)
        and os.path.getsize(warehouse_path) > 0
    ),
    "gold_analytics_readable": (
        analytics_check["readable"]
        and analytics_check["rows"] > 0
    ),
    "gold_rag_has_documents": (
        rag_check["readable"]
        and rag_check["rows"] > 0
    ),
    "master_audit_report": (
        os.path.exists(master_audit_path)
        and os.path.getsize(master_audit_path) > 0
    ),
    "audit_dashboard": (
        os.path.exists(dashboard_path)
        and os.path.getsize(dashboard_path) > 0
    ),
    "airflow_dag_manifest": (
        os.path.exists(airflow_dag_file)
        and os.path.getsize(airflow_dag_file) > 0
    )
}

passed_checks = [
    name for name, passed in required_checks.items()
    if passed
]

failed_checks = [
    name for name, passed in required_checks.items()
    if not passed
]

# ============================================================
# 4. BUILD MASTER ZIP SAFELY
# ============================================================

folders_to_include = [
    silver_dir,
    os.path.join(BASE_DIR, "data", "silver_parquet"),
    gold_dir,
    os.path.join(BASE_DIR, "data", "gold_parquet"),
    reports_dir,
    os.path.join(BASE_DIR, "data", "serving_api_payloads"),
]

temp_zip = master_zip_file + ".tmp"

zip_status = "FAILED"
zip_entries_count = 0

try:

    with zipfile.ZipFile(
        temp_zip,
        "w",
        compression=zipfile.ZIP_DEFLATED
    ) as zip_f:

        for folder in folders_to_include:

            if not os.path.isdir(folder):
                continue

            for root, _, files in os.walk(folder):

                for filename in files:

                    full_path = os.path.join(root, filename)

                    if full_path == master_zip_file:
                        continue

                    arcname = os.path.relpath(
                        full_path,
                        BASE_DIR
                    )

                    zip_f.write(full_path, arcname=arcname)

        if os.path.exists(warehouse_path):
            zip_f.write(
                warehouse_path,
                arcname=os.path.relpath(
                    warehouse_path,
                    BASE_DIR
                )
            )

    with zipfile.ZipFile(temp_zip, "r") as check:

        names = check.namelist()

        if len(names) != len(set(names)):
            raise RuntimeError("Duplicate ZIP entries detected")

        bad_file = check.testzip()

        if bad_file is not None:
            raise RuntimeError(
                f"ZIP integrity failure: {bad_file}"
            )

        zip_entries_count = len(names)

    os.replace(temp_zip, master_zip_file)
    zip_status = "UPDATED_AND_VALIDATED"

except Exception as exc:

    if os.path.exists(temp_zip):
        os.remove(temp_zip)

    logger.error(f"Master ZIP update failed: {exc}")
    zip_status = f"FAILED: {exc}"

# ============================================================
# 5. FINAL CI/CD READINESS
# ============================================================

required_checks["master_zip_valid"] = (
    zip_status == "UPDATED_AND_VALIDATED"
)

passed_checks = [
    name for name, passed in required_checks.items()
    if passed
]

failed_checks = [
    name for name, passed in required_checks.items()
    if not passed
]

cicd_passed = len(failed_checks) == 0

cicd_summary = {
    "cell": "14",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "cicd_passed": cicd_passed,
    "deployment_status": (
        "READY_FOR_PRODUCTION_DEPLOYMENT"
        if cicd_passed
        else "BLOCKED"
    ),
    "checks_passed": len(passed_checks),
    "checks_failed": len(failed_checks),
    "passed_checks": passed_checks,
    "failed_checks": failed_checks,
    "analytics": analytics_check,
    "rag_corpus": rag_check,
    "airflow_status": "TEMPLATE_GENERATED_NOT_CONNECTED",
    "zip_status": zip_status,
    "zip_entries_count": zip_entries_count
}

cicd_report_file = os.path.join(
    reports_dir,
    "cicd_deployment_readiness_report.json"
)

with open(cicd_report_file, "w", encoding="utf-8") as f:
    json.dump(
        cicd_summary,
        f,
        ensure_ascii=False,
        indent=2
    )

# ============================================================
# 6. FINAL OUTPUT
# ============================================================

final_bundle_size = (
    os.path.getsize(master_zip_file) / (1024 * 1024)
    if os.path.exists(master_zip_file)
    else 0
)

print("\n" + "=" * 75)
print("CELL 14 — CI/CD & AIRFLOW READINESS SUMMARY")
print("=" * 75)

print(f"Airflow DAG:          {airflow_dag_file}")
print(f"Airflow Status:       TEMPLATE_GENERATED_NOT_CONNECTED")
print(f"Analytics Rows:       {analytics_check['rows']}")
print(f"RAG Documents:        {rag_check['rows']}")
print(f"Checks Passed:        {len(passed_checks)}")
print(f"Checks Failed:        {len(failed_checks)}")
print(f"Deployment Status:    {cicd_summary['deployment_status']}")
print(f"ZIP Status:           {zip_status}")
print(f"ZIP Entries:          {zip_entries_count}")
print(f"Bundle Size:          {final_bundle_size:.2f} MB")
print(f"CI/CD Report:         {cicd_report_file}")

if failed_checks:
    print("\nFAILED CHECKS:")
    for item in failed_checks:
        print(f"- {item}")

print("=" * 75)
print("CELL 14 EXECUTION COMPLETED")
print("=" * 75)


CELL 14 — CI/CD & AIRFLOW READINESS SUMMARY
Airflow DAG:          /content/reports/airflow_pharma_pipeline_dag.py
Airflow Status:       TEMPLATE_GENERATED_NOT_CONNECTED
Analytics Rows:       227
RAG Documents:        150
Checks Passed:        7
Checks Failed:        0
Deployment Status:    READY_FOR_PRODUCTION_DEPLOYMENT
ZIP Status:           UPDATED_AND_VALIDATED
ZIP Entries:          37
Bundle Size:          0.31 MB
CI/CD Report:         /content/reports/cicd_deployment_readiness_report.json
CELL 14 EXECUTION COMPLETED


In [30]:
# =============================================================================
# CELL 15 — DATA SOURCE REGISTER & INGESTION INVENTORY
# Pharma Supply Chain Intelligence
# =============================================================================

import os
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

print("=" * 90)
print("CELL 15 — DATA SOURCE REGISTER & INGESTION INVENTORY")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. PATH CONFIGURATION
# -----------------------------------------------------------------------------

BASE_PATH = Path(BASE_DIR)
BRONZE_PATH = BASE_PATH / "data" / "bronze"
REPORTS_PATH = BASE_PATH / "reports"

REPORTS_PATH.mkdir(parents=True, exist_ok=True)

REGISTER_CSV = REPORTS_PATH / "cell_15_data_source_register.csv"
REGISTER_JSON = REPORTS_PATH / "cell_15_data_source_register.json"
AUDIT_JSON = REPORTS_PATH / "cell_15_ingestion_inventory_audit.json"

AUDIT_TIMESTAMP = datetime.now(timezone.utc).isoformat()

# -----------------------------------------------------------------------------
# 2. REGISTERED DATA SOURCES
# -----------------------------------------------------------------------------

SOURCE_DEFINITIONS = [
    {
        "source_id": "SRC-001",
        "source_name": "openFDA Drug NDC",
        "publisher": "U.S. Food and Drug Administration",
        "url": "https://api.fda.gov/drug/ndc.json",
        "access_method": "REST API / Official Bulk Download",
        "data_domain": "Drug Product Master Data",
        "expected_file": "openfda/fda_ndc_raw.json",
        "expected_fields": [
            "product_ndc", "brand_name", "generic_name",
            "active_ingredients", "dosage_form", "labeler_name"
        ],
        "update_frequency": "API-dependent",
        "limitations": "NDC listing does not establish current market availability or actual inventory."
    },
    {
        "source_id": "SRC-002",
        "source_name": "openFDA Drug Enforcement",
        "publisher": "U.S. Food and Drug Administration",
        "url": "https://api.fda.gov/drug/enforcement.json",
        "access_method": "REST API",
        "data_domain": "Drug Recalls and Safety",
        "expected_file": "openfda/fda_enforcement_raw.json",
        "expected_fields": [
            "recall_number", "reason_for_recall", "status",
            "recalling_firm", "recall_initiation_date"
        ],
        "update_frequency": "API-dependent",
        "limitations": "Availability depends on API response and query parameters."
    },
    {
        "source_id": "SRC-003",
        "source_name": "openFDA Drug Shortages",
        "publisher": "U.S. Food and Drug Administration",
        "url": "https://api.fda.gov/drug/shortages.json",
        "access_method": "REST API",
        "data_domain": "Drug Shortages",
        "expected_file": "openfda/fda_shortages_raw.json",
        "expected_fields": [
            "drug_name", "status", "reason", "availability"
        ],
        "update_frequency": "API-dependent",
        "limitations": "Endpoint availability and returned schema must be independently verified."
    },
    {
        "source_id": "SRC-004",
        "source_name": "FDA NDC Bulk Download",
        "publisher": "U.S. Food and Drug Administration",
        "url": "https://www.fda.gov/drugs/drug-approvals-and-databases/national-drug-code-directory",
        "access_method": "Official Bulk File Download",
        "data_domain": "Drug Product Master Data",
        "expected_file": "openfda/drug-ndc-0001-of-0001.json",
        "expected_fields": [
            "product_ndc", "brand_name", "generic_name",
            "active_ingredients", "dosage_form"
        ],
        "update_frequency": "Publisher-dependent",
        "limitations": "Bulk snapshot; not proof of current commercial availability."
    },
    {
        "source_id": "SRC-005",
        "source_name": "NIH RxNav",
        "publisher": "National Library of Medicine",
        "url": "https://rxnav.nlm.nih.gov/REST/",
        "access_method": "REST API",
        "data_domain": "Drug Reference and Identifier Matching",
        "expected_file": "rxnav/rxnav_concepts_raw.json",
        "expected_fields": [
            "rxcui", "name", "tty"
        ],
        "update_frequency": "API-dependent",
        "limitations": "Concept matching is not guaranteed for every submitted drug name."
    },
    {
        "source_id": "SRC-006",
        "source_name": "Egyptian Drug Authority Safety Alerts",
        "publisher": "Egyptian Drug Authority",
        "url": "https://www.edaegypt.gov.eg/en/media-center/safety-alerts/",
        "access_method": "Website Extraction / Local PDF",
        "data_domain": "Regulatory Documents and Safety Alerts",
        "expected_file": "eda/eda_alerts_raw.json",
        "expected_fields": [
            "alert_id", "alert_title", "pdf_url",
            "source_type", "ingestion_date_utc"
        ],
        "update_frequency": "Website-dependent",
        "limitations": "Website structure and accessible documents may change."
    },
    {
        "source_id": "SRC-007",
        "source_name": "WHO Essential Medicines List",
        "publisher": "World Health Organization",
        "url": "https://www.who.int/groups/expert-committee-on-selection-and-use-of-essential-medicines",
        "access_method": "Official Published Files / Local File Ingestion",
        "data_domain": "Drug Reference Data",
        "expected_file": "who/who_eml_raw.json",
        "expected_fields": [
            "medicine", "active_ingredient", "strength",
            "dosage_form"
        ],
        "update_frequency": "Periodic publication",
        "limitations": "No records are considered verified without a valid official source file."
    },
    {
        "source_id": "SRC-008",
        "source_name": "Kaggle Medicines Dataset",
        "publisher": "Kaggle / Original Dataset Publisher",
        "url": "https://www.kaggle.com/datasets",
        "access_method": "Dataset File Ingestion",
        "data_domain": "Supplementary Drug Reference Data",
        "expected_file": "kaggle/kaggle_medicines_raw.json",
        "expected_fields": [
            "medicine_name", "generic_name", "manufacturer"
        ],
        "update_frequency": "Dataset-dependent",
        "limitations": "Dataset provenance, license, and freshness require verification."
    },
    {
        "source_id": "SRC-009",
        "source_name": "Drugs.com Pricing Corpus",
        "publisher": "Drugs.com",
        "url": "https://www.drugs.com/",
        "access_method": "Public Website / Local Document Ingestion",
        "data_domain": "Drug Pricing Reference",
        "expected_file": "rag_corpus/drugs_com_pricing_raw.json",
        "expected_fields": [
            "drug_name", "price", "currency", "price_unit"
        ],
        "update_frequency": "Not established",
        "limitations": "No pricing values are assumed valid unless independently retrieved and verified."
    }
]

# -----------------------------------------------------------------------------
# 3. RAW FILE INSPECTION
# -----------------------------------------------------------------------------

def inspect_raw_source(relative_path):
    file_path = BRONZE_PATH / relative_path

    result = {
        "file_exists": file_path.exists(),
        "file_size_bytes": 0,
        "record_count": 0,
        "json_valid": False,
        "metadata": {},
        "verification_status": "NOT_VERIFIED",
        "inspection_error": None
    }

    if not file_path.exists():
        result["verification_status"] = "FILE_NOT_FOUND"
        return result

    result["file_size_bytes"] = file_path.stat().st_size

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        result["json_valid"] = True

        if isinstance(payload, dict):
            metadata = payload.get("metadata", {})
            records = payload.get("results", [])

            result["metadata"] = (
                metadata if isinstance(metadata, dict) else {}
            )

            result["record_count"] = (
                len(records) if isinstance(records, list) else 0
            )

            declared_status = str(
                result["metadata"].get(
                    "verification_status",
                    result["metadata"].get("execution_status", "")
                )
            ).upper()

            if result["record_count"] > 0:
                result["verification_status"] = "DATA_PRESENT_UNVERIFIED"
            elif declared_status:
                result["verification_status"] = declared_status
            else:
                result["verification_status"] = "EMPTY_RESPONSE"

        elif isinstance(payload, list):
            result["record_count"] = len(payload)
            result["verification_status"] = (
                "DATA_PRESENT_UNVERIFIED"
                if payload else "EMPTY_RESPONSE"
            )
        else:
            result["verification_status"] = "UNEXPECTED_JSON_STRUCTURE"

    except Exception as exc:
        result["inspection_error"] = str(exc)
        result["verification_status"] = "INVALID_JSON"

    return result


# -----------------------------------------------------------------------------
# 4. BUILD SOURCE REGISTER
# -----------------------------------------------------------------------------

register_records = []

for source in SOURCE_DEFINITIONS:

    inspection = inspect_raw_source(source["expected_file"])
    metadata = inspection["metadata"]

    record = {
        **source,
        "inspection_timestamp_utc": AUDIT_TIMESTAMP,
        "file_exists": inspection["file_exists"],
        "file_size_bytes": inspection["file_size_bytes"],
        "record_count": inspection["record_count"],
        "json_valid": inspection["json_valid"],
        "verification_status": inspection["verification_status"],
        "ingestion_timestamp_utc": metadata.get(
            "ingestion_timestamp_utc",
            metadata.get("execution_timestamp_utc", None)
        ),
        "source_status": metadata.get(
            "execution_status",
            metadata.get("status", "NOT_RECORDED")
        ),
        "source_url_recorded": metadata.get("source_url", None),
        "inspection_error": inspection["inspection_error"]
    }

    if not inspection["file_exists"]:
        record["inventory_status"] = "MISSING_FILE"
    elif not inspection["json_valid"]:
        record["inventory_status"] = "INVALID_JSON"
    elif inspection["record_count"] > 0:
        record["inventory_status"] = "RECORDS_AVAILABLE"
    else:
        record["inventory_status"] = "NO_RECORDS"

    register_records.append(record)


# -----------------------------------------------------------------------------
# 5. SAVE REGISTER
# -----------------------------------------------------------------------------

register_df = pd.DataFrame(register_records)

register_df.to_csv(
    REGISTER_CSV,
    index=False,
    encoding="utf-8-sig"
)

with open(REGISTER_JSON, "w", encoding="utf-8") as f:
    json.dump(
        register_records,
        f,
        ensure_ascii=False,
        indent=4,
        default=str
    )


# -----------------------------------------------------------------------------
# 6. INVENTORY AUDIT
# -----------------------------------------------------------------------------

total_sources = len(register_records)

files_found = sum(
    1 for x in register_records if x["file_exists"]
)

valid_json_files = sum(
    1 for x in register_records if x["json_valid"]
)

sources_with_records = sum(
    1 for x in register_records if x["record_count"] > 0
)

empty_sources = sum(
    1 for x in register_records
    if x["inventory_status"] == "NO_RECORDS"
)

missing_sources = sum(
    1 for x in register_records
    if x["inventory_status"] == "MISSING_FILE"
)

invalid_sources = sum(
    1 for x in register_records
    if x["inventory_status"] == "INVALID_JSON"
)

total_records = sum(
    x["record_count"] for x in register_records
)

audit_report = {
    "cell": "CELL 15",
    "audit_name": "Data Source Register and Ingestion Inventory",
    "execution_timestamp_utc": AUDIT_TIMESTAMP,
    "audit_type": "READ_ONLY",
    "total_registered_sources": total_sources,
    "files_found": files_found,
    "valid_json_files": valid_json_files,
    "sources_with_records": sources_with_records,
    "empty_sources": empty_sources,
    "missing_sources": missing_sources,
    "invalid_sources": invalid_sources,
    "total_records_across_registered_files": total_records,
    "register_csv": str(REGISTER_CSV),
    "register_json": str(REGISTER_JSON),
    "limitations": [
        "File presence does not prove source authenticity.",
        "Records are not classified as verified market inventory.",
        "Source URLs and publisher metadata require manual confirmation.",
        "Empty API responses are not automatically interpreted as end-of-data.",
        "This cell does not modify Bronze, Silver, or Gold datasets."
    ]
}

with open(AUDIT_JSON, "w", encoding="utf-8") as f:
    json.dump(audit_report, f, ensure_ascii=False, indent=4)


# -----------------------------------------------------------------------------
# 7. EXECUTION SUMMARY
# -----------------------------------------------------------------------------

print("\nSOURCE REGISTER SUMMARY")
print("-" * 90)

print(f"Registered Sources       : {total_sources}")
print(f"Files Found              : {files_found}")
print(f"Valid JSON Files         : {valid_json_files}")
print(f"Sources With Records     : {sources_with_records}")
print(f"Empty Sources            : {empty_sources}")
print(f"Missing Files            : {missing_sources}")
print(f"Invalid JSON Files       : {invalid_sources}")
print(f"Total Raw Records        : {total_records}")

print("\nSOURCE INVENTORY")
print("-" * 90)

display_columns = [
    "source_id",
    "source_name",
    "record_count",
    "inventory_status",
    "verification_status"
]

print(register_df[display_columns].to_string(index=False))

print("\nGENERATED FILES")
print("-" * 90)
print(f"CSV Register : {REGISTER_CSV}")
print(f"JSON Register: {REGISTER_JSON}")
print(f"Audit Report : {AUDIT_JSON}")

print("\nCELL 15 EXECUTION COMPLETED")
print("=" * 90)

CELL 15 — DATA SOURCE REGISTER & INGESTION INVENTORY

SOURCE REGISTER SUMMARY
------------------------------------------------------------------------------------------
Registered Sources       : 9
Files Found              : 9
Valid JSON Files         : 9
Sources With Records     : 4
Empty Sources            : 5
Missing Files            : 0
Invalid JSON Files       : 0
Total Raw Records        : 138653

SOURCE INVENTORY
------------------------------------------------------------------------------------------
source_id                           source_name  record_count  inventory_status             verification_status
  SRC-001                      openFDA Drug NDC           300 RECORDS_AVAILABLE         DATA_PRESENT_UNVERIFIED
  SRC-002              openFDA Drug Enforcement           150 RECORDS_AVAILABLE         DATA_PRESENT_UNVERIFIED
  SRC-003                openFDA Drug Shortages             0        NO_RECORDS       COMPLETED_END_OF_API_DATA
  SRC-004                 FDA NDC Bul

In [31]:
# =============================================================================
# CELL 16 — INGESTION RELIABILITY & HTTP STATUS VALIDATION
# Pharma Supply Chain Intelligence
# =============================================================================

import json
import time
import requests
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

print("=" * 90)
print("CELL 16 — INGESTION RELIABILITY & HTTP STATUS VALIDATION")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. PATHS AND CONFIGURATION
# -----------------------------------------------------------------------------

BASE_PATH = Path(BASE_DIR)
BRONZE_PATH = BASE_PATH / "data" / "bronze"
REPORTS_PATH = BASE_PATH / "reports"

REPORTS_PATH.mkdir(parents=True, exist_ok=True)

REGISTER_JSON = REPORTS_PATH / "cell_15_data_source_register.json"

RELIABILITY_CSV = REPORTS_PATH / "cell_16_ingestion_reliability.csv"
RELIABILITY_JSON = REPORTS_PATH / "cell_16_ingestion_reliability.json"
RETRY_POLICY_JSON = REPORTS_PATH / "cell_16_retry_policy.json"

TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

# -----------------------------------------------------------------------------
# 2. RETRY POLICY
# -----------------------------------------------------------------------------

RETRY_CONFIG = {
    "total_retries": 3,
    "backoff_factor": 1,
    "status_forcelist": [429, 500, 502, 503, 504],
    "allowed_methods": ["GET"],
    "respect_retry_after_header": True,
    "connect_timeout_seconds": 10,
    "read_timeout_seconds": 30
}

retry_policy = {
    "policy_name": "HTTP_INGESTION_RETRY_POLICY",
    "created_at_utc": TIMESTAMP_UTC,
    "configuration": RETRY_CONFIG,
    "retryable_status_codes": RETRY_CONFIG["status_forcelist"],
    "non_retryable_status_codes": [400, 401, 403, 404],
    "notes": [
        "This policy is provided for subsequent ingestion executions.",
        "This cell does not claim that previous ingestion requests used retries.",
        "HTTP 404 is not interpreted as successful end-of-data.",
        "HTTP 429 and transient server errors are retryable."
    ]
}

with open(RETRY_POLICY_JSON, "w", encoding="utf-8") as f:
    json.dump(retry_policy, f, ensure_ascii=False, indent=4)

# -----------------------------------------------------------------------------
# 3. SESSION FACTORY WITH RETRIES
# -----------------------------------------------------------------------------

def create_retry_session():

    retry_strategy = Retry(
        total=RETRY_CONFIG["total_retries"],
        connect=RETRY_CONFIG["total_retries"],
        read=RETRY_CONFIG["total_retries"],
        status=RETRY_CONFIG["total_retries"],
        backoff_factor=RETRY_CONFIG["backoff_factor"],
        status_forcelist=RETRY_CONFIG["status_forcelist"],
        allowed_methods=RETRY_CONFIG["allowed_methods"],
        respect_retry_after_header=True,
        raise_on_status=False
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)

    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    session.headers.update({
        "User-Agent": "PharmaSupplyChainIntelligence/1.0",
        "Accept": "application/json"
    })

    return session


# -----------------------------------------------------------------------------
# 4. LOAD SOURCE REGISTER
# -----------------------------------------------------------------------------

with open(REGISTER_JSON, "r", encoding="utf-8") as f:
    source_register = json.load(f)

reliability_records = []

# -----------------------------------------------------------------------------
# 5. INSPECT API RESPONSE METADATA
# -----------------------------------------------------------------------------

for source in source_register:

    relative_file = source.get("expected_file")
    file_path = BRONZE_PATH / relative_file

    payload = {}
    metadata = {}

    if file_path.exists():
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                payload = json.load(f)

            if isinstance(payload, dict):
                metadata = payload.get("metadata", {})

                if not isinstance(metadata, dict):
                    metadata = {}

        except Exception:
            payload = {}
            metadata = {}

    http_status = metadata.get(
        "http_status",
        metadata.get(
            "status_code",
            metadata.get("response_status", None)
        )
    )

    try:
        http_status = int(http_status) if http_status is not None else None
    except (TypeError, ValueError):
        http_status = None

    record_count = source.get("record_count", 0)

    execution_status = str(
        source.get("source_status", "NOT_RECORDED")
    ).upper()

    verification_status = str(
        source.get("verification_status", "NOT_VERIFIED")
    ).upper()

    # -------------------------------------------------------------------------
    # Classify ingestion outcome without inventing success
    # -------------------------------------------------------------------------

    if not file_path.exists():
        reliability_status = "MISSING_RAW_FILE"

    elif not source.get("json_valid", False):
        reliability_status = "INVALID_RAW_JSON"

    elif http_status is not None and http_status == 404:
        reliability_status = "HTTP_404_ENDPOINT_OR_RESOURCE_NOT_FOUND"

    elif http_status is not None and http_status in [429, 500, 502, 503, 504]:
        reliability_status = "HTTP_TRANSIENT_ERROR"

    elif http_status is not None and http_status in [401, 403]:
        reliability_status = "HTTP_ACCESS_DENIED"

    elif http_status is not None and http_status >= 400:
        reliability_status = "HTTP_CLIENT_OR_SERVER_ERROR"

    elif record_count > 0:
        reliability_status = "DATA_RECEIVED_HTTP_STATUS_NOT_FULLY_VERIFIED"

    elif execution_status in [
        "TIMEOUT",
        "FAILED",
        "ERROR",
        "HTTP_ERROR"
    ]:
        reliability_status = "INGESTION_EXECUTION_FAILED"

    elif verification_status == "NOT_VERIFIED":
        reliability_status = "SOURCE_NOT_VERIFIED"

    elif record_count == 0:
        reliability_status = "EMPTY_RESPONSE_REQUIRES_VALIDATION"

    else:
        reliability_status = "STATUS_REQUIRES_MANUAL_REVIEW"

    # -------------------------------------------------------------------------
    # Determine whether HTTP status is available
    # -------------------------------------------------------------------------

    if http_status is None:
        http_status_evidence = "NOT_RECORDED"
    else:
        http_status_evidence = str(http_status)

    reliability_records.append({
        "source_id": source.get("source_id"),
        "source_name": source.get("source_name"),
        "expected_file": relative_file,
        "record_count": record_count,
        "http_status": http_status_evidence,
        "execution_status": execution_status,
        "previous_verification_status": verification_status,
        "reliability_status": reliability_status,
        "retry_policy_defined": True,
        "retry_execution_verified": False,
        "requires_manual_review": reliability_status not in [
            "DATA_RECEIVED_HTTP_STATUS_NOT_FULLY_VERIFIED"
        ],
        "audit_timestamp_utc": TIMESTAMP_UTC
    })


# -----------------------------------------------------------------------------
# 6. BUILD RELIABILITY REPORT
# -----------------------------------------------------------------------------

reliability_df = pd.DataFrame(reliability_records)

reliability_df.to_csv(
    RELIABILITY_CSV,
    index=False,
    encoding="utf-8-sig"
)

with open(RELIABILITY_JSON, "w", encoding="utf-8") as f:
    json.dump(
        reliability_records,
        f,
        ensure_ascii=False,
        indent=4,
        default=str
    )


# -----------------------------------------------------------------------------
# 7. SUMMARY METRICS
# -----------------------------------------------------------------------------

total_sources = len(reliability_records)

data_received = sum(
    1 for x in reliability_records
    if x["record_count"] > 0
)

http_404_count = sum(
    1 for x in reliability_records
    if x["reliability_status"] ==
    "HTTP_404_ENDPOINT_OR_RESOURCE_NOT_FOUND"
)

transient_errors = sum(
    1 for x in reliability_records
    if x["reliability_status"] == "HTTP_TRANSIENT_ERROR"
)

empty_responses = sum(
    1 for x in reliability_records
    if x["reliability_status"] ==
    "EMPTY_RESPONSE_REQUIRES_VALIDATION"
)

manual_review_count = sum(
    1 for x in reliability_records
    if x["requires_manual_review"]
)

reliability_summary = {
    "cell": "CELL 16",
    "audit_name": "Ingestion Reliability and HTTP Status Validation",
    "execution_timestamp_utc": TIMESTAMP_UTC,
    "total_sources": total_sources,
    "sources_with_data": data_received,
    "http_404_count": http_404_count,
    "transient_error_count": transient_errors,
    "empty_response_count": empty_responses,
    "sources_requiring_manual_review": manual_review_count,
    "retry_policy_created": True,
    "retry_execution_verified": False,
    "retry_policy_file": str(RETRY_POLICY_JSON),
    "reliability_csv": str(RELIABILITY_CSV),
    "reliability_json": str(RELIABILITY_JSON),
    "important_note": (
        "This cell audits existing ingestion evidence and defines a reusable "
        "retry policy. It does not re-execute historical ingestion requests."
    )
}

SUMMARY_JSON = REPORTS_PATH / "cell_16_reliability_summary.json"

with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(
        reliability_summary,
        f,
        ensure_ascii=False,
        indent=4
    )


# -----------------------------------------------------------------------------
# 8. DISPLAY RESULTS
# -----------------------------------------------------------------------------

print("\nRELIABILITY SUMMARY")
print("-" * 90)

print(f"Total Sources                 : {total_sources}")
print(f"Sources With Data             : {data_received}")
print(f"HTTP 404 Errors                : {http_404_count}")
print(f"Transient HTTP Errors          : {transient_errors}")
print(f"Empty Responses                : {empty_responses}")
print(f"Sources Requiring Review       : {manual_review_count}")
print(f"Retry Policy Created           : YES")
print(f"Historical Retry Execution     : NOT VERIFIED")

print("\nSOURCE RELIABILITY DETAILS")
print("-" * 90)

print(
    reliability_df[
        [
            "source_id",
            "source_name",
            "record_count",
            "http_status",
            "reliability_status"
        ]
    ].to_string(index=False)
)

print("\nGENERATED FILES")
print("-" * 90)
print(f"Reliability CSV : {RELIABILITY_CSV}")
print(f"Reliability JSON: {RELIABILITY_JSON}")
print(f"Retry Policy    : {RETRY_POLICY_JSON}")
print(f"Summary Report  : {SUMMARY_JSON}")

print("\nCELL 16 EXECUTION COMPLETED")
print("=" * 90)

CELL 16 — INGESTION RELIABILITY & HTTP STATUS VALIDATION

RELIABILITY SUMMARY
------------------------------------------------------------------------------------------
Total Sources                 : 9
Sources With Data             : 4
HTTP 404 Errors                : 0
Transient HTTP Errors          : 0
Empty Responses                : 4
Sources Requiring Review       : 5
Retry Policy Created           : YES
Historical Retry Execution     : NOT VERIFIED

SOURCE RELIABILITY DETAILS
------------------------------------------------------------------------------------------
source_id                           source_name  record_count  http_status                           reliability_status
  SRC-001                      openFDA Drug NDC           300 NOT_RECORDED DATA_RECEIVED_HTTP_STATUS_NOT_FULLY_VERIFIED
  SRC-002              openFDA Drug Enforcement           150 NOT_RECORDED DATA_RECEIVED_HTTP_STATUS_NOT_FULLY_VERIFIED
  SRC-003                openFDA Drug Shortages             0

In [32]:
# =============================================================================
# CELL 17 — LIVE API CONNECTIVITY & RETRY EXECUTION TEST
# Pharma Supply Chain Intelligence
# =============================================================================

import json
import requests
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

print("=" * 90)
print("CELL 17 — LIVE API CONNECTIVITY & RETRY EXECUTION TEST")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. CONFIGURATION
# -----------------------------------------------------------------------------

BASE_PATH = Path(BASE_DIR)
REPORTS_PATH = BASE_PATH / "reports"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

OUTPUT_CSV = REPORTS_PATH / "cell_17_api_connectivity_tests.csv"
OUTPUT_JSON = REPORTS_PATH / "cell_17_api_connectivity_tests.json"
SUMMARY_JSON = REPORTS_PATH / "cell_17_api_connectivity_summary.json"

# -----------------------------------------------------------------------------
# 2. API TEST DEFINITIONS
# -----------------------------------------------------------------------------

API_TESTS = [
    {
        "source_id": "SRC-001",
        "source_name": "openFDA Drug NDC",
        "url": "https://api.fda.gov/drug/ndc.json",
        "params": {"limit": 1}
    },
    {
        "source_id": "SRC-002",
        "source_name": "openFDA Drug Enforcement",
        "url": "https://api.fda.gov/drug/enforcement.json",
        "params": {"limit": 1}
    },
    {
        "source_id": "SRC-005",
        "source_name": "NIH RxNav",
        "url": "https://rxnav.nlm.nih.gov/REST/version.json",
        "params": {}
    }
]

# -----------------------------------------------------------------------------
# 3. RETRY-ENABLED SESSION
# -----------------------------------------------------------------------------

RETRY_CONFIG = {
    "total": 3,
    "connect": 3,
    "read": 3,
    "status": 3,
    "backoff_factor": 1,
    "status_forcelist": [429, 500, 502, 503, 504],
    "allowed_methods": ["GET"],
    "respect_retry_after_header": True
}

retry_strategy = Retry(
    total=RETRY_CONFIG["total"],
    connect=RETRY_CONFIG["connect"],
    read=RETRY_CONFIG["read"],
    status=RETRY_CONFIG["status"],
    backoff_factor=RETRY_CONFIG["backoff_factor"],
    status_forcelist=RETRY_CONFIG["status_forcelist"],
    allowed_methods=RETRY_CONFIG["allowed_methods"],
    respect_retry_after_header=RETRY_CONFIG["respect_retry_after_header"],
    raise_on_status=False
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session = requests.Session()
session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": "PharmaSupplyChainIntelligence/1.0",
    "Accept": "application/json"
})

# -----------------------------------------------------------------------------
# 4. EXECUTE CONNECTIVITY TESTS
# -----------------------------------------------------------------------------

test_results = []

for test in API_TESTS:

    print(f"\nTesting: {test['source_name']}")

    result = {
        "source_id": test["source_id"],
        "source_name": test["source_name"],
        "url": test["url"],
        "test_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "http_status": None,
        "response_time_seconds": None,
        "response_valid_json": False,
        "response_has_data": False,
        "response_record_count": 0,
        "retry_attempts": 0,
        "retry_history": [],
        "connectivity_status": "NOT_TESTED",
        "error": None
    }

    try:
        import time

        start_time = time.time()

        response = session.get(
            test["url"],
            params=test["params"],
            timeout=(10, 30)
        )

        elapsed = time.time() - start_time

        result["http_status"] = response.status_code
        result["response_time_seconds"] = round(elapsed, 3)

        retry_object = getattr(response.raw, "retries", None)

        if retry_object is not None:
            retry_history = getattr(retry_object, "history", [])

            result["retry_attempts"] = len(retry_history)

            result["retry_history"] = [
                {
                    "status": item.status,
                    "method": item.method,
                    "error": str(item.error) if item.error else None
                }
                for item in retry_history
            ]

        if response.status_code == 200:

            try:
                payload = response.json()
                result["response_valid_json"] = True

                if isinstance(payload, dict):
                    records = payload.get("results", [])

                    if isinstance(records, list):
                        result["response_record_count"] = len(records)
                        result["response_has_data"] = len(records) > 0
                    else:
                        result["response_has_data"] = bool(payload)

                elif isinstance(payload, list):
                    result["response_record_count"] = len(payload)
                    result["response_has_data"] = len(payload) > 0

                else:
                    result["response_has_data"] = bool(payload)

                result["connectivity_status"] = (
                    "SUCCESS_WITH_DATA"
                    if result["response_has_data"]
                    else "SUCCESS_EMPTY_RESPONSE"
                )

            except ValueError:
                result["connectivity_status"] = "INVALID_JSON_RESPONSE"

        elif response.status_code == 404:
            result["connectivity_status"] = "ENDPOINT_NOT_FOUND"

        elif response.status_code in [401, 403]:
            result["connectivity_status"] = "ACCESS_DENIED"

        elif response.status_code == 429:
            result["connectivity_status"] = "RATE_LIMITED"

        elif response.status_code >= 500:
            result["connectivity_status"] = "SERVER_ERROR"

        else:
            result["connectivity_status"] = "HTTP_ERROR"

        response.close()

    except requests.exceptions.Timeout as exc:
        result["connectivity_status"] = "TIMEOUT"
        result["error"] = str(exc)

    except requests.exceptions.ConnectionError as exc:
        result["connectivity_status"] = "CONNECTION_ERROR"
        result["error"] = str(exc)

    except requests.exceptions.RequestException as exc:
        result["connectivity_status"] = "REQUEST_ERROR"
        result["error"] = str(exc)

    except Exception as exc:
        result["connectivity_status"] = "UNEXPECTED_ERROR"
        result["error"] = str(exc)

    test_results.append(result)

    print(f"HTTP Status : {result['http_status']}")
    print(f"Result      : {result['connectivity_status']}")
    print(f"Retries     : {result['retry_attempts']}")


# -----------------------------------------------------------------------------
# 5. SAVE TEST RESULTS
# -----------------------------------------------------------------------------

results_df = pd.DataFrame(test_results)

results_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(
        test_results,
        f,
        ensure_ascii=False,
        indent=4,
        default=str
    )


# -----------------------------------------------------------------------------
# 6. SUMMARY
# -----------------------------------------------------------------------------

total_tests = len(test_results)

successful_tests = sum(
    1 for x in test_results
    if x["connectivity_status"] in [
        "SUCCESS_WITH_DATA",
        "SUCCESS_EMPTY_RESPONSE"
    ]
)

failed_tests = total_tests - successful_tests

actual_retry_attempts = sum(
    x["retry_attempts"] for x in test_results
)

retry_policy_applied = True

summary = {
    "cell": "CELL 17",
    "test_name": "Live API Connectivity and Retry Execution",
    "execution_timestamp_utc": TIMESTAMP_UTC,
    "total_tests": total_tests,
    "successful_tests": successful_tests,
    "failed_tests": failed_tests,
    "actual_retry_attempts_observed": actual_retry_attempts,
    "retry_policy_applied": retry_policy_applied,
    "retry_execution_observed": actual_retry_attempts > 0,
    "connectivity_success_rate_percent": round(
        (successful_tests / total_tests) * 100, 2
    ) if total_tests else 0,
    "results_csv": str(OUTPUT_CSV),
    "results_json": str(OUTPUT_JSON),
    "limitations": [
        "Tests verify current API connectivity only.",
        "Tests do not re-ingest or overwrite Bronze datasets.",
        "Zero retry attempts means no retryable failure occurred during this test.",
        "Successful connectivity does not guarantee complete historical data."
    ]
}

with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=4)


# -----------------------------------------------------------------------------
# 7. DISPLAY
# -----------------------------------------------------------------------------

print("\n" + "=" * 90)
print("API CONNECTIVITY TEST SUMMARY")
print("-" * 90)

print(f"Total Tests              : {total_tests}")
print(f"Successful Tests         : {successful_tests}")
print(f"Failed Tests             : {failed_tests}")
print(f"Actual Retry Attempts    : {actual_retry_attempts}")
print(f"Retry Policy Applied     : {retry_policy_applied}")
print(f"Retry Execution Observed : {actual_retry_attempts > 0}")
print(
    f"Connectivity Success Rate: "
    f"{summary['connectivity_success_rate_percent']}%"
)

print("\nDETAILED RESULTS")
print("-" * 90)

print(
    results_df[
        [
            "source_id",
            "source_name",
            "http_status",
            "response_time_seconds",
            "retry_attempts",
            "connectivity_status"
        ]
    ].to_string(index=False)
)

print("\nGENERATED FILES")
print("-" * 90)
print(f"CSV Results    : {OUTPUT_CSV}")
print(f"JSON Results   : {OUTPUT_JSON}")
print(f"Summary Report : {SUMMARY_JSON}")

print("\nCELL 17 EXECUTION COMPLETED")
print("=" * 90)

CELL 17 — LIVE API CONNECTIVITY & RETRY EXECUTION TEST

Testing: openFDA Drug NDC
HTTP Status : 200
Result      : SUCCESS_WITH_DATA
Retries     : 0

Testing: openFDA Drug Enforcement
HTTP Status : 200
Result      : SUCCESS_WITH_DATA
Retries     : 0

Testing: NIH RxNav
HTTP Status : 200
Result      : SUCCESS_EMPTY_RESPONSE
Retries     : 0

API CONNECTIVITY TEST SUMMARY
------------------------------------------------------------------------------------------
Total Tests              : 3
Successful Tests         : 3
Failed Tests             : 0
Actual Retry Attempts    : 0
Retry Policy Applied     : True
Retry Execution Observed : False
Connectivity Success Rate: 100.0%

DETAILED RESULTS
------------------------------------------------------------------------------------------
source_id              source_name  http_status  response_time_seconds  retry_attempts    connectivity_status
  SRC-001         openFDA Drug NDC          200                  0.483               0      SUCCESS_WITH

In [33]:
# =============================================================================
# CELL 18 — BRONZE LIVE INGESTION & EXECUTION LOGGING
# Pharma Supply Chain Intelligence
# =============================================================================

import json
import requests
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

print("=" * 90)
print("CELL 18 — BRONZE LIVE INGESTION & EXECUTION LOGGING")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. CONFIGURATION
# -----------------------------------------------------------------------------

BASE_PATH = Path(BASE_DIR)
BRONZE_PATH = BASE_PATH / "data" / "bronze" / "live_api"
REPORTS_PATH = BASE_PATH / "reports"

BRONZE_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now(timezone.utc)
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%dT%H%M%SZ")

# -----------------------------------------------------------------------------
# 2. RETRY-ENABLED HTTP SESSION
# -----------------------------------------------------------------------------

retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
    raise_on_status=False
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session = requests.Session()
session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": "PharmaSupplyChainIntelligence/1.0",
    "Accept": "application/json"
})

# -----------------------------------------------------------------------------
# 3. INGESTION SOURCE DEFINITIONS
# -----------------------------------------------------------------------------

INGESTION_SOURCES = [
    {
        "source_id": "SRC-001",
        "source_name": "openFDA Drug NDC",
        "endpoint": "https://api.fda.gov/drug/ndc.json",
        "params": {"limit": 100},
        "output_name": "openfda_ndc"
    },
    {
        "source_id": "SRC-002",
        "source_name": "openFDA Drug Enforcement",
        "endpoint": "https://api.fda.gov/drug/enforcement.json",
        "params": {"limit": 100},
        "output_name": "openfda_enforcement"
    },
    {
        "source_id": "SRC-005",
        "source_name": "NIH RxNav Version",
        "endpoint": "https://rxnav.nlm.nih.gov/REST/version.json",
        "params": {},
        "output_name": "rxnav_version"
    }
]

# -----------------------------------------------------------------------------
# 4. REUSABLE INGESTION FUNCTION
# -----------------------------------------------------------------------------

def ingest_api_source(source):

    started_at = datetime.now(timezone.utc)

    result = {
        "run_id": RUN_ID,
        "source_id": source["source_id"],
        "source_name": source["source_name"],
        "endpoint": source["endpoint"],
        "started_at_utc": started_at.isoformat(),
        "completed_at_utc": None,
        "http_status": None,
        "response_time_seconds": None,
        "record_count": 0,
        "response_valid_json": False,
        "ingestion_status": "FAILED",
        "retry_attempts": 0,
        "output_file": None,
        "error": None
    }

    try:
        import time

        start_time = time.time()

        response = session.get(
            source["endpoint"],
            params=source["params"],
            timeout=(10, 30)
        )

        result["response_time_seconds"] = round(
            time.time() - start_time, 3
        )

        result["http_status"] = response.status_code

        retry_object = getattr(response.raw, "retries", None)

        if retry_object is not None:
            result["retry_attempts"] = len(
                getattr(retry_object, "history", [])
            )

        if response.status_code != 200:
            result["ingestion_status"] = (
                "HTTP_ERROR_" + str(response.status_code)
            )
            result["error"] = response.text[:1000]
            response.close()
            result["completed_at_utc"] = datetime.now(
                timezone.utc
            ).isoformat()
            return result

        try:
            payload = response.json()
            result["response_valid_json"] = True
        except ValueError:
            result["ingestion_status"] = "INVALID_JSON"
            result["error"] = "Response could not be parsed as JSON."
            response.close()
            result["completed_at_utc"] = datetime.now(
                timezone.utc
            ).isoformat()
            return result

        response.close()

        if isinstance(payload, dict):
            records = payload.get("results")

            if isinstance(records, list):
                result["record_count"] = len(records)
            elif payload:
                result["record_count"] = 1

        elif isinstance(payload, list):
            result["record_count"] = len(payload)

        # Preserve the complete source response and its lineage metadata.
        bronze_payload = {
            "metadata": {
                "source_id": source["source_id"],
                "source_name": source["source_name"],
                "source_endpoint": source["endpoint"],
                "ingestion_timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "run_id": RUN_ID,
                "http_status": result["http_status"],
                "response_time_seconds": result[
                    "response_time_seconds"
                ],
                "record_count": result["record_count"],
                "ingestion_method": "HTTP_GET",
                "data_layer": "BRONZE"
            },
            "data": payload
        }

        output_file = (
            BRONZE_PATH /
            f"{source['output_name']}_{RUN_ID}.json"
        )

        with open(output_file, "w", encoding="utf-8") as file:
            json.dump(
                bronze_payload,
                file,
                ensure_ascii=False,
                indent=2,
                default=str
            )

        result["output_file"] = str(output_file)

        result["ingestion_status"] = (
            "SUCCESS_WITH_DATA"
            if result["record_count"] > 0
            else "SUCCESS_EMPTY_RESPONSE"
        )

    except requests.exceptions.Timeout as exc:
        result["ingestion_status"] = "TIMEOUT"
        result["error"] = str(exc)

    except requests.exceptions.ConnectionError as exc:
        result["ingestion_status"] = "CONNECTION_ERROR"
        result["error"] = str(exc)

    except requests.exceptions.RequestException as exc:
        result["ingestion_status"] = "REQUEST_ERROR"
        result["error"] = str(exc)

    except Exception as exc:
        result["ingestion_status"] = "UNEXPECTED_ERROR"
        result["error"] = str(exc)

    result["completed_at_utc"] = datetime.now(
        timezone.utc
    ).isoformat()

    return result


# -----------------------------------------------------------------------------
# 5. RUN BRONZE INGESTION
# -----------------------------------------------------------------------------

ingestion_results = []

for source in INGESTION_SOURCES:

    print(f"\nIngesting: {source['source_name']}")

    ingestion_result = ingest_api_source(source)
    ingestion_results.append(ingestion_result)

    print(f"HTTP Status : {ingestion_result['http_status']}")
    print(f"Records     : {ingestion_result['record_count']}")
    print(f"Status      : {ingestion_result['ingestion_status']}")
    print(f"Retries     : {ingestion_result['retry_attempts']}")

# -----------------------------------------------------------------------------
# 6. SAVE EXECUTION LOGS
# -----------------------------------------------------------------------------

LOG_CSV = REPORTS_PATH / "cell_18_bronze_ingestion_log.csv"
LOG_JSON = REPORTS_PATH / "cell_18_bronze_ingestion_log.json"
SUMMARY_JSON = REPORTS_PATH / "cell_18_bronze_ingestion_summary.json"

results_df = pd.DataFrame(ingestion_results)

results_df.to_csv(
    LOG_CSV,
    index=False,
    encoding="utf-8-sig"
)

with open(LOG_JSON, "w", encoding="utf-8") as file:
    json.dump(
        ingestion_results,
        file,
        ensure_ascii=False,
        indent=4,
        default=str
    )

# -----------------------------------------------------------------------------
# 7. BUILD SUMMARY
# -----------------------------------------------------------------------------

successful = sum(
    1 for result in ingestion_results
    if result["ingestion_status"] == "SUCCESS_WITH_DATA"
)

empty = sum(
    1 for result in ingestion_results
    if result["ingestion_status"] == "SUCCESS_EMPTY_RESPONSE"
)

failed = len(ingestion_results) - successful - empty

total_records = sum(
    result["record_count"] for result in ingestion_results
)

summary = {
    "cell": "CELL 18",
    "run_id": RUN_ID,
    "execution_timestamp_utc": RUN_TIMESTAMP.isoformat(),
    "total_sources_attempted": len(ingestion_results),
    "successful_sources_with_data": successful,
    "successful_empty_responses": empty,
    "failed_sources": failed,
    "total_records_ingested": total_records,
    "bronze_output_directory": str(BRONZE_PATH),
    "execution_log_csv": str(LOG_CSV),
    "execution_log_json": str(LOG_JSON),
    "data_preservation": "NEW_TIMESTAMPED_FILES; EXISTING_BRONZE_FILES_NOT_OVERWRITTEN",
    "airflow_execution": "NOT_EXECUTED",
    "limitations": [
        "Only the configured API samples were ingested.",
        "The NDC and enforcement endpoints request up to 100 records each.",
        "The RxNav version endpoint is metadata, not a drug-concept dataset.",
        "A successful API request does not establish complete source coverage."
    ]
}

with open(SUMMARY_JSON, "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=4)

# -----------------------------------------------------------------------------
# 8. DISPLAY RESULTS
# -----------------------------------------------------------------------------

print("\n" + "=" * 90)
print("BRONZE INGESTION SUMMARY")
print("-" * 90)

print(f"Sources Attempted       : {len(ingestion_results)}")
print(f"Successful With Data    : {successful}")
print(f"Successful Empty        : {empty}")
print(f"Failed Sources          : {failed}")
print(f"Total Records Ingested  : {total_records}")

print("\nINGESTION DETAILS")
print("-" * 90)

print(
    results_df[
        [
            "source_id",
            "source_name",
            "http_status",
            "record_count",
            "retry_attempts",
            "ingestion_status"
        ]
    ].to_string(index=False)
)

print("\nGENERATED FILES")
print("-" * 90)
print(f"Bronze Directory : {BRONZE_PATH}")
print(f"Execution CSV    : {LOG_CSV}")
print(f"Execution JSON   : {LOG_JSON}")
print(f"Summary Report   : {SUMMARY_JSON}")

print("\nCELL 18 EXECUTION COMPLETED")
print("=" * 90)

CELL 18 — BRONZE LIVE INGESTION & EXECUTION LOGGING

Ingesting: openFDA Drug NDC
HTTP Status : 200
Records     : 100
Status      : SUCCESS_WITH_DATA
Retries     : 0

Ingesting: openFDA Drug Enforcement
HTTP Status : 200
Records     : 100
Status      : SUCCESS_WITH_DATA
Retries     : 0

Ingesting: NIH RxNav Version
HTTP Status : 200
Records     : 1
Status      : SUCCESS_WITH_DATA
Retries     : 0

BRONZE INGESTION SUMMARY
------------------------------------------------------------------------------------------
Sources Attempted       : 3
Successful With Data    : 3
Successful Empty        : 0
Failed Sources          : 0
Total Records Ingested  : 201

INGESTION DETAILS
------------------------------------------------------------------------------------------
source_id              source_name  http_status  record_count  retry_attempts  ingestion_status
  SRC-001         openFDA Drug NDC          200           100               0 SUCCESS_WITH_DATA
  SRC-002 openFDA Drug Enforcement       

In [34]:
# =============================================================================
# CELL 19 — AIRFLOW DAG GENERATION & DEPLOYMENT VALIDATION
# Pharma Supply Chain Intelligence
# =============================================================================

import ast
import json
from pathlib import Path
from datetime import datetime, timezone

print("=" * 90)
print("CELL 19 — AIRFLOW DAG GENERATION & DEPLOYMENT VALIDATION")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. CONFIGURATION
# -----------------------------------------------------------------------------

BASE_PATH = Path(BASE_DIR)
REPORTS_PATH = BASE_PATH / "reports"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

DAG_PATH = REPORTS_PATH / "airflow_pharma_pipeline_dag.py"
RUNNER_PATH = REPORTS_PATH / "pharma_ingestion_runner.py"
SUMMARY_PATH = REPORTS_PATH / "cell_19_airflow_validation.json"

# -----------------------------------------------------------------------------
# 2. GENERATE STANDALONE INGESTION RUNNER
# -----------------------------------------------------------------------------

runner_code = r'''
import json
import os
from pathlib import Path
from datetime import datetime, timezone

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BRONZE_PATH = Path(
    os.environ.get(
        "PHARMA_BRONZE_PATH",
        "/content/data/bronze/live_api"
    )
)
BRONZE_PATH.mkdir(parents=True, exist_ok=True)

SOURCES = [
    {
        "source_id": "SRC-001",
        "source_name": "openFDA Drug NDC",
        "endpoint": "https://api.fda.gov/drug/ndc.json",
        "params": {"limit": 100},
        "filename": "openfda_ndc"
    },
    {
        "source_id": "SRC-002",
        "source_name": "openFDA Drug Enforcement",
        "endpoint": "https://api.fda.gov/drug/enforcement.json",
        "params": {"limit": 100},
        "filename": "openfda_enforcement"
    },
    {
        "source_id": "SRC-005",
        "source_name": "NIH RxNav Version",
        "endpoint": "https://rxnav.nlm.nih.gov/REST/version.json",
        "params": {},
        "filename": "rxnav_version"
    }
]


def build_session():
    retry = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True,
        raise_on_status=False
    )

    adapter = HTTPAdapter(max_retries=retry)
    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({
        "User-Agent": "PharmaSupplyChainIntelligence/1.0",
        "Accept": "application/json"
    })
    return session


def ingest_source(source):
    session = build_session()
    started_at = datetime.now(timezone.utc)

    try:
        response = session.get(
            source["endpoint"],
            params=source["params"],
            timeout=(10, 30)
        )
        response.raise_for_status()
        payload = response.json()

        retry_object = getattr(response.raw, "retries", None)
        retry_attempts = (
            len(getattr(retry_object, "history", []))
            if retry_object is not None else 0
        )

        if isinstance(payload, dict):
            records = payload.get("results")
            record_count = (
                len(records) if isinstance(records, list)
                else (1 if payload else 0)
            )
        elif isinstance(payload, list):
            record_count = len(payload)
        else:
            record_count = 0

        run_id = datetime.now(timezone.utc).strftime(
            "%Y%m%dT%H%M%SZ"
        )

        output = {
            "metadata": {
                "source_id": source["source_id"],
                "source_name": source["source_name"],
                "source_endpoint": source["endpoint"],
                "request_params": source["params"],
                "ingestion_timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "started_at_utc": started_at.isoformat(),
                "run_id": run_id,
                "http_status": response.status_code,
                "retry_attempts": retry_attempts,
                "record_count": record_count,
                "ingestion_method": "AIRFLOW_PYTHON_OPERATOR",
                "data_layer": "BRONZE"
            },
            "data": payload
        }

        output_file = (
            BRONZE_PATH /
            f"{source['filename']}_{run_id}.json"
        )

        with open(output_file, "w", encoding="utf-8") as file:
            json.dump(output, file, ensure_ascii=False, indent=2)

        response.close()

        return {
            "source_id": source["source_id"],
            "record_count": record_count,
            "http_status": response.status_code,
            "output_file": str(output_file),
            "retry_attempts": retry_attempts
        }

    finally:
        session.close()


def run_source(source_id):
    source = next(
        (item for item in SOURCES if item["source_id"] == source_id),
        None
    )

    if source is None:
        raise ValueError("Unknown source_id: " + source_id)

    return ingest_source(source)
'''

RUNNER_PATH.write_text(runner_code, encoding="utf-8")

# -----------------------------------------------------------------------------
# 3. GENERATE AIRFLOW DAG
# -----------------------------------------------------------------------------

dag_code = r'''
from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.python import PythonOperator

from pharma_ingestion_runner import run_source

default_args = {
    "owner": "pharma_supply_chain_team",
    "depends_on_past": False,
    "retries": 3,
    "retry_delay": timedelta(minutes=2),
    "retry_exponential_backoff": True,
    "max_retry_delay": timedelta(minutes=15)
}

with DAG(
    dag_id="pharma_supply_chain_ingestion",
    description="Bronze ingestion from openFDA and NIH RxNav APIs",
    default_args=default_args,
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    max_active_runs=1,
    tags=["pharma", "data-engineering", "bronze", "ingestion"]
) as dag:

    ingest_openfda_ndc = PythonOperator(
        task_id="ingest_openfda_ndc",
        python_callable=run_source,
        op_kwargs={"source_id": "SRC-001"}
    )

    ingest_openfda_enforcement = PythonOperator(
        task_id="ingest_openfda_enforcement",
        python_callable=run_source,
        op_kwargs={"source_id": "SRC-002"}
    )

    ingest_rxnav_version = PythonOperator(
        task_id="ingest_rxnav_version",
        python_callable=run_source,
        op_kwargs={"source_id": "SRC-005"}
    )

    [
        ingest_openfda_ndc,
        ingest_openfda_enforcement,
        ingest_rxnav_version
    ]
'''

DAG_PATH.write_text(dag_code, encoding="utf-8")

# -----------------------------------------------------------------------------
# 4. VALIDATE SYNTAX
# -----------------------------------------------------------------------------

runner_syntax_valid = False
dag_syntax_valid = False

try:
    ast.parse(runner_code)
    runner_syntax_valid = True
except SyntaxError as exc:
    print("Runner syntax error:", exc)

try:
    ast.parse(dag_code)
    dag_syntax_valid = True
except SyntaxError as exc:
    print("DAG syntax error:", exc)

# -----------------------------------------------------------------------------
# 5. CHECK AIRFLOW INSTALLATION
# -----------------------------------------------------------------------------

try:
    import airflow
    airflow_installed = True
    airflow_version = getattr(airflow, "__version__", "UNKNOWN")
except ImportError:
    airflow_installed = False
    airflow_version = None

# -----------------------------------------------------------------------------
# 6. VALIDATE DAG CONFIGURATION
# -----------------------------------------------------------------------------

required_task_ids = [
    "ingest_openfda_ndc",
    "ingest_openfda_enforcement",
    "ingest_rxnav_version"
]

task_checks = {
    task_id: task_id in dag_code
    for task_id in required_task_ids
}

all_tasks_present = all(task_checks.values())

runner_import_present = (
    "from pharma_ingestion_runner import run_source" in dag_code
)

retry_configuration_present = (
    '"retries": 3' in dag_code
    and '"retry_exponential_backoff": True' in dag_code
)

deployment_status = (
    "READY_FOR_AIRFLOW_IMPORT_TEST"
    if runner_syntax_valid
    and dag_syntax_valid
    and all_tasks_present
    and runner_import_present
    and retry_configuration_present
    else "GENERATION_VALIDATION_FAILED"
)

airflow_test_status = (
    "AIRFLOW_NOT_INSTALLED"
    if not airflow_installed
    else "IMPORT_TEST_REQUIRED_IN_AIRFLOW_ENVIRONMENT"
)

# -----------------------------------------------------------------------------
# 7. SAVE VALIDATION REPORT
# -----------------------------------------------------------------------------

summary = {
    "cell": "CELL 19",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "runner_file": str(RUNNER_PATH),
    "dag_file": str(DAG_PATH),
    "runner_syntax_valid": runner_syntax_valid,
    "dag_syntax_valid": dag_syntax_valid,
    "required_tasks_present": task_checks,
    "runner_import_present": runner_import_present,
    "retry_configuration_present": retry_configuration_present,
    "airflow_installed_in_current_environment": airflow_installed,
    "airflow_version": airflow_version,
    "airflow_test_status": airflow_test_status,
    "deployment_status": deployment_status,
    "dag_execution_status": "NOT_EXECUTED",
    "limitations": [
        "The DAG and runner were generated but not executed by Airflow.",
        "The DAG uses manual scheduling (schedule=None).",
        "The Airflow environment must have requests installed.",
        "Both Python files must be in the Airflow DAG import path.",
        "Configure PHARMA_BRONZE_PATH for the Airflow worker environment."
    ]
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=4)

# -----------------------------------------------------------------------------
# 8. DISPLAY RESULTS
# -----------------------------------------------------------------------------

print("\nGENERATED ARTIFACTS")
print("-" * 90)
print(f"Ingestion Runner : {RUNNER_PATH}")
print(f"Airflow DAG      : {DAG_PATH}")
print(f"Validation JSON  : {SUMMARY_PATH}")

print("\nVALIDATION RESULTS")
print("-" * 90)
print(f"Runner Syntax Valid      : {runner_syntax_valid}")
print(f"DAG Syntax Valid         : {dag_syntax_valid}")
print(f"All Required Tasks       : {all_tasks_present}")
print(f"Runner Import Defined    : {runner_import_present}")
print(f"Retry Configuration      : {retry_configuration_present}")
print(f"Airflow Installed        : {airflow_installed}")
print(f"Airflow Version          : {airflow_version}")
print(f"Airflow Test Status      : {airflow_test_status}")
print(f"Deployment Status        : {deployment_status}")
print("DAG Execution Status     : NOT_EXECUTED")

print("\nCELL 19 EXECUTION COMPLETED")
print("=" * 90)

CELL 19 — AIRFLOW DAG GENERATION & DEPLOYMENT VALIDATION

GENERATED ARTIFACTS
------------------------------------------------------------------------------------------
Ingestion Runner : /content/reports/pharma_ingestion_runner.py
Airflow DAG      : /content/reports/airflow_pharma_pipeline_dag.py
Validation JSON  : /content/reports/cell_19_airflow_validation.json

VALIDATION RESULTS
------------------------------------------------------------------------------------------
Runner Syntax Valid      : True
DAG Syntax Valid         : True
All Required Tasks       : True
Runner Import Defined    : True
Retry Configuration      : True
Airflow Installed        : False
Airflow Version          : None
Airflow Test Status      : AIRFLOW_NOT_INSTALLED
Deployment Status        : READY_FOR_AIRFLOW_IMPORT_TEST
DAG Execution Status     : NOT_EXECUTED

CELL 19 EXECUTION COMPLETED


In [35]:
# =============================================================================
# CELL 20 — BRONZE INGESTION VALIDATION & MEMBER 1 HANDOFF
# Pharma Supply Chain Intelligence
# =============================================================================

import json
import ast
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

print("=" * 90)
print("CELL 20 — BRONZE INGESTION VALIDATION & MEMBER 1 HANDOFF")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. CONFIGURATION
# -----------------------------------------------------------------------------

BASE_PATH = Path(BASE_DIR)
BRONZE_PATH = BASE_PATH / "data" / "bronze" / "live_api"
REPORTS_PATH = BASE_PATH / "reports"

REPORTS_PATH.mkdir(parents=True, exist_ok=True)

INGESTION_LOG = REPORTS_PATH / "cell_18_bronze_ingestion_log.json"
INGESTION_SUMMARY = REPORTS_PATH / "cell_18_bronze_ingestion_summary.json"
AIRFLOW_VALIDATION = REPORTS_PATH / "cell_19_airflow_validation.json"

VALIDATION_CSV = REPORTS_PATH / "cell_20_bronze_validation.csv"
VALIDATION_JSON = REPORTS_PATH / "cell_20_bronze_validation.json"
HANDOFF_JSON = REPORTS_PATH / "member_1_ingestion_handoff.json"

# -----------------------------------------------------------------------------
# 2. LOAD PREVIOUS EXECUTION REPORTS
# -----------------------------------------------------------------------------

required_reports = {
    "ingestion_log": INGESTION_LOG,
    "ingestion_summary": INGESTION_SUMMARY,
    "airflow_validation": AIRFLOW_VALIDATION
}

report_availability = {
    name: path.exists()
    for name, path in required_reports.items()
}

def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)

ingestion_log = (
    load_json(INGESTION_LOG)
    if INGESTION_LOG.exists() else []
)

ingestion_summary = (
    load_json(INGESTION_SUMMARY)
    if INGESTION_SUMMARY.exists() else {}
)

airflow_validation = (
    load_json(AIRFLOW_VALIDATION)
    if AIRFLOW_VALIDATION.exists() else {}
)

# -----------------------------------------------------------------------------
# 3. VALIDATE BRONZE FILES
# -----------------------------------------------------------------------------

expected_sources = {
    "SRC-001": "openfda_ndc",
    "SRC-002": "openfda_enforcement",
    "SRC-005": "rxnav_version"
}

validation_results = []

for source_id, filename_prefix in expected_sources.items():

    matching_files = sorted(
        BRONZE_PATH.glob(f"{filename_prefix}_*.json")
    ) if BRONZE_PATH.exists() else []

    if not matching_files:
        validation_results.append({
            "source_id": source_id,
            "expected_prefix": filename_prefix,
            "file_count": 0,
            "valid_json": False,
            "metadata_present": False,
            "data_present": False,
            "record_count": 0,
            "validation_status": "MISSING_BRONZE_FILE",
            "file_path": None
        })
        continue

    latest_file = matching_files[-1]

    try:
        with open(latest_file, "r", encoding="utf-8") as file:
            payload = json.load(file)

        metadata = payload.get("metadata", {})
        data = payload.get("data")

        metadata_present = bool(
            metadata.get("source_id")
            and metadata.get("ingestion_timestamp_utc")
            and metadata.get("http_status") == 200
        )

        if isinstance(data, dict):
            records = data.get("results")
            record_count = (
                len(records) if isinstance(records, list)
                else (1 if data else 0)
            )
        elif isinstance(data, list):
            record_count = len(data)
        else:
            record_count = 1 if data else 0

        data_present = data is not None

        if metadata_present and data_present:
            status = (
                "VALID_WITH_RECORDS"
                if record_count > 0
                else "VALID_EMPTY_RESPONSE"
            )
        else:
            status = "METADATA_OR_DATA_VALIDATION_FAILED"

        validation_results.append({
            "source_id": source_id,
            "expected_prefix": filename_prefix,
            "file_count": len(matching_files),
            "valid_json": True,
            "metadata_present": metadata_present,
            "data_present": data_present,
            "record_count": record_count,
            "validation_status": status,
            "file_path": str(latest_file)
        })

    except Exception as exc:
        validation_results.append({
            "source_id": source_id,
            "expected_prefix": filename_prefix,
            "file_count": len(matching_files),
            "valid_json": False,
            "metadata_present": False,
            "data_present": False,
            "record_count": 0,
            "validation_status": "INVALID_BRONZE_FILE",
            "file_path": str(latest_file),
            "error": str(exc)
        })

validation_df = pd.DataFrame(validation_results)

# -----------------------------------------------------------------------------
# 4. VALIDATE INGESTION LOG AGAINST BRONZE OUTPUT
# -----------------------------------------------------------------------------

log_checks = []

for entry in ingestion_log:

    source_id = entry.get("source_id")
    expected_file = entry.get("output_file")
    output_exists = bool(
        expected_file and Path(expected_file).exists()
    )

    log_checks.append({
        "source_id": source_id,
        "logged_status": entry.get("ingestion_status"),
        "logged_record_count": entry.get("record_count", 0),
        "output_file_exists": output_exists,
        "log_validation": (
            "PASS"
            if output_exists
            and entry.get("http_status") == 200
            else "REVIEW"
        )
    })

log_df = pd.DataFrame(log_checks)

# -----------------------------------------------------------------------------
# 5. VALIDATE GENERATED AIRFLOW ARTIFACTS
# -----------------------------------------------------------------------------

RUNNER_PATH = REPORTS_PATH / "pharma_ingestion_runner.py"
DAG_PATH = REPORTS_PATH / "airflow_pharma_pipeline_dag.py"

runner_exists = RUNNER_PATH.exists()
dag_exists = DAG_PATH.exists()

runner_syntax_valid = False
dag_syntax_valid = False

if runner_exists:
    try:
        ast.parse(RUNNER_PATH.read_text(encoding="utf-8"))
        runner_syntax_valid = True
    except SyntaxError:
        runner_syntax_valid = False

if dag_exists:
    try:
        ast.parse(DAG_PATH.read_text(encoding="utf-8"))
        dag_syntax_valid = True
    except SyntaxError:
        dag_syntax_valid = False

airflow_installed = airflow_validation.get(
    "airflow_installed_in_current_environment",
    False
)

# -----------------------------------------------------------------------------
# 6. BUILD VALIDATION CHECKS
# -----------------------------------------------------------------------------

checks = []

def add_check(check_name, passed, details):
    checks.append({
        "check_name": check_name,
        "status": "PASS" if passed else "REVIEW",
        "details": details
    })

add_check(
    "ingestion_log_available",
    report_availability["ingestion_log"],
    str(INGESTION_LOG)
)

add_check(
    "ingestion_summary_available",
    report_availability["ingestion_summary"],
    str(INGESTION_SUMMARY)
)

add_check(
    "airflow_validation_available",
    report_availability["airflow_validation"],
    str(AIRFLOW_VALIDATION)
)

add_check(
    "bronze_files_valid",
    len(validation_results) == len(expected_sources)
    and all(row["valid_json"] for row in validation_results),
    "Valid JSON check across expected live API sources"
)

add_check(
    "bronze_metadata_present",
    all(row["metadata_present"] for row in validation_results),
    "Source ID, timestamp and HTTP status metadata"
)

add_check(
    "bronze_data_present",
    all(row["data_present"] for row in validation_results),
    "Source response payload exists"
)

add_check(
    "ingestion_outputs_exist",
    bool(log_checks) and all(
        row["output_file_exists"] for row in log_checks
    ),
    "Cell 18 output files referenced in ingestion log"
)

add_check(
    "airflow_runner_exists",
    runner_exists,
    str(RUNNER_PATH)
)

add_check(
    "airflow_dag_exists",
    dag_exists,
    str(DAG_PATH)
)

add_check(
    "airflow_runner_syntax_valid",
    runner_syntax_valid,
    "Python AST validation"
)

add_check(
    "airflow_dag_syntax_valid",
    dag_syntax_valid,
    "Python AST validation"
)

# -----------------------------------------------------------------------------
# 7. SAVE VALIDATION REPORTS
# -----------------------------------------------------------------------------

validation_report = {
    "cell": "CELL 20",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "bronze_source_validation": validation_results,
    "ingestion_log_validation": log_checks,
    "airflow_artifacts": {
        "runner_exists": runner_exists,
        "dag_exists": dag_exists,
        "runner_syntax_valid": runner_syntax_valid,
        "dag_syntax_valid": dag_syntax_valid,
        "airflow_installed": airflow_installed,
        "dag_execution_verified": False
    },
    "checks": checks,
    "checks_passed": sum(
        1 for check in checks if check["status"] == "PASS"
    ),
    "checks_requiring_review": sum(
        1 for check in checks if check["status"] != "PASS"
    )
}

with open(VALIDATION_JSON, "w", encoding="utf-8") as file:
    json.dump(
        validation_report,
        file,
        ensure_ascii=False,
        indent=4
    )

validation_df.to_csv(
    VALIDATION_CSV,
    index=False,
    encoding="utf-8-sig"
)

# -----------------------------------------------------------------------------
# 8. CREATE MEMBER 1 HANDOFF
# -----------------------------------------------------------------------------

handoff = {
    "project": "Pharma Supply Chain Intelligence",
    "team_role": "Member 1 — Data Ingestion & Workflow Orchestration",
    "handoff_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "completed_deliverables": [
        "Data source register and ingestion inventory",
        "Ingestion reliability audit and retry policy definition",
        "Live API connectivity smoke tests",
        "Timestamped Bronze API ingestion snapshots",
        "Ingestion execution logs and summary reports",
        "Standalone ingestion runner generation",
        "Airflow DAG template generation",
        "Python syntax validation for runner and DAG",
        "Bronze data and metadata validation"
    ],
    "live_sources_ingested": [
        {
            "source_id": row["source_id"],
            "record_count": row["record_count"],
            "validation_status": row["validation_status"]
        }
        for row in validation_results
    ],
    "artifacts": {
        "source_register": str(
            REPORTS_PATH / "cell_15_data_source_register.csv"
        ),
        "reliability_audit": str(
            REPORTS_PATH / "cell_16_ingestion_reliability.csv"
        ),
        "api_connectivity_tests": str(
            REPORTS_PATH / "cell_17_api_connectivity_tests.csv"
        ),
        "bronze_ingestion_log": str(INGESTION_LOG),
        "bronze_ingestion_summary": str(INGESTION_SUMMARY),
        "ingestion_runner": str(RUNNER_PATH),
        "airflow_dag": str(DAG_PATH),
        "bronze_validation": str(VALIDATION_JSON)
    },
    "handoff_limitations": [
        "Airflow is not installed in the current notebook environment.",
        "The generated DAG has not been executed by an Airflow scheduler.",
        "Airflow import and deployment remain to be tested in the target environment.",
        "Only the three configured live API endpoints are included in this generated DAG.",
        "The RxNav version response is API metadata, not a drug records dataset.",
        "Other registered sources remain empty, unavailable or unverified as recorded in the source register."
    ],
    "next_owner_actions": [
        "Member 2: validate and transform available Bronze datasets into Silver.",
        "Member 2: implement data quality rules and schema validation.",
        "Team: deploy and test the generated DAG in an Airflow environment.",
        "Team: verify source coverage and data freshness before production claims."
    ],
    "overall_status": "HANDOFF_PREPARED_NOT_PRODUCTION_DEPLOYED"
}

with open(HANDOFF_JSON, "w", encoding="utf-8") as file:
    json.dump(handoff, file, ensure_ascii=False, indent=4)

# -----------------------------------------------------------------------------
# 9. DISPLAY RESULTS
# -----------------------------------------------------------------------------

passed = validation_report["checks_passed"]
review = validation_report["checks_requiring_review"]

print("\nBRONZE SOURCE VALIDATION")
print("-" * 90)
print(validation_df.to_string(index=False))

print("\nINGESTION LOG VALIDATION")
print("-" * 90)

if not log_df.empty:
    print(log_df.to_string(index=False))
else:
    print("No ingestion log entries found.")

print("\nVALIDATION CHECKS")
print("-" * 90)

for check in checks:
    print(
        f"{check['status']:6} | "
        f"{check['check_name']} | "
        f"{check['details']}"
    )

print("\nVALIDATION SUMMARY")
print("-" * 90)
print(f"Checks Passed       : {passed}")
print(f"Checks For Review   : {review}")
print(f"Airflow Installed   : {airflow_installed}")
print(f"DAG Executed        : False")
print("Handoff Status      : HANDOFF_PREPARED_NOT_PRODUCTION_DEPLOYED")

print("\nGENERATED FILES")
print("-" * 90)
print(f"Validation CSV : {VALIDATION_CSV}")
print(f"Validation JSON: {VALIDATION_JSON}")
print(f"Member 1 Handoff: {HANDOFF_JSON}")

print("\nCELL 20 EXECUTION COMPLETED")
print("=" * 90)

CELL 20 — BRONZE INGESTION VALIDATION & MEMBER 1 HANDOFF

BRONZE SOURCE VALIDATION
------------------------------------------------------------------------------------------
source_id     expected_prefix  file_count  valid_json  metadata_present  data_present  record_count  validation_status                                                               file_path
  SRC-001         openfda_ndc           1        True              True          True           100 VALID_WITH_RECORDS         /content/data/bronze/live_api/openfda_ndc_20261102T155044Z.json
  SRC-002 openfda_enforcement           1        True              True          True           100 VALID_WITH_RECORDS /content/data/bronze/live_api/openfda_enforcement_20261102T155044Z.json
  SRC-005       rxnav_version           1        True              True          True             1 VALID_WITH_RECORDS       /content/data/bronze/live_api/rxnav_version_20261102T155044Z.json

INGESTION LOG VALIDATION
------------------------------------

In [36]:
# =============================================================================
# CELL 21 — MEMBER 1 FINAL DELIVERABLE PACKAGE
# Pharma Supply Chain Intelligence
# =============================================================================

import json
import zipfile
import hashlib

from pathlib import Path
from datetime import datetime, timezone

print("=" * 90)
print("CELL 21 — MEMBER 1 FINAL DELIVERABLE PACKAGE")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. CONFIGURATION
# -----------------------------------------------------------------------------

BASE_PATH = Path(BASE_DIR)
REPORTS_PATH = BASE_PATH / "reports"
BRONZE_PATH = BASE_PATH / "data" / "bronze" / "live_api"

PACKAGE_DIR = BASE_PATH / "member_1_deliverables"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

PACKAGE_ZIP = BASE_PATH / "member_1_deliverables.zip"
MANIFEST_JSON = PACKAGE_DIR / "member_1_package_manifest.json"
README_PATH = PACKAGE_DIR / "README_MEMBER_1.txt"

# -----------------------------------------------------------------------------
# 2. DEFINE DELIVERABLES
# -----------------------------------------------------------------------------

deliverables = [
    REPORTS_PATH / "cell_15_data_source_register.csv",
    REPORTS_PATH / "cell_15_data_source_register.json",
    REPORTS_PATH / "cell_15_ingestion_inventory_audit.json",
    REPORTS_PATH / "cell_16_ingestion_reliability.csv",
    REPORTS_PATH / "cell_16_ingestion_reliability.json",
    REPORTS_PATH / "cell_16_retry_policy.json",
    REPORTS_PATH / "cell_16_reliability_summary.json",
    REPORTS_PATH / "cell_17_api_connectivity_tests.csv",
    REPORTS_PATH / "cell_17_api_connectivity_tests.json",
    REPORTS_PATH / "cell_17_api_connectivity_summary.json",
    REPORTS_PATH / "cell_18_bronze_ingestion_log.csv",
    REPORTS_PATH / "cell_18_bronze_ingestion_log.json",
    REPORTS_PATH / "cell_18_bronze_ingestion_summary.json",
    REPORTS_PATH / "pharma_ingestion_runner.py",
    REPORTS_PATH / "airflow_pharma_pipeline_dag.py",
    REPORTS_PATH / "cell_19_airflow_validation.json",
    REPORTS_PATH / "cell_20_bronze_validation.csv",
    REPORTS_PATH / "cell_20_bronze_validation.json",
    REPORTS_PATH / "member_1_ingestion_handoff.json"
]

bronze_files = (
    sorted(BRONZE_PATH.glob("*.json"))
    if BRONZE_PATH.exists()
    else []
)

# -----------------------------------------------------------------------------
# 3. COPY AVAILABLE REPORTS
# -----------------------------------------------------------------------------

copied_files = []
missing_files = []

for source_file in deliverables:

    if source_file.exists() and source_file.is_file():

        destination = PACKAGE_DIR / source_file.name

        destination.write_bytes(source_file.read_bytes())

        copied_files.append({
            "file_name": source_file.name,
            "source_path": str(source_file),
            "package_path": str(destination),
            "file_size_bytes": destination.stat().st_size,
            "sha256": hashlib.sha256(
                destination.read_bytes()
            ).hexdigest()
        })

    else:
        missing_files.append(str(source_file))

# -----------------------------------------------------------------------------
# 4. COPY BRONZE SNAPSHOTS
# -----------------------------------------------------------------------------

bronze_package_dir = PACKAGE_DIR / "bronze_live_api"
bronze_package_dir.mkdir(exist_ok=True)

copied_bronze = []

for source_file in bronze_files:

    destination = bronze_package_dir / source_file.name
    destination.write_bytes(source_file.read_bytes())

    copied_bronze.append({
        "file_name": source_file.name,
        "package_path": str(destination),
        "file_size_bytes": destination.stat().st_size,
        "sha256": hashlib.sha256(
            destination.read_bytes()
        ).hexdigest()
    })

# -----------------------------------------------------------------------------
# 5. CREATE README
# -----------------------------------------------------------------------------

readme_content = """
PHARMA SUPPLY CHAIN INTELLIGENCE
MEMBER 1 — DATA INGESTION & WORKFLOW ORCHESTRATION

PROJECT DELIVERABLES
--------------------

1. Data source register and ingestion inventory.
2. Ingestion reliability audit and retry policy.
3. Live API connectivity test reports.
4. Timestamped Bronze API ingestion snapshots.
5. Ingestion execution logs and summary.
6. Standalone Python ingestion runner.
7. Airflow DAG template.
8. Bronze validation reports.
9. Member 1 handoff report.

LIVE API SOURCES INCLUDED
-------------------------

SRC-001: openFDA Drug NDC
SRC-002: openFDA Drug Enforcement
SRC-005: NIH RxNav Version

DATA LINEAGE
------------

Bronze snapshots preserve:
- Source identifier and source name.
- Source endpoint.
- Ingestion timestamp in UTC.
- HTTP status.
- Record count.
- Request parameters.
- Ingestion method.

AIRFLOW STATUS
--------------

The Airflow DAG and Python runner were generated and passed
Python syntax validation.

Airflow was not installed in the notebook environment.
The DAG was not executed by an Airflow scheduler.
Deployment and runtime testing remain pending.

DATA LIMITATIONS
----------------

Only the configured live API sources are included in the
generated ingestion runner and DAG.

Empty, unavailable, or unverified sources are documented
in the source register. No synthetic records were added.

The RxNav version endpoint returns API metadata and is not
a drug-concept dataset.

HANDOFF
-------

Member 1 deliverables are packaged for team review and handoff.
This package does not represent a production deployment.
"""

README_PATH.write_text(
    readme_content.strip(),
    encoding="utf-8"
)

# -----------------------------------------------------------------------------
# 6. CREATE PACKAGE MANIFEST
# -----------------------------------------------------------------------------

manifest = {
    "project": "Pharma Supply Chain Intelligence",
    "team_role": "Member 1 — Data Ingestion & Workflow Orchestration",
    "package_created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "report_files_copied": len(copied_files),
    "bronze_files_copied": len(copied_bronze),
    "missing_report_files": missing_files,
    "report_files": copied_files,
    "bronze_files": copied_bronze,
    "airflow_execution": "NOT_EXECUTED",
    "production_deployment": "NOT_CONFIRMED",
    "package_status": (
        "PACKAGE_CREATED_WITH_MISSING_FILES"
        if missing_files
        else "PACKAGE_CREATED"
    )
}

with open(MANIFEST_JSON, "w", encoding="utf-8") as file:
    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=4
    )

# -----------------------------------------------------------------------------
# 7. CREATE ZIP PACKAGE
# -----------------------------------------------------------------------------

if PACKAGE_ZIP.exists():
    PACKAGE_ZIP.unlink()

with zipfile.ZipFile(
    PACKAGE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for file_path in PACKAGE_DIR.rglob("*"):

        if file_path.is_file():
            archive.write(
                file_path,
                arcname=str(
                    Path("member_1_deliverables") /
                    file_path.relative_to(PACKAGE_DIR)
                )
            )

# -----------------------------------------------------------------------------
# 8. VERIFY ZIP PACKAGE
# -----------------------------------------------------------------------------

with zipfile.ZipFile(PACKAGE_ZIP, "r") as archive:

    zip_test = archive.testzip()
    archive_file_count = len(archive.namelist())

package_valid = (
    zip_test is None
    and PACKAGE_ZIP.exists()
    and PACKAGE_ZIP.stat().st_size > 0
)

# -----------------------------------------------------------------------------
# 9. FINAL SUMMARY
# -----------------------------------------------------------------------------

print("\nPACKAGE CONTENTS")
print("-" * 90)

print(f"Report Files Copied : {len(copied_files)}")
print(f"Bronze Files Copied : {len(copied_bronze)}")
print(f"Missing Reports     : {len(missing_files)}")

if missing_files:
    print("\nMISSING FILES")
    for missing in missing_files:
        print(f"- {missing}")

print("\nPACKAGE VERIFICATION")
print("-" * 90)
print(f"ZIP File Valid      : {package_valid}")
print(f"Files In ZIP        : {archive_file_count}")
print(f"ZIP Size (MB)       : {round(PACKAGE_ZIP.stat().st_size / (1024 * 1024), 3)}")
print(f"Manifest            : {MANIFEST_JSON}")
print(f"README              : {README_PATH}")

print("\nFINAL PACKAGE")
print("-" * 90)
print(f"ZIP Path : {PACKAGE_ZIP}")

print("\nCELL 21 EXECUTION COMPLETED")
print("=" * 90)

CELL 21 — MEMBER 1 FINAL DELIVERABLE PACKAGE

PACKAGE CONTENTS
------------------------------------------------------------------------------------------
Report Files Copied : 19
Bronze Files Copied : 3
Missing Reports     : 0

PACKAGE VERIFICATION
------------------------------------------------------------------------------------------
ZIP File Valid      : True
Files In ZIP        : 24
ZIP Size (MB)       : 0.072
Manifest            : /content/member_1_deliverables/member_1_package_manifest.json
README              : /content/member_1_deliverables/README_MEMBER_1.txt

FINAL PACKAGE
------------------------------------------------------------------------------------------
ZIP Path : /content/member_1_deliverables.zip

CELL 21 EXECUTION COMPLETED


In [37]:
# =============================================================================
# CELL 22 — MEMBER 1 FINAL READINESS & HANDOFF REPORT
# Pharma Supply Chain Intelligence
# =============================================================================

import json
import zipfile
import hashlib

from pathlib import Path
from datetime import datetime, timezone

print("=" * 90)
print("CELL 22 — MEMBER 1 FINAL READINESS & HANDOFF REPORT")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. PATHS
# -----------------------------------------------------------------------------

BASE_PATH = Path(BASE_DIR)
REPORTS_PATH = BASE_PATH / "reports"
PACKAGE_DIR = BASE_PATH / "member_1_deliverables"
BRONZE_PATH = BASE_PATH / "data" / "bronze" / "live_api"

PACKAGE_ZIP = BASE_PATH / "member_1_deliverables.zip"

HANDOFF_REPORT = REPORTS_PATH / "cell_22_member_1_final_readiness.json"
HANDOFF_TEXT = REPORTS_PATH / "member_1_final_handoff.txt"

REPORTS_PATH.mkdir(parents=True, exist_ok=True)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. LOAD VALIDATION RESULTS
# -----------------------------------------------------------------------------

def load_json(path):
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)

cell20 = load_json(
    REPORTS_PATH / "cell_20_bronze_validation.json"
)

cell19 = load_json(
    REPORTS_PATH / "cell_19_airflow_validation.json"
)

cell18 = load_json(
    REPORTS_PATH / "cell_18_bronze_ingestion_summary.json"
)

cell15 = load_json(
    REPORTS_PATH / "cell_15_data_source_register.json"
)

# -----------------------------------------------------------------------------
# 3. VALIDATE CORE DELIVERABLES
# -----------------------------------------------------------------------------

required_artifacts = {
    "source_register": REPORTS_PATH / "cell_15_data_source_register.csv",
    "reliability_audit": REPORTS_PATH / "cell_16_ingestion_reliability.csv",
    "connectivity_tests": REPORTS_PATH / "cell_17_api_connectivity_tests.csv",
    "bronze_ingestion_log": REPORTS_PATH / "cell_18_bronze_ingestion_log.json",
    "ingestion_runner": REPORTS_PATH / "pharma_ingestion_runner.py",
    "airflow_dag": REPORTS_PATH / "airflow_pharma_pipeline_dag.py",
    "bronze_validation": REPORTS_PATH / "cell_20_bronze_validation.json",
    "member_handoff": REPORTS_PATH / "member_1_ingestion_handoff.json"
}

artifact_checks = {
    name: path.exists() and path.is_file()
    for name, path in required_artifacts.items()
}

all_artifacts_present = all(artifact_checks.values())

# -----------------------------------------------------------------------------
# 4. SUMMARIZE BRONZE DATA
# -----------------------------------------------------------------------------

bronze_files = (
    list(BRONZE_PATH.glob("*.json"))
    if BRONZE_PATH.exists()
    else []
)

bronze_record_count = sum(
    row.get("record_count", 0)
    for row in cell20.get("bronze_source_validation", [])
)

validated_sources = sum(
    1 for row in cell20.get("bronze_source_validation", [])
    if row.get("validation_status") == "VALID_WITH_RECORDS"
)

# -----------------------------------------------------------------------------
# 5. EVALUATE AIRFLOW STATUS
# -----------------------------------------------------------------------------

airflow_generated = (
    artifact_checks["ingestion_runner"]
    and artifact_checks["airflow_dag"]
    and cell19.get("runner_syntax_valid", False)
    and cell19.get("dag_syntax_valid", False)
)

airflow_executed = False

# -----------------------------------------------------------------------------
# 6. FINAL STATUS
# -----------------------------------------------------------------------------

validation_checks = cell20.get("checks", [])

passed_checks = sum(
    1 for check in validation_checks
    if check.get("status") == "PASS"
)

review_checks = sum(
    1 for check in validation_checks
    if check.get("status") != "PASS"
)

final_status = (
    "MEMBER_1_HANDOFF_READY_WITH_AIRFLOW_DEPLOYMENT_PENDING"
    if all_artifacts_present
    and passed_checks == len(validation_checks)
    and airflow_generated
    else "MEMBER_1_HANDOFF_REQUIRES_REVIEW"
)

# -----------------------------------------------------------------------------
# 7. CREATE FINAL HANDOFF REPORT
# -----------------------------------------------------------------------------

final_report = {
    "project": "Pharma Supply Chain Intelligence",
    "role": "Member 1 — Data Ingestion & Workflow Orchestration",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "final_status": final_status,
    "source_registry": {
        "registered_sources": 9,
        "live_api_sources_ingested": 3,
        "sources_with_validated_bronze_data": validated_sources
    },
    "bronze_layer": {
        "bronze_json_files": len(bronze_files),
        "validated_record_count": bronze_record_count,
        "bronze_directory": str(BRONZE_PATH)
    },
    "validation": {
        "checks_passed": passed_checks,
        "checks_requiring_review": review_checks,
        "all_required_artifacts_present": all_artifacts_present,
        "artifact_checks": artifact_checks
    },
    "orchestration": {
        "ingestion_runner_generated": artifact_checks["ingestion_runner"],
        "airflow_dag_generated": artifact_checks["airflow_dag"],
        "python_syntax_validated": airflow_generated,
        "airflow_installed": cell19.get(
            "airflow_installed_in_current_environment",
            False
        ),
        "airflow_dag_executed": airflow_executed,
        "deployment_status": "PENDING_TARGET_AIRFLOW_ENVIRONMENT"
    },
    "known_limitations": [
        "Airflow was not installed or executed in the notebook environment.",
        "The generated DAG requires deployment and runtime testing.",
        "Only three live API endpoints are configured in the generated runner.",
        "Other registered sources remain empty, unavailable or unverified.",
        "The RxNav version endpoint provides API metadata, not drug records.",
        "The available Bronze records are API samples, not proof of complete source coverage."
    ],
    "next_steps_for_team": [
        "Member 2: consume validated Bronze snapshots and build Silver transformations.",
        "Member 2: implement schema, completeness and data quality checks.",
        "Team: deploy and test the DAG in the target Airflow environment.",
        "Team: validate source coverage, freshness and downstream dependencies."
    ]
}

with open(HANDOFF_REPORT, "w", encoding="utf-8") as file:
    json.dump(final_report, file, ensure_ascii=False, indent=4)

# -----------------------------------------------------------------------------
# 8. CREATE HUMAN-READABLE HANDOFF
# -----------------------------------------------------------------------------

handoff_text = f"""
PHARMA SUPPLY CHAIN INTELLIGENCE
MEMBER 1 — FINAL HANDOFF

FINAL STATUS
------------
{final_status}

DELIVERABLES
------------
Source register:                  {'AVAILABLE' if artifact_checks['source_register'] else 'MISSING'}
Reliability audit:                {'AVAILABLE' if artifact_checks['reliability_audit'] else 'MISSING'}
API connectivity tests:           {'AVAILABLE' if artifact_checks['connectivity_tests'] else 'MISSING'}
Bronze ingestion logs:            {'AVAILABLE' if artifact_checks['bronze_ingestion_log'] else 'MISSING'}
Python ingestion runner:          {'AVAILABLE' if artifact_checks['ingestion_runner'] else 'MISSING'}
Airflow DAG template:             {'AVAILABLE' if artifact_checks['airflow_dag'] else 'MISSING'}
Bronze validation report:         {'AVAILABLE' if artifact_checks['bronze_validation'] else 'MISSING'}

BRONZE SUMMARY
--------------
Bronze JSON files: {len(bronze_files)}
Validated records: {bronze_record_count}
Sources with validated records: {validated_sources}

QUALITY VALIDATION
------------------
Checks passed: {passed_checks}
Checks requiring review: {review_checks}

AIRFLOW
-------
DAG generated: {airflow_generated}
DAG executed: {airflow_executed}
Deployment: PENDING TARGET AIRFLOW ENVIRONMENT

HANDOFF NOTES
-------------
Member 1 has prepared the ingestion artifacts and Bronze snapshots.
The generated Airflow DAG has not been executed.
The team must deploy and test orchestration in its target environment.

Member 2 can proceed with the available validated Bronze snapshots,
subject to downstream schema and data quality validation.
"""

HANDOFF_TEXT.write_text(
    handoff_text.strip(),
    encoding="utf-8"
)

# -----------------------------------------------------------------------------
# 9. UPDATE PACKAGE AND ZIP
# -----------------------------------------------------------------------------

for source_file in [HANDOFF_REPORT, HANDOFF_TEXT]:

    destination = PACKAGE_DIR / source_file.name
    destination.write_bytes(source_file.read_bytes())

if PACKAGE_ZIP.exists():
    PACKAGE_ZIP.unlink()

with zipfile.ZipFile(
    PACKAGE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for file_path in PACKAGE_DIR.rglob("*"):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=str(
                    Path("member_1_deliverables") /
                    file_path.relative_to(PACKAGE_DIR)
                )
            )

with zipfile.ZipFile(PACKAGE_ZIP, "r") as archive:
    zip_test = archive.testzip()
    zip_file_count = len(archive.namelist())

package_valid = (
    zip_test is None
    and PACKAGE_ZIP.exists()
    and PACKAGE_ZIP.stat().st_size > 0
)

# -----------------------------------------------------------------------------
# 10. DISPLAY FINAL RESULTS
# -----------------------------------------------------------------------------

print("\nFINAL READINESS")
print("-" * 90)
print(f"Final Status              : {final_status}")
print(f"Artifacts Present         : {all_artifacts_present}")
print(f"Validation Checks Passed  : {passed_checks}")
print(f"Checks Requiring Review   : {review_checks}")
print(f"Validated Bronze Records  : {bronze_record_count}")
print(f"Bronze JSON Files         : {len(bronze_files)}")

print("\nAIRFLOW STATUS")
print("-" * 90)
print(f"Runner and DAG Generated  : {airflow_generated}")
print(f"Airflow Executed          : {airflow_executed}")
print("Deployment                : PENDING")

print("\nFINAL PACKAGE")
print("-" * 90)
print(f"ZIP Valid                 : {package_valid}")
print(f"Files In ZIP              : {zip_file_count}")
print(f"Package Path              : {PACKAGE_ZIP}")
print(f"JSON Handoff              : {HANDOFF_REPORT}")
print(f"Text Handoff              : {HANDOFF_TEXT}")

print("\nCELL 22 EXECUTION COMPLETED")
print("=" * 90)

CELL 22 — MEMBER 1 FINAL READINESS & HANDOFF REPORT

FINAL READINESS
------------------------------------------------------------------------------------------
Final Status              : MEMBER_1_HANDOFF_READY_WITH_AIRFLOW_DEPLOYMENT_PENDING
Artifacts Present         : True
Validation Checks Passed  : 11
Checks Requiring Review   : 0
Validated Bronze Records  : 201
Bronze JSON Files         : 3

AIRFLOW STATUS
------------------------------------------------------------------------------------------
Runner and DAG Generated  : True
Airflow Executed          : False
Deployment                : PENDING

FINAL PACKAGE
------------------------------------------------------------------------------------------
ZIP Valid                 : True
Files In ZIP              : 26
Package Path              : /content/member_1_deliverables.zip
JSON Handoff              : /content/reports/cell_22_member_1_final_readiness.json
Text Handoff              : /content/reports/member_1_final_handoff.txt

CE

In [38]:
# من هنا إضافات من عندي

In [39]:
import kagglehub
import shutil
import os

# 1. تحديد مسار مجلد الـ Bronze الخاص بمشروعك
bronze_kaggle_dir = "/content/data/bronze/kaggle"
os.makedirs(bronze_kaggle_dir, exist_ok=True)

# 2. تحميل البيانات من Kaggle
download_path = kagglehub.dataset_download("rushil09/1mg-medicine-data")

# 3. نقل الملفات المحملة إلى مجلد الـ Bronze
for filename in os.listdir(download_path):
    src_file = os.path.join(download_path, filename)
    dst_file = os.path.join(bronze_kaggle_dir, filename)
    if os.path.isfile(src_file):
        shutil.copy(src_file, dst_file)

print(f"✅ تم تحميل ونقل الملفات بنجاح إلى: {bronze_kaggle_dir}")

Using Colab cache for faster access to the '1mg-medicine-data' dataset.
✅ تم تحميل ونقل الملفات بنجاح إلى: /content/data/bronze/kaggle


In [40]:
import kagglehub
import shutil
import os

# 1. تحديد مسار مجلد الـ Bronze الخاص بـ Kaggle
bronze_kaggle_dir = "/content/data/bronze/kaggle"
os.makedirs(bronze_kaggle_dir, exist_ok=True)

# 2. تحميل البيانات الثانية من Kaggle
download_path = kagglehub.dataset_download("singhnavjot2062001/11000-medicine-details")

# 3. نقل الملفات المحملة إلى مجلد الـ Bronze
for filename in os.listdir(download_path):
    src_file = os.path.join(download_path, filename)
    dst_file = os.path.join(bronze_kaggle_dir, filename)
    if os.path.isfile(src_file):
        shutil.copy(src_file, dst_file)

print(f"✅ تم تحميل ونقل مجموعة البيانات الثانية بنجاح إلى: {bronze_kaggle_dir}")

Using Colab cache for faster access to the '11000-medicine-details' dataset.
✅ تم تحميل ونقل مجموعة البيانات الثانية بنجاح إلى: /content/data/bronze/kaggle


In [41]:
import os
import gdown

# 1. إنشاء مجلد الـ Bronze الخاص بـ WHO
who_dir = "/content/data/bronze/who"
os.makedirs(who_dir, exist_ok=True)

# 2. تنزيل الملف باستخدام ID الخاص بك
file_id = '1wFCQVGyfxvnOQ811IksZCZ1kleZ4Uv6-'
url = f'https://drive.google.com/uc?id={file_id}'
output_path = os.path.join(who_dir, "WHO_EML_2023.pdf")

gdown.download(url, output_path, quiet=False)

print(f"\n✅ تم تنزيل كتاب WHO EML بنجاح وحفظه في: {output_path}")

Downloading...
From: https://drive.google.com/uc?id=1wFCQVGyfxvnOQ811IksZCZ1kleZ4Uv6-
To: /content/data/bronze/who/WHO_EML_2023.pdf
100%|██████████| 927k/927k [00:00<00:00, 15.9MB/s]


✅ تم تنزيل كتاب WHO EML بنجاح وحفظه في: /content/data/bronze/who/WHO_EML_2023.pdf


In [42]:
import os

folders = {
    "FDA Bronze": "/content/data/bronze/fda",
    "Kaggle Bronze": "/content/data/bronze/kaggle",
    "WHO Bronze": "/content/data/bronze/who",
    "EDA Bronze": "/content/data/bronze/eda"
}

print("=== فحص ملفات طبقة الـ Bronze ===")
for label, path in folders.items():
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"\n📂 {label} ({path}):")
        if not files:
            print("  ⚠️ المجلد فارغ!")
        for f in files:
            file_path = os.path.join(path, f)
            if os.path.isfile(file_path):
                size_kb = os.path.getsize(file_path) / 1024
                print(f"  - {f}: {size_kb:.2f} KB")
    else:
        print(f"\n❌ المجلد غير موجود: {path}")

=== فحص ملفات طبقة الـ Bronze ===

❌ المجلد غير موجود: /content/data/bronze/fda

📂 Kaggle Bronze (/content/data/bronze/kaggle):
  - kaggle_medicines_raw.json: 0.31 KB
  - medicine_data.csv: 864.93 KB
  - Medicine_Details.csv: 4269.59 KB

📂 WHO Bronze (/content/data/bronze/who):
  - who_eml_raw.json: 0.33 KB
  - WHO_EML_2023.pdf: 904.93 KB

📂 EDA Bronze (/content/data/bronze/eda):
  - eda_alerts_raw.json: 0.28 KB


In [43]:
import os
import requests
import zipfile

# 1. إنشاء مجلد FDA Bronze
fda_dir = "/content/data/bronze/fda"
os.makedirs(fda_dir, exist_ok=True)

# 2. تنزيل بيانات FDA NDC المباشرة
fda_ndc_url = "https://download.open.fda.gov/drug/ndc/drug-ndc-0001-of-0001.json.zip"
zip_path = os.path.join(fda_dir, "fda_ndc.zip")

print("⏳ جاري تنزيل بيانات FDA NDC...")
response = requests.get(fda_ndc_url, stream=True)
if response.status_code == 200:
    with open(zip_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024):
            f.write(chunk)

    # فك الضغط داخل المجلد
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(fda_dir)
    os.remove(zip_path) # حذف ملف الـ zip المؤقت
    print(f"✅ تم تنزيل واستخراج بيانات FDA بنجاح داخل: {fda_dir}")
else:
    print(f"❌ فشل تنزيل ملفات FDA، كود الاستجابة: {response.status_code}")

⏳ جاري تنزيل بيانات FDA NDC...
✅ تم تنزيل واستخراج بيانات FDA بنجاح داخل: /content/data/bronze/fda


In [44]:
import os

folders = {
    "FDA Bronze": "/content/data/bronze/fda",
    "Kaggle Bronze": "/content/data/bronze/kaggle",
    "WHO Bronze": "/content/data/bronze/who",
    "EDA Bronze": "/content/data/bronze/eda"
}

print("=== فحص ملفات طبقة الـ Bronze ===")
for label, path in folders.items():
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"\n📂 {label} ({path}):")
        if not files:
            print("  ⚠️ المجلد فارغ!")
        for f in files:
            file_path = os.path.join(path, f)
            if os.path.isfile(file_path):
                size_kb = os.path.getsize(file_path) / 1024
                print(f"  - {f}: {size_kb:.2f} KB")
    else:
        print(f"\n❌ المجلد غير موجود: {path}")

=== فحص ملفات طبقة الـ Bronze ===

📂 FDA Bronze (/content/data/bronze/fda):
  - drug-ndc-0001-of-0001.json: 237634.10 KB

📂 Kaggle Bronze (/content/data/bronze/kaggle):
  - kaggle_medicines_raw.json: 0.31 KB
  - medicine_data.csv: 864.93 KB
  - Medicine_Details.csv: 4269.59 KB

📂 WHO Bronze (/content/data/bronze/who):
  - who_eml_raw.json: 0.33 KB
  - WHO_EML_2023.pdf: 904.93 KB

📂 EDA Bronze (/content/data/bronze/eda):
  - eda_alerts_raw.json: 0.28 KB


In [45]:
# =============================================================================
# CELL 54 — PROCESS NEW KAGGLE & WHO FILES
# =============================================================================

import os
import json
import glob
import hashlib
import pandas as pd
from datetime import datetime, timezone

print("=" * 85)
print("CELL 54 — PROCESSING DOWNLOADED KAGGLE AND WHO FILES")
print("=" * 85)

BASE_DIR = "/content" if os.path.exists("/content") else os.getcwd()

BRONZE_KAGGLE = os.path.join(BASE_DIR, "data", "bronze", "kaggle")
BRONZE_WHO = os.path.join(BASE_DIR, "data", "bronze", "who")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")

os.makedirs(BRONZE_KAGGLE, exist_ok=True)
os.makedirs(BRONZE_WHO, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# 1. FIND AND READ KAGGLE CSV FILES
# -----------------------------------------------------------------------------

kaggle_candidates = (
    glob.glob(os.path.join(BRONZE_KAGGLE, "**", "*.csv"), recursive=True)
    + glob.glob(os.path.join(BASE_DIR, "**", "*1mg*.csv"), recursive=True)
    + glob.glob(os.path.join(BASE_DIR, "**", "*11000*.csv"), recursive=True)
)

kaggle_files = sorted(set(
    path for path in kaggle_candidates
    if os.path.isfile(path)
    and "silver" not in path.lower()
    and "reports" not in path.lower()
    and "checkpoint" not in path.lower()
))

kaggle_summary = []

for file_path in kaggle_files:
    try:
        df = pd.read_csv(file_path, low_memory=False)

        file_hash = hashlib.sha256(
            open(file_path, "rb").read()
        ).hexdigest()

        kaggle_summary.append({
            "file_name": os.path.basename(file_path),
            "file_path": file_path,
            "file_size_bytes": os.path.getsize(file_path),
            "sha256": file_hash,
            "rows": int(len(df)),
            "columns_count": int(len(df.columns)),
            "columns": [str(c) for c in df.columns],
            "duplicate_rows": int(df.duplicated().sum()),
            "empty_rows": int(df.isna().all(axis=1).sum()),
            "status": "READ_SUCCESSFULLY" if not df.empty else "EMPTY_FILE"
        })

        print(f"\nKaggle file: {os.path.basename(file_path)}")
        print(f"Rows: {len(df):,}")
        print(f"Columns: {len(df.columns)}")
        print(f"Duplicate rows: {df.duplicated().sum():,}")
        print("Column names:", list(df.columns))

    except Exception as e:
        kaggle_summary.append({
            "file_name": os.path.basename(file_path),
            "file_path": file_path,
            "status": "READ_FAILED",
            "error": str(e)
        })
        print(f"\nERROR reading {file_path}: {e}")

# -----------------------------------------------------------------------------
# 2. FIND AND EXTRACT WHO PDF
# -----------------------------------------------------------------------------

who_candidates = (
    glob.glob(os.path.join(BRONZE_WHO, "**", "*.pdf"), recursive=True)
    + glob.glob(os.path.join(BASE_DIR, "**", "WHO_EML*.pdf"), recursive=True)
)

who_files = sorted(set(
    path for path in who_candidates
    if os.path.isfile(path)
    and "checkpoint" not in path.lower()
))

who_summary = []

for pdf_path in who_files:
    pdf_result = {
        "file_name": os.path.basename(pdf_path),
        "file_path": pdf_path,
        "file_size_bytes": os.path.getsize(pdf_path),
        "extraction_status": "NOT_ATTEMPTED",
        "pages": 0,
        "extracted_text_characters": 0
    }

    try:
        pdf_reader = None

        try:
            from pypdf import PdfReader
            pdf_reader = PdfReader(pdf_path)
        except ImportError:
            try:
                from PyPDF2 import PdfReader
                pdf_reader = PdfReader(pdf_path)
            except ImportError:
                pdf_result["extraction_status"] = "PDF_LIBRARY_NOT_AVAILABLE"

        if pdf_reader is not None:
            pages = []
            for page_number, page in enumerate(pdf_reader.pages, start=1):
                page_text = page.extract_text() or ""
                pages.append({
                    "page_number": page_number,
                    "text": page_text
                })

            extracted_text = "\n".join(
                page["text"] for page in pages
            )

            text_path = os.path.join(
                BRONZE_WHO,
                os.path.splitext(os.path.basename(pdf_path))[0] + "_extracted.txt"
            )

            with open(text_path, "w", encoding="utf-8") as f:
                f.write(extracted_text)

            structured_path = os.path.join(
                BRONZE_WHO,
                os.path.splitext(os.path.basename(pdf_path))[0] + "_pages.json"
            )

            with open(structured_path, "w", encoding="utf-8") as f:
                json.dump({
                    "source_file": os.path.basename(pdf_path),
                    "source_type": "WHO_OFFICIAL_PDF",
                    "extraction_timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    "page_count": len(pages),
                    "pages": pages
                }, f, ensure_ascii=False, indent=2)

            pdf_result.update({
                "extraction_status": (
                    "TEXT_EXTRACTED"
                    if extracted_text.strip()
                    else "NO_EXTRACTABLE_TEXT"
                ),
                "pages": len(pages),
                "extracted_text_characters": len(extracted_text),
                "text_output": text_path,
                "json_output": structured_path
            })

        who_summary.append(pdf_result)

        print(f"\nWHO PDF: {os.path.basename(pdf_path)}")
        print("Extraction:", pdf_result["extraction_status"])
        print("Pages:", pdf_result["pages"])
        print("Extracted characters:", pdf_result["extracted_text_characters"])

    except Exception as e:
        pdf_result["extraction_status"] = "EXTRACTION_FAILED"
        pdf_result["error"] = str(e)
        who_summary.append(pdf_result)
        print(f"\nERROR extracting WHO PDF: {e}")

# -----------------------------------------------------------------------------
# 3. SAVE PROCESSING REPORT
# -----------------------------------------------------------------------------

summary = {
    "execution_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "kaggle_files_found": len(kaggle_files),
    "kaggle_files_read_successfully": sum(
        x.get("status") == "READ_SUCCESSFULLY" for x in kaggle_summary
    ),
    "kaggle_total_rows": sum(
        x.get("rows", 0) for x in kaggle_summary
    ),
    "who_pdf_files_found": len(who_files),
    "who_pdfs_with_extracted_text": sum(
        x.get("extraction_status") == "TEXT_EXTRACTED"
        for x in who_summary
    ),
    "kaggle_details": kaggle_summary,
    "who_details": who_summary
}

report_path = os.path.join(
    REPORTS_DIR,
    "cell_54_kaggle_who_processing_report.json"
)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 85)
print("CELL 54 SUMMARY")
print("=" * 85)
print("Kaggle files found:", summary["kaggle_files_found"])
print("Kaggle files read successfully:", summary["kaggle_files_read_successfully"])
print("Total Kaggle rows:", f'{summary["kaggle_total_rows"]:,}')
print("WHO PDF files found:", summary["who_pdf_files_found"])
print("WHO PDFs with extracted text:", summary["who_pdfs_with_extracted_text"])
print("Report:", report_path)
print("CELL 54 EXECUTION COMPLETED")
print("=" * 85)

CELL 54 — PROCESSING DOWNLOADED KAGGLE AND WHO FILES

Kaggle file: Medicine_Details.csv
Rows: 11,825
Columns: 9
Duplicate rows: 84
Column names: ['Medicine Name', 'Composition', 'Uses', 'Side_effects', 'Image URL', 'Manufacturer', 'Excellent Review %', 'Average Review %', 'Poor Review %']

Kaggle file: medicine_data.csv
Rows: 10,320
Columns: 5
Duplicate rows: 2,169
Column names: ['Name', 'Price', 'Prescription Required', 'Company', 'Salt']

WHO PDF: WHO_EML_2023.pdf
Extraction: PDF_LIBRARY_NOT_AVAILABLE
Pages: 0
Extracted characters: 0

CELL 54 SUMMARY
Kaggle files found: 2
Kaggle files read successfully: 2
Total Kaggle rows: 22,145
WHO PDF files found: 1
WHO PDFs with extracted text: 0
Report: /content/reports/cell_54_kaggle_who_processing_report.json
CELL 54 EXECUTION COMPLETED


In [46]:
# =============================================================================
# CELL 55 — INSTALL PDF READER
# =============================================================================

import subprocess
import sys

print("Installing PDF reading library...")

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pypdf"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("PDF READER INSTALLED SUCCESSFULLY")
else:
    print("INSTALLATION FAILED")
    print(result.stderr[-2000:])

try:
    import pypdf
    print("pypdf version:", pypdf.__version__)
    print("CELL 55 COMPLETED SUCCESSFULLY")
except Exception as e:
    print("PDF READER VERIFICATION FAILED:", str(e))

Installing PDF reading library...
PDF READER INSTALLED SUCCESSFULLY
pypdf version: 6.21.4
CELL 55 COMPLETED SUCCESSFULLY


In [47]:
# =============================================================================
# CELL 56 — EXTRACT WHO PDF TEXT
# =============================================================================

import os
import glob
import json
from pypdf import PdfReader

who_files = glob.glob(
    "/content/**/*WHO_EML*.pdf",
    recursive=True
)

if not who_files:
    print("WHO PDF NOT FOUND")
else:
    for pdf_path in who_files:
        print("Processing:", os.path.basename(pdf_path))

        reader = PdfReader(pdf_path)
        pages_text = []

        for page in reader.pages:
            pages_text.append(page.extract_text() or "")

        full_text = "\n".join(pages_text)

        output_dir = "/content/data/bronze/who"
        os.makedirs(output_dir, exist_ok=True)

        text_path = os.path.join(
            output_dir, "WHO_EML_2023_extracted.txt"
        )

        json_path = os.path.join(
            output_dir, "WHO_EML_2023_pages.json"
        )

        with open(text_path, "w", encoding="utf-8") as f:
            f.write(full_text)

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "source_file": os.path.basename(pdf_path),
                    "pages_count": len(reader.pages),
                    "extracted_characters": len(full_text),
                    "pages": pages_text
                },
                f,
                ensure_ascii=False,
                indent=2
            )

        print("Pages:", len(reader.pages))
        print("Extracted characters:", len(full_text))
        print("Text saved:", text_path)
        print("JSON saved:", json_path)

    print("CELL 56 COMPLETED")

Processing: WHO_EML_2023.pdf
Pages: 71
Extracted characters: 163146
Text saved: /content/data/bronze/who/WHO_EML_2023_extracted.txt
JSON saved: /content/data/bronze/who/WHO_EML_2023_pages.json
CELL 56 COMPLETED


In [48]:
# =============================================================================
# CELL 57 — VALIDATE NEW BRONZE DATA
# =============================================================================

import os
import json
import pandas as pd

bronze_dir = "/content/data/bronze"
report_path = "/content/reports/cell_54_kaggle_who_processing_report.json"

print("=" * 70)
print("CELL 57 — BRONZE DATA VALIDATION")
print("=" * 70)

# Kaggle validation
with open(report_path, "r", encoding="utf-8") as f:
    kaggle_who_report = json.load(f)

for root, dirs, files in os.walk(bronze_dir):
    for file in files:
        if file.lower().endswith((".csv", ".pdf", ".txt", ".json")):
            path = os.path.join(root, file)
            print(f"FOUND: {path} | {os.path.getsize(path):,} bytes")

# WHO validation
who_text = os.path.join(
    bronze_dir, "who", "WHO_EML_2023_extracted.txt"
)

who_json = os.path.join(
    bronze_dir, "who", "WHO_EML_2023_pages.json"
)

assert os.path.exists(who_text), "WHO text file missing"
assert os.path.exists(who_json), "WHO JSON file missing"

with open(who_text, "r", encoding="utf-8") as f:
    extracted_text = f.read()

with open(who_json, "r", encoding="utf-8") as f:
    extracted_pages = json.load(f)

assert len(extracted_text) > 0, "WHO extracted text is empty"
assert len(extracted_pages["pages"]) == 71, "WHO page count mismatch"

print("\nWHO validation: PASSED")
print("WHO pages:", len(extracted_pages["pages"]))
print("WHO characters:", len(extracted_text))

print("\nKaggle files successfully processed:")
print("Medicine_Details.csv: 11,825 rows")
print("medicine_data.csv: 10,320 rows")

print("\nCELL 57 VALIDATION COMPLETED")

CELL 57 — BRONZE DATA VALIDATION
FOUND: /content/data/bronze/fda/drug-ndc-0001-of-0001.json | 243,337,316 bytes
FOUND: /content/data/bronze/rxnav/rxnav_concepts_raw.json | 7,423 bytes
FOUND: /content/data/bronze/who/who_eml_raw.json | 336 bytes
FOUND: /content/data/bronze/who/WHO_EML_2023.pdf | 926,649 bytes
FOUND: /content/data/bronze/who/WHO_EML_2023_extracted.txt | 164,557 bytes
FOUND: /content/data/bronze/who/WHO_EML_2023_pages.json | 168,925 bytes
FOUND: /content/data/bronze/rag_corpus/drugs_com_pricing_raw.json | 267 bytes
FOUND: /content/data/bronze/rag_corpus/fda_safety_corpus.json | 82,659 bytes
FOUND: /content/data/bronze/eda/eda_alerts_raw.json | 288 bytes
FOUND: /content/data/bronze/live_api/openfda_enforcement_20261102T155044Z.json | 167,320 bytes
FOUND: /content/data/bronze/live_api/openfda_ndc_20261102T155044Z.json | 149,479 bytes
FOUND: /content/data/bronze/live_api/rxnav_version_20261102T155044Z.json | 483 bytes
FOUND: /content/data/bronze/kaggle/kaggle_medicines_raw.j

In [49]:
# =============================================================================
# CELL 58 — INSPECT RAW JSON STRUCTURE
# =============================================================================

import os
import json

files_to_inspect = [
    "/content/data/bronze/kaggle/kaggle_medicines_raw.json",
    "/content/data/bronze/who/who_eml_raw.json",
    "/content/data/bronze/who/WHO_EML_2023_pages.json"
]

for path in files_to_inspect:
    print("\n" + "=" * 70)
    print("FILE:", os.path.basename(path))
    print("=" * 70)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Top-level type:", type(data).__name__)

    if isinstance(data, dict):
        print("Keys:", list(data.keys()))

    print("Sample:")
    print(json.dumps(data, ensure_ascii=False)[:1500])

print("\nCELL 58 COMPLETED")


FILE: kaggle_medicines_raw.json
Top-level type: dict
Keys: ['metadata', 'results']
Sample:
{"metadata": {"ingestion_timestamp_utc": "2026-11-02T15:49:15.740839+00:00", "sources": ["Kaggle 1mg Medicine Data", "Kaggle 11,000+ Medicine Details"], "discovered_files_count": 0, "total_records_ingested": 0, "execution_status": "NO_KAGGLE_FILES_FOUND"}, "results": []}

FILE: who_eml_raw.json
Top-level type: dict
Keys: ['metadata', 'results']
Sample:
{"metadata": {"source": "WHO Essential Medicines List", "verification_status": "NOT_VERIFIED", "source_file": null, "ingestion_timestamp_utc": "2026-11-02T15:49:15.946798+00:00", "total_records": 0, "has_synthetic_data": false, "execution_status": "WAITING_FOR_VERIFIED_WHO_SOURCE"}, "results": []}

FILE: WHO_EML_2023_pages.json
Top-level type: dict
Keys: ['source_file', 'pages_count', 'extracted_characters', 'pages']
Sample:
{"source_file": "WHO_EML_2023.pdf", "pages_count": 71, "extracted_characters": 163146, "pages": [" \n \n \n \n \n \n \n \n \

In [50]:
# =============================================================================
# CELL 59 — UPDATE KAGGLE AND WHO RAW BRONZE FILES
# =============================================================================

import os
import json
import pandas as pd
from datetime import datetime, timezone

BASE = "/content/data/bronze"

# -------------------------------------------------------------------------
# 1. LOAD KAGGLE CSV FILES
# -------------------------------------------------------------------------

kaggle_files = {
    "Medicine_Details.csv": "Kaggle 11,000+ Medicine Details",
    "medicine_data.csv": "Kaggle 1mg Medicine Data"
}

kaggle_results = []

for filename, source_name in kaggle_files.items():

    path = os.path.join(BASE, "kaggle", filename)

    df = pd.read_csv(path)

    records = df.where(pd.notna(df), None).to_dict("records")

    for record in records:
        kaggle_results.append({
            "source": source_name,
            "data": record
        })

# -------------------------------------------------------------------------
# 2. SAVE KAGGLE RAW JSON
# -------------------------------------------------------------------------

kaggle_output = {
    "metadata": {
        "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "sources": list(kaggle_files.values()),
        "discovered_files_count": len(kaggle_files),
        "total_records_ingested": len(kaggle_results),
        "execution_status": "COMPLETED"
    },
    "results": kaggle_results
}

kaggle_json = os.path.join(
    BASE, "kaggle", "kaggle_medicines_raw.json"
)

with open(kaggle_json, "w", encoding="utf-8") as f:
    json.dump(kaggle_output, f, ensure_ascii=False, indent=2)

# -------------------------------------------------------------------------
# 3. LOAD WHO EXTRACTED PAGES
# -------------------------------------------------------------------------

who_pages_path = os.path.join(
    BASE, "who", "WHO_EML_2023_pages.json"
)

with open(who_pages_path, "r", encoding="utf-8") as f:
    who_data = json.load(f)

who_results = []

for page_number, page_text in enumerate(who_data["pages"], start=1):

    if page_text.strip():
        who_results.append({
            "page_number": page_number,
            "text": page_text.strip()
        })

# -------------------------------------------------------------------------
# 4. SAVE WHO RAW JSON
# -------------------------------------------------------------------------

who_output = {
    "metadata": {
        "source": "WHO Essential Medicines List",
        "verification_status": "PDF_TEXT_EXTRACTED_NOT_STRUCTURED",
        "source_file": "WHO_EML_2023.pdf",
        "pages_count": who_data["pages_count"],
        "total_pages_with_text": len(who_results),
        "total_records": len(who_results),
        "has_synthetic_data": False,
        "execution_status": "COMPLETED_TEXT_EXTRACTION"
    },
    "results": who_results
}

who_json = os.path.join(
    BASE, "who", "who_eml_raw.json"
)

with open(who_json, "w", encoding="utf-8") as f:
    json.dump(who_output, f, ensure_ascii=False, indent=2)

# -------------------------------------------------------------------------
# 5. VERIFY
# -------------------------------------------------------------------------

print("=" * 70)
print("CELL 59 — RAW DATA UPDATE SUMMARY")
print("=" * 70)

print("Kaggle records:", len(kaggle_results))
print("Kaggle JSON size:", os.path.getsize(kaggle_json))

print("WHO pages:", who_data["pages_count"])
print("WHO pages with text:", len(who_results))
print("WHO JSON size:", os.path.getsize(who_json))

print("\nKaggle status:", kaggle_output["metadata"]["execution_status"])
print("WHO status:", who_output["metadata"]["execution_status"])

print("\nCELL 59 COMPLETED SUCCESSFULLY")

CELL 59 — RAW DATA UPDATE SUMMARY
Kaggle records: 22145
Kaggle JSON size: 11044473
WHO pages: 71
WHO pages with text: 69
WHO JSON size: 172048

Kaggle status: COMPLETED
WHO status: COMPLETED_TEXT_EXTRACTION

CELL 59 COMPLETED SUCCESSFULLY


In [51]:
# =============================================================================
# CELL 60 — UPDATE MEMBER 1 HANDOFF PACKAGE
# =============================================================================

import os
import zipfile
import json
import shutil
from datetime import datetime, timezone

ZIP_PATH = "/content/member_1_deliverables.zip"
REPORT_DIR = "/content/reports"

os.makedirs(REPORT_DIR, exist_ok=True)

# Files to add to the handoff package
new_files = [
    "/content/data/bronze/kaggle/Medicine_Details.csv",
    "/content/data/bronze/kaggle/medicine_data.csv",
    "/content/data/bronze/kaggle/kaggle_medicines_raw.json",
    "/content/data/bronze/who/WHO_EML_2023.pdf",
    "/content/data/bronze/who/WHO_EML_2023_extracted.txt",
    "/content/data/bronze/who/WHO_EML_2023_pages.json",
    "/content/data/bronze/who/who_eml_raw.json",
]

# Verify files
missing = [path for path in new_files if not os.path.exists(path)]

if missing:
    print("MISSING FILES:")
    for path in missing:
        print(path)
    raise FileNotFoundError("Cannot update package: files are missing")

# Preserve existing package contents and add/update new files
temp_zip = ZIP_PATH + ".tmp"

with zipfile.ZipFile(ZIP_PATH, "r") as old_zip:
    existing_files = {
        name: old_zip.read(name)
        for name in old_zip.namelist()
    }

with zipfile.ZipFile(temp_zip, "w", zipfile.ZIP_DEFLATED) as new_zip:

    for name, content in existing_files.items():
        new_zip.writestr(name, content)

    for path in new_files:
        archive_name = os.path.relpath(path, "/content")
        new_zip.write(path, archive_name)

os.replace(temp_zip, ZIP_PATH)

# Verify ZIP
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    bad_file = z.testzip()
    names = z.namelist()

if bad_file:
    raise RuntimeError(f"ZIP validation failed: {bad_file}")

summary = {
    "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    "package": ZIP_PATH,
    "new_files_added": len(new_files),
    "total_files": len(names),
    "zip_valid": True,
    "kaggle_records": 22145,
    "who_pages": 71,
    "who_pages_with_text": 69,
    "airflow_status": "GENERATED_NOT_EXECUTED"
}

summary_path = os.path.join(
    REPORT_DIR, "cell_60_handoff_package_update.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("=" * 70)
print("CELL 60 — HANDOFF PACKAGE UPDATE")
print("=" * 70)
print("New files added:", len(new_files))
print("Total files in ZIP:", len(names))
print("ZIP valid:", True)
print("ZIP size:", f"{os.path.getsize(ZIP_PATH):,}", "bytes")
print("Package:", ZIP_PATH)
print("Report:", summary_path)
print("\nCELL 60 COMPLETED SUCCESSFULLY")

CELL 60 — HANDOFF PACKAGE UPDATE
New files added: 7
Total files in ZIP: 33
ZIP valid: True
ZIP size: 3,197,092 bytes
Package: /content/member_1_deliverables.zip
Report: /content/reports/cell_60_handoff_package_update.json

CELL 60 COMPLETED SUCCESSFULLY


In [52]:
# =============================================================================
# CELL 61 — FINAL MEMBER 1 HANDOFF CHECK
# =============================================================================

import os
import zipfile
import json

zip_path = "/content/member_1_deliverables.zip"

required_files = [
    "data/bronze/kaggle/kaggle_medicines_raw.json",
    "data/bronze/who/who_eml_raw.json",
    "data/bronze/who/WHO_EML_2023_pages.json",
    "reports/member_1_final_handoff.txt",
    "reports/airflow_pharma_pipeline_dag.py",
    "reports/cell_60_handoff_package_update.json"
]

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()
    missing = [f for f in required_files if f not in names]
    bad_file = z.testzip()

print("=" * 70)
print("CELL 61 — FINAL HANDOFF CHECK")
print("=" * 70)

print("ZIP exists:", os.path.exists(zip_path))
print("ZIP valid:", bad_file is None)
print("Total files:", len(names))

print("\nRequired files:")
for file in required_files:
    print(("FOUND: " if file in names else "MISSING: ") + file)

print("\nMissing required files:", len(missing))

if not missing and bad_file is None:
    print("\nFINAL STATUS: MEMBER 1 HANDOFF PACKAGE VERIFIED")
else:
    print("\nFINAL STATUS: HANDOFF NEEDS REVIEW")

print("\nCELL 61 COMPLETED")

CELL 61 — FINAL HANDOFF CHECK
ZIP exists: True
ZIP valid: True
Total files: 33

Required files:
FOUND: data/bronze/kaggle/kaggle_medicines_raw.json
FOUND: data/bronze/who/who_eml_raw.json
FOUND: data/bronze/who/WHO_EML_2023_pages.json
MISSING: reports/member_1_final_handoff.txt
MISSING: reports/airflow_pharma_pipeline_dag.py
MISSING: reports/cell_60_handoff_package_update.json

Missing required files: 3

FINAL STATUS: HANDOFF NEEDS REVIEW

CELL 61 COMPLETED


In [53]:
# =============================================================================
# CELL 62 — FIX MISSING HANDOFF FILES
# =============================================================================

import os
import zipfile

ZIP_PATH = "/content/member_1_deliverables.zip"

required_files = [
    "member_1_final_handoff.txt",
    "airflow_pharma_pipeline_dag.py",
    "cell_60_handoff_package_update.json"
]

search_dirs = [
    "/content/reports",
    "/content"
]

found_files = {}

for filename in required_files:
    for search_dir in search_dirs:
        for root, dirs, files in os.walk(search_dir):
            if filename in files:
                found_files[filename] = os.path.join(root, filename)
                break
        if filename in found_files:
            break

print("FILES FOUND:")
for filename, path in found_files.items():
    print(filename, "->", path)

missing_on_disk = [
    f for f in required_files if f not in found_files
]

if missing_on_disk:
    print("\nFILES NOT FOUND ON DISK:")
    for f in missing_on_disk:
        print(f)
    raise FileNotFoundError("Some handoff files are missing from disk")

# Add missing files without deleting existing ZIP contents
with zipfile.ZipFile(ZIP_PATH, "a", zipfile.ZIP_DEFLATED) as z:
    existing = set(z.namelist())

    for filename, path in found_files.items():
        archive_path = "reports/" + filename

        if archive_path not in existing:
            z.write(path, archive_path)
            print("ADDED:", archive_path)
        else:
            print("ALREADY EXISTS:", archive_path)

# Final verification
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = set(z.namelist())
    bad_file = z.testzip()

missing = [
    "reports/" + f
    for f in required_files
    if "reports/" + f not in names
]

print("\nZIP VALID:", bad_file is None)
print("TOTAL FILES:", len(names))
print("STILL MISSING:", missing)

if not missing and bad_file is None:
    print("\nFINAL STATUS: HANDOFF PACKAGE VERIFIED")
else:
    print("\nFINAL STATUS: NEEDS REVIEW")

print("\nCELL 62 COMPLETED")

FILES FOUND:
member_1_final_handoff.txt -> /content/reports/member_1_final_handoff.txt
airflow_pharma_pipeline_dag.py -> /content/reports/airflow_pharma_pipeline_dag.py
cell_60_handoff_package_update.json -> /content/reports/cell_60_handoff_package_update.json
ADDED: reports/member_1_final_handoff.txt
ADDED: reports/airflow_pharma_pipeline_dag.py
ADDED: reports/cell_60_handoff_package_update.json

ZIP VALID: True
TOTAL FILES: 36
STILL MISSING: []

FINAL STATUS: HANDOFF PACKAGE VERIFIED

CELL 62 COMPLETED


In [54]:
# DOWNLOAD FINAL MEMBER 1 DELIVERABLES

import os
from google.colab import files

zip_path = "/content/member_1_deliverables.zip"

if os.path.exists(zip_path):
    print("File found successfully!")
    print("File size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")
    files.download(zip_path)
else:
    print("ERROR: ZIP file not found!")

File found successfully!
File size: 3.05 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
import os

print("Python version:")
!python --version

print("\nDocker availability:")
!docker --version

print("\nAirflow availability:")
!airflow version

Python version:
Python 3.13.18

Docker availability:
/bin/bash: line 1: docker: command not found

Airflow availability:
/bin/bash: line 1: airflow: command not found
